# LayoutLM Document QA (impira) — DIMER E2E document question-answering fine-tuning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/layoutlm-document-qa-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/layoutlm-document-qa-pipeline/blob/main/tutorials/layoutlm_document_qa_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-impira%2Flayoutlm--document--qa-ffcc4d?style=flat)](https://huggingface.co/impira/layoutlm-document-qa) [![Upstream](https://img.shields.io/badge/Upstream-microsoft%2Funilm%20(layoutlm)-181717?style=flat&logo=github&logoColor=white)](https://github.com/microsoft/unilm/tree/master/layoutlm) [![arXiv](https://img.shields.io/badge/arXiv-1912.13318-b31b1b.svg)](https://arxiv.org/abs/1912.13318)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.2 — **standalone** (§4)  
**Capability:** extractive document question answering over OCR words and boxes and bounded supervised fine-tuning of the last encoder blocks and the span head on a gold-span receipt dataset, using the pinned `impira/layoutlm-document-qa` weights

**This notebook is standalone.** It carries the repository's package (3 modules under `src/layoutlm_document_qa_pipeline/`, at revision `d14e37221df0`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `beed3c4d02d86017ebca5bd0fdf210046b907aa6` (~514 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2.1); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime builds an isolated environment from the hash-locked pins (the kernel's own packages are left alone, so no restart is needed), stages and digest-verifies the pinned `impira/layoutlm-document-qa` snapshot (safetensors, 511 MB), reads the `ground_truth` column of two digest-pinned CORD-v2 receipt shards from the Hugging Face Hub (about 0.4 MB over HTTP range requests, no credential), turns the 199 unique receipts into templated question/answer records over their real OCR words and boxes and cuts them by receipt into 119 / 30 / 50 training, validation and test pages, drops any record that would not fit one 512-token window, answers five authored questions over a rendered invoice through the inference contract with an input manifest and a rejection probe, scores the frozen model on the test receipts with ANLS and exact match beside the last-number and keyword-lookup baselines, runs a bounded fine-tuning of the last four encoder blocks and the span head on the training receipts with validation-ANLS epoch selection, scores the held-out receipts again, re-reads the invoice with the adapted model, exports the adapter as safetensors with a manifest, reloads that artifact into a fresh pipeline to verify answer parity, and puts three questions the invoice cannot answer to both models. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.2 §5). On CPU the model stages take about twenty minutes after the downloads; a CUDA runtime is used automatically when present (the six epochs took 82 s on a Kaggle Tesla T4). Building the isolated environment comes on top of this.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4, select that cell and choose **Runtime → Run after** (it re-runs Section 4 and every later cell; Sections 5, 6 and 7 reload the pretrained model first, so the earlier adaptation never carries over into a *frozen* number or into your adapter). Supply one JSON array or JSONL file of `{id, page_id, question, words, boxes, image_size, answer_start, answer_end}` records — the page's OCR words with one pixel `[x0, y0, x1, y1]` box each, the page size, and the inclusive word indices of the gold span — through the upload dialog on Colab, or as a path in `BYOD_PATH` on any runtime. It needs **at least 12 records with one question per page**: the seeded split keeps 20 % for test and 15 % for validation and must leave at least 8 training, 2 validation and 2 test records, and a page with several questions moves as a whole, so such a dataset can need more; Section 4 checks this before any model runs and names the split it refuses, and a file that is not valid JSON is refused with its line number. The records pass through the same validation, seeded page-disjoint split, fit check, baselines, fine-tuning, held-out evaluation, artifact export and reload-parity cells as the CORD-v2 sample; a result where the adapted model does not beat the frozen one is reported, not treated as an error. Uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

`impira/layoutlm-document-qa` is Impira's LayoutLM (v1) — a 12-layer BERT-style encoder with 2-D position embeddings, 128 M parameters, RoBERTa vocabulary — fine-tuned on SQuAD 2.0 and DocVQA for extractive document question answering and published under the **MIT** licence. At inference the encoder reads the question and the page's **words with their boxes** (normalised to a 0–1000 grid) as one token sequence and emits a start and an end logit per token; the carried module scores every span of at most 15 tokens by the product of the start and end softmax probabilities and returns the best one as a word span with its indices. **The model never sees pixels:** OCR is an input provider outside the model, so words and boxes come from the caller (bring-your-own OCR), and the model has no abstention — it returns its best span for every question.

What this notebook adds to inference is **adaptation with gold spans**. The dataset is real: CORD-v2 (Park et al., 2019; NAVER Clova; **CC BY 4.0**) — photographed Indonesian receipts whose every printed word carries a pixel box and a field category. Because the model needs words and boxes only, the notebook reads just the `ground_truth` column of two pinned Hub parquet shards (about 0.4 MB of the 476 MB the shards hold with their images) and turns each receipt into templated questions — *What is the total amount?*, *How much change was given?*, *What is the name of the first item?* — whose gold answer is the value words of the matching field, a contiguous span of the page's OCR words by construction. The frozen model already reads most receipt totals (ANLS 0.843 on the test receipts in the recorded Kaggle Tesla T4 run), so the fine-tuning question is narrower and more honest than a rescue: does a bounded adaptation of the last four encoder blocks and the span head on 119 receipts lift the fields it gets wrong — item names, quantities, cash and change lines — on held-out receipts, and does any field get worse? Two metrics are implemented in the carried modules (mean **ANLS**, DocVQA's official normalised-Levenshtein score, and the **exact-match** rate after the same normalisation), and two **non-neural baselines** — last number on the page and keyword lookup — show where a reader with no model sits. Nothing here is a quality claim about your documents: it is one seeded split of one corpus with one OCR convention.

**Snapshot note:** the pinned revision ships a fast `tokenizer.json` (an 8-file manifest) and a float32 `model.safetensors` — no pickle is opened anywhere in this notebook; the upstream `pytorch_model.bin` and `tf_model.h5` are neither listed nor fetched. Section 3 stages and digest-verifies those eight files before the tokenizer or the model is constructed.

**Who this is for.** A learner who knows basic Python and PIL, has met the idea of a pretrained model, and wants to see how a document question-answering model is measured against gold answers and adapted to one document family. No prior experience with LayoutLM, span extraction, ANLS or fine-tuning is assumed; each term is explained where it is first used and again in the **Glossary** at the end. A CPU runtime is enough (Prerequisites).

**Input → Model → Output.**

| | Inference | Fine-tuning |
|---|---|---|
| Input | one question (at most 256 characters) and one page's OCR words (1 to 2,000) with one pixel box each and the page size | 595 templated questions over 119 receipts, each with the page's words and boxes and the word indices of the gold answer |
| Model | LayoutLM v1: words + 2-D boxes → 12 encoder blocks → a start and an end logit per token; the best span of at most 15 tokens | the same model with the last four encoder blocks and the span head trained (28.4 M of 127.8 M parameters), everything else frozen |
| Output | a word span (`answer`, `start`, `end`) and a span `score` — a ranking signal, not a probability | an adapter (safetensors) and held-out ANLS / exact match per field beside two baselines and the frozen model |

**How to use this notebook.** Choose a runtime, then **Runtime → Run all**. Sections 1–3 are **infrastructure** — the isolated environment, the carried code and the model verification — and can be run without study; their code is collapsed. The learning path starts in Section 4. Form fields (`# @param`) are the only values meant to be edited; the defaults reproduce the recorded path. Each stage asks you to **predict** before it runs, and the next cell opens with **What to notice** and a collapsible **Check your reasoning** block with a worked answer from the recorded Kaggle Tesla T4 run of the previous notebook version (same model stages; a CPU or another GPU gives the same frozen numbers and training results within about 0.01 — see **Run-to-run spread** in Section 7). Section 10 is a short probe you can edit, Section 11 a **change-one-thing activity**, and each optional experiment names the cell to change and the cells to re-run. **Troubleshooting**, a **Glossary** and a **Conclusion** template are at the end. Your notes are optional and are not required submissions.

**Roadmap:** 4 fetch, validate and split the receipts → 5 answer questions on a rendered invoice with the pretrained model → 6 two baselines and the pretrained model on the test receipts → 7 fine-tune the last four blocks → 8 compare on the held-out receipts, field by field → 9 re-read the invoice, export and reload → 10 ask questions the page cannot answer → 11 **change one thing: two trainable blocks** → conclude. Core concepts are Sections 5, 6 and 10 (what the model returns and how it is scored); evaluation practice is Sections 4, 7 and 8 (split, selection, held-out comparison); engineering and reproducibility are Sections 1–3 and 9.

**Learning objectives:** by the end you should be able to (1) explain *question + OCR words and boxes → start/end logits → best word span* and say why the span `score` is not a calibrated probability (Sections 5 and 10); (2) read ANLS and exact match against a last-number and a keyword-lookup baseline and say what each baseline rules out (Section 6); (3) explain why every question on one page stays in one split and how the validation split, never the test split, chooses the kept epoch (Sections 4 and 7); (4) predict and then check how each field moves after a bounded fine-tuning, including a field that gets worse, and say which per-field numbers are too small to read (Section 8); (5) check that an exported adapter reproduces the evaluated model (reload parity, Section 9); (6) show that the model returns a span even when the page cannot answer the question (Section 10); and (7) predict and then observe what training two blocks instead of four changes (Section 11).

**This notebook does not demonstrate:** OCR itself (the notebook installs no Tesseract and ships no OCR model; the default path uses the corpus's annotated words and boxes and the invoice renderer's own boxes), PDF or multi-page documents (one page per call), abstention (the model always returns its best span, even for an unanswerable question), answers that are not a contiguous span of the OCR words, arithmetic or reasoning, evaluation on the DocVQA benchmark (registration-gated and not bundled), full-model or embedding fine-tuning, training on pages that need more than one window, languages and scripts other than the Latin-script receipts and English questions used here, and any claim that a CORD-v2 split stands in for your documents. The repository exposes none of these.

## Prerequisites

- **Learner:** basic Python and PIL, and Colab or Jupyter familiarity. No prior experience with LayoutLM, span extraction, ANLS or fine-tuning: the notebook explains span scores, ANLS and exact match, the two baselines, the fit check and validation-based epoch selection where they are first used, and the Glossary repeats them.
- **Runtime:** a fresh **Linux x86_64** runtime — Google Colab, Kaggle or Linux Jupyter. Section 1 builds its own **Python 3.12.12** environment from a hash-locked list of manylinux wheels, so the kernel's own Python version does not matter (Colab's kernel runs Python 3.13 as of September 2026; the stages still run on 3.12.12), and a Windows or macOS kernel is not supported (Section 1 stops with that message). The default path runs on CPU (float32) and uses CUDA automatically when present. CPU is adequate but not fast: a CPU pre-flight of the previous notebook version took about 7 s to load and digest-verify the 514 MB snapshot, 41 s to score the 229 test questions, and 934 s for the six epochs of fine-tuning with validation scoring; the recorded Kaggle Tesla T4 run took 82 s for the six epochs. The locked install (PyTorch 2.14.0 with its CUDA libraries, a few GB) and the 511 MB checkpoint are the large downloads of the run.
- **Knowledge:** what extractive (span) question answering over OCR tokens is; why a start/end softmax product is not a calibrated confidence; what normalised Levenshtein similarity (ANLS) measures and why neither it nor exact match is a human judgement. Each is explained again where it is used.
- **Data contract:** records are `{id, page_id, question, words, boxes, image_size, answer_start, answer_end}` — the page's OCR words (1..`MAX_WORDS` = 2,000, non-empty) with one pixel `[x0, y0, x1, y1]` box per word inside the page, the page size in pixels (`MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE` = 1..10,000), a question of at most `MAX_QUESTION_CHARS` (256) characters, and the inclusive word indices of the gold span; an optional `answers` list must start with the span text. Ids match `[A-Za-z0-9_.:-]{1,64}` and are unique; a training split needs 8..20,000 records and the validation and test splits at least 2 each, so **a BYOD dataset needs at least 12 records with one question per page** (Section 4 computes and prints this minimum, checks it before any model runs and names the split it refuses); every question on the same page lands in the same split so a test page is never trained on; a question + page that needs more than one 512-token window is dropped from training by the fit check, never truncated.
- **BYOD layout:** one `.json` file holding an array of records, or one `.jsonl` file with one record object per line, by upload on Colab or by path in `BYOD_PATH` on any runtime. A line that is not valid JSON is refused with the file name and the line number.
- **Validation is structural, not semantic:** nothing checks that a question is answerable from its page beyond the gold span lying inside the words, or that a gold span is the *best* answer — a mislabelled corpus is fine-tuned on without complaint.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — an internal document set with its field annotations is exactly that. The default path uploads nothing.
- **External access (data):** besides the model snapshot, the default path reads the `ground_truth` column of two objects in the Hub dataset repository `naver-clova-ix/cord-v2` at the immutable revision `7f0115a4…` (`data/test-…parquet`, 234,202,795 bytes, SHA-256 `51c65f17…`, and `data/validation-…parquet`, 242,080,800 bytes, SHA-256 `0d0f6dac…`): the declared size and SHA-256 of each file are checked against the pins before any byte is read, only the parquet footer and that one column are fetched over HTTPS range requests, and the decoded column is refused unless its own SHA-256 matches; the corpus is CC BY 4.0 (Park et al., 2019). No image is downloaded.
- **External access:** the Hugging Face Hub only, to fetch the pinned `impira/layoutlm-document-qa` snapshot (~514 MB in total) at revision `beed3c4d02d8…`. No repository access and no credentials are required; nothing is installed from this repository. The isolated environment also needs PyPI (`pypi.org`, `files.pythonhosted.org`) for the pinned `uv` wheel and the 48 hash-locked packages, and the managed CPython 3.12.12 build (python-build-standalone) that `uv` downloads.

## 1. Install the pinned runtime (isolated environment)

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

Hosted runtimes such as Colab import some packages, NumPy among them, before the first cell runs, and Python cannot swap a module that is already loaded. Installing the pins into the notebook's own Python would therefore leave mixed versions or require a manual restart. Instead, the next cell builds a separate environment (`dimer_isolated_env/`) and leaves the kernel's packages untouched: it downloads the pinned `uv` 0.12.15 wheel and refuses it unless its size and SHA-256 match, has `uv` install the managed CPython **3.12.12** (whatever Python the kernel itself runs), and installs the 48 packages of the carried hash-locked requirements (`tutorials/requirements-colab.lock.txt`, compiled from the pins below) with `--require-hashes --only-binary :all:`, so every package, direct or transitive, is the exact file that was locked. No repository code is installed. The lock holds manylinux x86_64 wheels, so the notebook supports **Linux x86_64 runtimes only** (Google Colab, Kaggle or Linux Jupyter); on any other platform the cell stops with that message. The printed dictionary names the isolated Python version and the kernel's.

In [ ]:
# @title Infrastructure: install the locked runtime into an isolated environment
# dimer: kernel cell (runs in the notebook kernel, not in the isolated environment)
import hashlib
import io
import os
import platform
import subprocess
import sys
import time
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
    'pyarrow==25.0.1',
]
MANAGED_PYTHON = '3.12.12'
UV_URL = 'https://files.pythonhosted.org/packages/1e/fd/432451d732917c49152a291de3ef171aa6b0f1a22d39780fb2c1f085ca4c/uv-0.12.15-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
UV_BYTES = 20081404
UV_SHA256 = 'aee9802f46bae436bd91751bb33ddeb379ef1596b5c19df193219d545d244b60'
LOCK_NAME = 'requirements-colab.lock.txt'
LOCK_SHA256 = 'a55548d4e338fb504e610b965eb5cca22bd3017c35a22c7472a2550d75109d64'
LOCKED_PACKAGES = 48
# The hash-locked requirements, compiled from the PINS above with `uv pip compile --generate-hashes` for manylinux x86_64.
LOCK_TEXT = r'''# This file was autogenerated by uv via the following command:
#    uv pip compile pyproject.toml --python-version 3.12 --python-platform x86_64-manylinux_2_28 --generate-hashes --only-binary :all: -o tutorials/requirements-colab.lock.txt
certifi==2026.7.22 \
    --hash=sha256:62f22742b58a1a33014a2b6b706588a8d7e2a88ae7bd1a6ebe8c992928483775 \
    --hash=sha256:741e2c3b351ddf169a738da9f2c048608ff7f2c5cc02f1ebc6b118bb090d5d55
    # via requests
charset-normalizer==3.5.2 \
    --hash=sha256:01077390b03f7988f11d700a2194e69b119741a86b1a638b1db88891e3eced8e \
    --hash=sha256:01b0c0d2262a9e28e8484a278c7e1b5d650e3ac8cf2683d2967e25899f208bdf \
    --hash=sha256:04851f73ae72b8413dddadb16a49dfee95263553741fd42d546f7d66907e6be5 \
    --hash=sha256:0521c5665880b33d603717defa76c094048900010897909952397feb3039da56 \
    --hash=sha256:0774bf9bf620249fee3e0b8b9fd3065de213be30f3aa94ce2494b3b638949e26 \
    --hash=sha256:0891b9d3903c5571c03771ca669a4b0ec5618ca722a5c957d3d29cd4e5062848 \
    --hash=sha256:0c951d5e6dd9c2ff60609476752bee49da4206adde960ebc247766937f72e718 \
    --hash=sha256:0fed1d06615f022ee3b13caf5e8b180cfea32bb2c5aded8a9d44277afc040f93 \
    --hash=sha256:114e4d0c92d618409ed82a99e22b5c5e768fe995f2973f78265f4524f49d4640 \
    --hash=sha256:11912e4bb14baae7c5d8791aa55ba0a3a03ec6729073307b0f57270abaa713d3 \
    --hash=sha256:11a4d68a6ecda3292cb1e50239e111543ba5d709bb62a6b4ea1afcfa729d8875 \
    --hash=sha256:124fbf1a8ff966d87ae05bb8bd45a71f966055ed8bba320d0c7cf450bc5f4d0e \
    --hash=sha256:1461ac396c4fdb983a675f20aa555624f0ee18ac83d832b9244ffff3d8055275 \
    --hash=sha256:1503bccbeb36d5527790c3930327704c39af22de3112f1b1666a9f3ce15ee204 \
    --hash=sha256:15bb4005af6320d259dc7593ca84a38d7fe06a421dbcf7b910ae23979101e787 \
    --hash=sha256:15c44f7edfd477b06f517a5cc317fc1707edb9de2c865f43d4b6513907473234 \
    --hash=sha256:16fa0eccf81304b79c5cd87f9271c3b85dd9dd99245e4422ae9c0dd45e0f99d3 \
    --hash=sha256:183b88127acdb4fabe59d951ab424faf1af7b63cdbb5f776186c1ea2ffcaed98 \
    --hash=sha256:195c26fb65950f8fce54e26349852b7bdd7c5f120aeefbcc440b8a20faaed4a3 \
    --hash=sha256:1afb975bd5d68d5ce9f6b6d44fdf2f7e34b895a35e95708a7a91b20a3b51d187 \
    --hash=sha256:1b4cbc7c3491ccb4aa17fcd8165649d01cf39f76de1696da8631b5f71b85401d \
    --hash=sha256:1bc0baf5ef96b6ede57d47f4b8fe4d9d84019c3bfcbeb20a41edc6a6ee341f1f \
    --hash=sha256:1c50fe28bbc2ced33386f298650d91218076c05420e6cbd790b913adc41659e7 \
    --hash=sha256:1db38f4c5496827c1a501846d64d14c3b80c7e6714e406cd7dc36a9899fa1011 \
    --hash=sha256:211d5a3eb6af8f513b8d4ca19a8c1b7accab1b5f0d3175f9826b03c1a920dc1f \
    --hash=sha256:23851fb4e1b85ed3f6c2a27b777cdfe2e19fb5b38429a8faf38c7542b7665869 \
    --hash=sha256:254eb48b9fa5ee9898a3c445825a1f340fe53712a098904b39b0bddba8ea3cb1 \
    --hash=sha256:2625388c6c754520c37abaf3b41eb34d1cc4a373f457898f08606c8e362b891d \
    --hash=sha256:281cb91036248400f4cc957495cccd44c275c2e0c5854f7e45ac5cf7dc193847 \
    --hash=sha256:28a15fdad492a99b6eccfaaed66ef3f74050680545ea61ec8b2f4c538f1f1320 \
    --hash=sha256:28b4f0d66fb834ff90f28209ac7bce77868c45d8c93e26f906709d9b7c2e1af9 \
    --hash=sha256:2a925889534b3748302dae5dead07cc13480de1dac3aea80a941b729b471ef93 \
    --hash=sha256:2b7b3bbfb4fe8ef40600792d762fbaa9057559f9d3fad209525b7a22b99e91fd \
    --hash=sha256:2c9ad19a6cfcd5ea5c0d41161d22f9df1dcc277e9bef2751391334546a314c00 \
    --hash=sha256:2cc961b171b3f3440f410489ab3573e86aea8736134ebbb40ea1338b7f0831bc \
    --hash=sha256:2ce45c6627b22c47e390bc91a41c3d13032192e699fa0bea96e9671b373d69b0 \
    --hash=sha256:2e06a3a98f916dd41d27f3105e02e7a40181c98c94b9158733d03a6f80506c09 \
    --hash=sha256:304d5463e65a35d7bb0850550e0780395395f6fcf452f04db7d5ca7cecc425ac \
    --hash=sha256:304d8e4d493af723536393eee0c689eb7813f4a474c8b479dee63f1fdd98f621 \
    --hash=sha256:30fcd120b732aa79317f08dee04d7de0847822e4cf7ee0e9f445bb958832252c \
    --hash=sha256:31f3930700408d211f13378ccbe1c40845d8da54bd0681fac3a9b5aae81c7aa8 \
    --hash=sha256:34276fd796040bf0993ab33a369aa572e6979c7aab225a88893667ad8eac8f7a \
    --hash=sha256:355ad8011081dec5412240c087a9a0c9d4d5039f3ed11a3f13e18c2b29b56c51 \
    --hash=sha256:38a873987f3be698494da8b2e3085e29da02da7b633dce73e79c699a113d7bf0 \
    --hash=sha256:39de2a259fc954455c57274dc94c79d5842774e1247a016aff30bc0efed0f4ef \
    --hash=sha256:3d14b50de6bf4d0edf857a9386836846f982b8f524e188e2e68b96d702bcf4aa \
    --hash=sha256:3d21b8b13c7592db2ac5e544a6d83187b995257472b0c9e8351b6d507ae37ed6 \
    --hash=sha256:3d31298449090ab8d47b7b1b2a555ff73cac7ed438a08b7ac160980c7ebed649 \
    --hash=sha256:3ddacd27458c45bdacd6bd6db644bfb730efbf9e830310186e3045c9c5be8fb2 \
    --hash=sha256:3df041de8887954562c9b261cba85ca0e9ded74048daf125f45edcfaa4832229 \
    --hash=sha256:40ab6bffa02ae10a0581e6c198be7d2d8ca5c2a0c64e4ed3465d766df457573e \
    --hash=sha256:4275811936e2f06feff5e598fb42a1b7ae852da8e39605211892b56b81a34efd \
    --hash=sha256:443eae2bf318abeaf6f15d785138f71fd6de770e99a92158b8b814265e079115 \
    --hash=sha256:447441e76ec720b15e64418d32e092297340387053047c7c694f579efb0ee1d9 \
    --hash=sha256:4495c5002a7b28557e7e222e77e0b661183e432b7d6d2e788101e3f240e05b8c \
    --hash=sha256:44bd4fbb29dfbeba60e7d2bd000c59e4b21ddb3cc53912b14048d37092706d7c \
    --hash=sha256:4685902cf26edf013ed7a3da0f426ebba7a00ebb9541386d835afbf002c11cab \
    --hash=sha256:498dc3188ca05a68231ac3fdbfc7f57eb67e1343c30e0fea17f8218c1599b253 \
    --hash=sha256:4c2b5031f63e331e3839b40aed2dd6f191e9c07edbde303e7876846ea1946995 \
    --hash=sha256:4d48f2d08b9de5864e2c8744d4461b862fb149a18274abc8b698c45975573438 \
    --hash=sha256:4f87960d57feabfb618e4e0af6e7371645fa26a277860739d6e5d6e0012c92f0 \
    --hash=sha256:50e3adfb96fc189eb27b1cf62d3b598b89b4bb0420d93a3d3e42e137409011be \
    --hash=sha256:51cf45226a9b588d0d2b4880c62d686934b63ab0bd79ca23ab0e9762eb27441b \
    --hash=sha256:52aa6992700996af31f375de0c6bacd402b0097fe40b53c426b9f51a90ebabc7 \
    --hash=sha256:55ea99acb17b9325618de155a0cd6a2e8f5d10be008113e1d433bbb58db543b2 \
    --hash=sha256:56bc200a365efb37383b7852e4cc5898d3b2da5987289b543956cf8cad71018a \
    --hash=sha256:588461c2e8384d309bd63e5826019b6977bc66d629b99ac8737bb795d7b2cb5a \
    --hash=sha256:58ca3755ee7ff7f59b57789ec9833c9de9ea275405cdd240eda1f193112e398a \
    --hash=sha256:58f361dcbab699cf8f42db3f47c8e7fd1036f138c23a5d08de9fde5f425a730c \
    --hash=sha256:598a11a2c7ebaa5334bf698bf29568c9c390abac6a154d8170fedecd1cea38c5 \
    --hash=sha256:59f63901b0031c3136cf64704dcb21de0bbae62ce2c9529bc39d27665463de37 \
    --hash=sha256:5cde776b7cc66e4f6c99612cea4aa7269aa65863f7a15841b2c264f103822f4e \
    --hash=sha256:5e2b6b57e9733d39f0c9fd3185efa6b8e29652c4cd8fe94180272cf6ed9a78c4 \
    --hash=sha256:5fb29fb8cd1a46c27a1bf9613ad5ec2599310d46b4025d9556404a6b6a292800 \
    --hash=sha256:6045373d5a89a5ec71afde535db987ca28e76dfa276c2d4c818265b375d4b055 \
    --hash=sha256:619799369eeef6366ed3e8755a5670f4f2f0fb6b30a0fd7264dc0fdc2357058e \
    --hash=sha256:62588a277bfb59def052abd940703fa35107152bf479781a878617d60faf8fb5 \
    --hash=sha256:62603db9a7caa0802eaa28c1c46fecd7b3a263a774069c24c3c28c302448721c \
    --hash=sha256:65cd72beeeca9d3aaea1201e5923859f308f952f9c71de93f06063c79f0f7a3b \
    --hash=sha256:68eb192d85ab8e5f6ec69c2bc6ac0179fbf04a5ac1569d12fbef74883fe102d0 \
    --hash=sha256:6bd128f206a7752ae1f2ab6c61bf8a24ba28913a10df8b14c2637b973ff97a80 \
    --hash=sha256:6be488a102b8cf28d0391d8c4ba7748938ae28b78ad901f8585520fca33ead1a \
    --hash=sha256:7218e8f32b0956cfcd048fd42d9d5779809745ca1d86113ca56f66e7ae1549c4 \
    --hash=sha256:7441d755b7ab94f8d4eb3e43ec05482d760842fd263d003a99102d742cd835e2 \
    --hash=sha256:749e97e1b32313717a565abbe321bc2190bc8b35f1a67e4cdbc7c56c8d8ffe58 \
    --hash=sha256:75a3ceed0724d625d64b86ca20aba182e4df462e04c2414fc941c0f523f06aac \
    --hash=sha256:780fbe7cab297b81dad9fb8dc5eb003c0468ffb0d9e5f65068c53a34661a96bc \
    --hash=sha256:78456a747de8dc58360ffa581f30a002baf5aa28cb262536545e91f113ed7639 \
    --hash=sha256:7967d08cf06dee78443b874f98c98036f624f3a4e73e11f9f64f5be4d25393cf \
    --hash=sha256:7a881931aa470808df94a8c380eed2bbbc76cd9dc622310f99665658c821eb6d \
    --hash=sha256:7dcd882da75ef9adf94903b1e3b9419e8aa8fb4c7396822b834b9ef7fb96954f \
    --hash=sha256:7e841fb9010836c992c9f12fcbd43a831de93a5f726fc1ccd8ca1d0268c5014c \
    --hash=sha256:7fdde2c9fd9e3eca40631e024664cf2584272cc8f96308cbe5fdfc930f51d8bc \
    --hash=sha256:8024d00c3faf3fc0c16e07a69f4405e8eac7cc0ab15f65fe6cf43827c4cf72b4 \
    --hash=sha256:80d02b6f04e92601a081dd97b23d3128033098bff5d35d392ddcc0476ea11253 \
    --hash=sha256:838dcc90063569a0448120554591a1d6c4a4ffe11babf048908793154ab86ade \
    --hash=sha256:849df64e889b2e17230d58410a03dba311a65b163508fd33679b2b737d4b7858 \
    --hash=sha256:87475fabc8d9996fd9c27debb395e642e8c838d78a00b6e932227a0e06b81e26 \
    --hash=sha256:87e50a3e7cb90af586b6c5faf23e302a970415ac73bd7bd90a515a04b427ef96 \
    --hash=sha256:89b53f3cda69831909888e0494f4fa0bcd3537e3e138dabeb620bd6ad946bae8 \
    --hash=sha256:8a893cc101149f80a653f82062ebc95b34525a2614382e1da5458fe7c6997249 \
    --hash=sha256:8b2bfab86aa71ae13aa41a6a26aab338e0db2b8bc75434b05aea89e011ff35a4 \
    --hash=sha256:8d86d6fc60743dc916eb79e2eb1ec4818e21e427731543af40a3021851174a13 \
    --hash=sha256:915563965d418f986e7e145accc592eae9e1a1be3566ff98a05d7a9ec42a76e1 \
    --hash=sha256:92888bb3187c5ba50500b00b3b310c9f2c651709d28036077680cb5255450a03 \
    --hash=sha256:93223adc95033dd47133a46ccfc316a0139176fd79085762e27202ec56018f03 \
    --hash=sha256:9373ad13ef0d2c0fb761e04e55bfdee5a08b52cef2c882c8fbe9935b1517152e \
    --hash=sha256:9409a8bf35cf78353942504b24a57de3d75b708997a1e4bd8db71ac8633ce364 \
    --hash=sha256:9b7f416ff0978e2f2249330527f0ad6fa02f4932e6199692d3b52da2048c19e4 \
    --hash=sha256:9bde855991b7e362c146535e3136a50bfaffc0487d38b33ca7e5edefc6e23849 \
    --hash=sha256:9cae88599c7219005d879f98e5ed53341e9a122af585e1091200358a3003d2a0 \
    --hash=sha256:9cf9b1a857e25c4baceeb3624e92a56df3668f398c4acba74e174d81fb4d1d3a \
    --hash=sha256:9f56f72050826f63dcee7a7f55b0a77168cb3bfc553fd405e7f8f9ece75a4036 \
    --hash=sha256:a090bb2c68df85450502e3e20d665e3a5af9c65a84d6508ed477badd49166fd3 \
    --hash=sha256:a192e2c40070d92c3ccf777e3a5c4ff515573cd2bb7ed0c537fdadbbec5bbf21 \
    --hash=sha256:a19a731138fc27d5682277d3b9df22855cea1239bce7fcec5f78f42ef2d1f3c3 \
    --hash=sha256:a66c3bc5ab1f0ff2164fc9965ddd611ff0802173f4b9d24554c563f6ab7e1d6e \
    --hash=sha256:a815775b6c38d4e0ff7bcffbeba67feded90202bb6a226b8dd35f1c855217413 \
    --hash=sha256:a89012d6d5476ee112d20d998570ed58df2260a852afb1758809cd6900411d21 \
    --hash=sha256:ae4f5fea5b8b8ccff88238cc8569303e5ee95efae67fa62922a311397a71f346 \
    --hash=sha256:b6856554c4f44d79fc2307d5768854310a8f0096e501c75637542c82292b0429 \
    --hash=sha256:b6b751274acb69d77b3323d6b7dbaa3c7fdfc1eb829b7eb61d262f32e1af9685 \
    --hash=sha256:b736353c0a625bbd5fcec108576e2385db3496f4f771f785ff32e108d3c3bc45 \
    --hash=sha256:b7fd005a73d9e657273b7a10dc71a9e03c8fb9ee6999798d6918ce095b81ac7f \
    --hash=sha256:b91363207bd9dc966a691e959bb47f64b30f7ac4b072be9968b366982f7db77c \
    --hash=sha256:ba0b1d2620edf869789c3879223f52bf2afc5d31b3cb47cc57b3a12c05e2aa9d \
    --hash=sha256:bbbfc8e28816f19d7c0f1816664980c0a9875d01b27cdf8eedddb639d9e108ad \
    --hash=sha256:bd16aabe4a02a297c23417aa17ac6299dbd8c49f673bcd645b4929b11f5a4400 \
    --hash=sha256:c0afc6800ba57ccc350374c5bd6150419915d95ce93cdbab2d783d75eaf30ecb \
    --hash=sha256:c6708715abcf3c73b99508253e961a9967f02fe536532834149574eda6de0d1c \
    --hash=sha256:c7c9ab723cde841fefb34efbad91e87f00a674b1fe1cd0784fde742bf2c154dc \
    --hash=sha256:c8f3d67aeaf55f017982b73683f0e7342ba2f6635a78f69ce89ebb26aa411e5c \
    --hash=sha256:c9790464842f85f437dbbb54417eda1e0e6bfc52dd8d22d6fd1c994b73b2dc74 \
    --hash=sha256:ca403d7e4798f525fdfc78e258820419cbbd0f0ecbab9de7840e3c017cf6b8cf \
    --hash=sha256:d008d90a7f2471519aef0c90dfbe73b3e6e4d5e66ac48e19154c17e89e98b604 \
    --hash=sha256:d19fbd981a488e22cd04883659ca6b08f50b5974f9fd7c95655ef6a043e5893f \
    --hash=sha256:d1befeed746d247c81127bb14de9dc3d30edb6e5976d34f83f86ed262b1d9105 \
    --hash=sha256:d2374b62878abb00cd8309b32af6c0b715cd02dec0ca74ef12e5069bdc64144a \
    --hash=sha256:d376bbd28b3a8999db1a103b3b388aee6f1ddeb3e51bc2172993efdcd86e064d \
    --hash=sha256:d4a7319f304a774bed22115bc891618e45f85065ab44ea6acd07d274e750519a \
    --hash=sha256:d6734d2ef8a50fbf8445c139477da401f50d62a0606bf00e20ec6d87773fefb1 \
    --hash=sha256:d760fe2a4d7c3b226cb9026d6a842868d52a7901bd98420e1baf14e80da85cf5 \
    --hash=sha256:d913de495d90407cd859d263bee2e5d1a4ed3eb6573c04e70d9ec619a7cbed7f \
    --hash=sha256:db19d07e2e0129e974a0e65d0064fc222a446cd5122c2fd4184d2af9fc734a9e \
    --hash=sha256:dca9ab98072a5a54ebacebdc45f53e645336b320c667410b061be1ca588ae709 \
    --hash=sha256:ddc7dacc8ece3a182e7f15cb862d1fd616b46d076cb1ae9dd232b2c38b655874 \
    --hash=sha256:ddf19c062bea7a0cc80f519243d2c01dd091be0cf952a0750d4ad576709559f5 \
    --hash=sha256:def79fa35ef0cef8d2accec024f4fdc7ead3012ff02f5215c783f39f03ef8cfc \
    --hash=sha256:df29a0a7107f7011e77f4eebdddec4c7331e24d787a0b21a46d63bdf7445da95 \
    --hash=sha256:e09a3942ecbdee5cce73ea9d42da82b81b72ac1bf031ce069b93b5adf4eac8cd \
    --hash=sha256:e242bb1c5e76e97dfa9e7f209a71e93a01d7f19ffdd5cfbb2e2d55b4f08f8ab0 \
    --hash=sha256:e243bd13217235fc7290c621941c3f5cc8b66e4872495be821d7436ba2fb838d \
    --hash=sha256:e2af3aad578aa6bd1384bcf4750fc285e5a9de53f40b7d41e5a0bf748edeb2b3 \
    --hash=sha256:e4e81e09c1578b8df602e3db08b0b3ea0a6947ad612f52bf8dc5ea8d47691f0c \
    --hash=sha256:e54da4baf05720032d527874d40b65fa4d7e5c6c6a43d0c3adbeffcaf275a2b3 \
    --hash=sha256:e80e6c2f55656b4824d72065abb4ddd6a525c74bd78a0aab5d9fc2cf4fb5af50 \
    --hash=sha256:ed2a239c0ea213acc1908150a3037257083c7c083128f1a4cec2ec4b97dca491 \
    --hash=sha256:ed905975ab14056a2e5eb1c376cb2e1ebc5396baf84163939c518556fccde9f5 \
    --hash=sha256:ee21e28f0430bd6dc9086c6e525d5e818a44a5ad19720c8a0ef766792f3eb5e5 \
    --hash=sha256:ee43c17b173d46a3212baa6ead3ae258eeabdae48c263a01ccf0218c366dd655 \
    --hash=sha256:ef4fcbf3327382cd4c9f540babd61248208af7b93eec4de397b4d5f58a09e288 \
    --hash=sha256:eff0ac9dbe711a4aee69bf04a83896aa9b85f19641264053a9f6d48573abb7dd \
    --hash=sha256:f0aa869112ef88429ae17820d99c3dd9504c9e9c671d3c246f3d7442cb051084 \
    --hash=sha256:f3c96f633825733f735c5a9cf21d21a257d8e1edf0b1cee0a064b9c424ca0f7d \
    --hash=sha256:f5833ad231be5eb6553de524a70f48d71b2c8563101750531e0b80184e175cd4 \
    --hash=sha256:f5ec61164adcec446f8969a3358ec3f9b26bbda3b9213e5586d219afa8df2915 \
    --hash=sha256:f7d486c83842422badd511868fd8a9a20e9407ace71564b6af47ce7e60a336c1 \
    --hash=sha256:fb9e68df06293761f9fe66ade60a9bc6d0f5e42b8acf2939a9158af86ab0e5bd \
    --hash=sha256:fc14a032f813bf5fe624d991960ea83e9715adc27e4c1830a2361eb1d02ac341 \
    --hash=sha256:fcff63213e8e6e47770541a4607175404f47cbb3ebea7b6058cc82d524a0e424 \
    --hash=sha256:fd1fbe0f116b6e55da77aca2c6ddcddcfac2186cbf78bdebf40fc156efca389d \
    --hash=sha256:fe9753dfee015c570d73df76f899f18444d41388bffcde097deba51c4fadbb9f
    # via requests
cuda-bindings==13.4.3 \
    --hash=sha256:044c03b056dcc5cecfad426a071187dd9e1e6817fbb363bc2f3a7520170b3e67 \
    --hash=sha256:0deff5b22462bb410859684299b1fc6a621780c43a4729342aa6ec28783485b4 \
    --hash=sha256:130ff1daae550db2cef559ba477f3a63d040756bd135f5deba4b5bdc4e110252 \
    --hash=sha256:145cc9b02dfb7bcc3288701b6f19c69c590e4451c62916ba0a8a3db9a64a91a8 \
    --hash=sha256:1fd7d8459b364aedc11f3e59703453ced823135f78a9111ca70feef8d56d4d21 \
    --hash=sha256:3d7a6506e625be59cdc119ef4423afa0fc3a4830b129dc010cbd51324b93d66d \
    --hash=sha256:4796864ce829bd95ef2ef0d23c6ba21bb64e08f7fab0a377302ed1affb6605c7 \
    --hash=sha256:52d7f3f5f7f014dddddc66cd802ed0ddf65ae99ad42b5619fae7b82e5ddd6771 \
    --hash=sha256:6c6bb1a4c4f7520e062669c6f3605a7016b2e65ba51b8922f92edd91d5fadf8d \
    --hash=sha256:7e11cfe8fec4c85ce79feda18124971c52596f0cbd642a94f5dafc257124a4b3 \
    --hash=sha256:81eef62bbb95cb4a705fb423b2ad1c63d716edb352ee2af0c28bf8a29cc8258c \
    --hash=sha256:a9ea13b9cfe711515ae83b5efc99101af4d3f8ad882f99724a839c8b5417fb7e \
    --hash=sha256:b89d6e738494b7b95c38e3413f86d32c24682c8e714870531a5a2b193a2fc50a \
    --hash=sha256:bbacde6f75665b197016b986164cfdaa33b17515e5e635a63ddb75926aaa71c3 \
    --hash=sha256:bc51990309b416e0780a11a921ac597ef9c0e52e1bdf5ae0b8e8c4766b66442b \
    --hash=sha256:bfbd3f7d4ac04dd41dc49121b9e408c8283992f47124c2290ecb79bbbadcca8e \
    --hash=sha256:c2af7e69d2557fdb4e5fc1169159fd08da7e861aa84fb83307a1a0396b6b0973 \
    --hash=sha256:d5f72bcfcdf3be23e1da3c792f68f508586f48d037bca8b10f552c4cca5971f2 \
    --hash=sha256:d6eb969920e28f66f8fc3b0b3afcb6e09381cc96bf8e8158d774e9488ae89980 \
    --hash=sha256:d7c6c9f46fca7f3fc61959ef9a2398ac656172145b43f408e0a6492360cf1c0c \
    --hash=sha256:df8b3767facca8acde216460df684dfc3826d519096c13adfd3030a55870dc79 \
    --hash=sha256:e52f66340785a51b8b77f4487329de188f3cabbf37c62e68f410a8299e8677ae \
    --hash=sha256:f8519603001c92bf83e7095df3b8211e3999a9c4ded096b57de0f3ff52b66368
    # via torch
cuda-pathfinder==1.8.3 \
    --hash=sha256:e29e59829c297a7a5233bd9cc71094fc5bddbd076951482670178f9eade39b1f
    # via cuda-bindings
cuda-toolkit==13.0.3.0 \
    --hash=sha256:d693caaa261214ddd7dbb60d68e71cbed884e68c2be7509778f3051da0b91c3f
    # via torch
filelock==4.0.9 \
    --hash=sha256:635e7d67fa92654eed444e75e9ca18426d34e77ad9c469bf4373f75a932f7b22 \
    --hash=sha256:9287fd61b99a806e5202be29a83034c0808a1b9830e820537c2f5773981f7eeb
    # via
    #   huggingface-hub
    #   torch
    #   transformers
fsspec==2026.9.0 \
    --hash=sha256:0f08147951c8cb31d844c3547d631053b127863b60be04cf06e121333ee0e2fe \
    --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f
    # via
    #   huggingface-hub
    #   torch
hf-xet==1.6.0 \
    --hash=sha256:0e6e21fa3cdfcdcd76748564bf593870a5e013f47d97cf10aed63aa222cff5b7 \
    --hash=sha256:23379c2f9ec8696d952b16414a2bae72cad86a52df869b050698ba60f538c675 \
    --hash=sha256:2e58454a340b3556dfa4972d5451aff4fba8dd42a236600ba1a1d2b1514f0fef \
    --hash=sha256:35cec30d75c6f9eb9c16a77cef68e85a103b72e24d4b473714ec9ff06428bab9 \
    --hash=sha256:3dc3e35441ba395006af5aaacc40ef2e603c51ef46c3530b9156185f00935ea3 \
    --hash=sha256:4fc74352a17015bd0ee90038bc9efe38db894cde45f268b6712b04fce8cd0acb \
    --hash=sha256:5153e6bb103ad49d6ea9f1b2e230db5a2ea32551ad09a706d2f61d7c7c80d80e \
    --hash=sha256:5789835d7c6bc9436962853192082374297fb72d7eff7e7762ec25ceb7e25338 \
    --hash=sha256:633dc0cd71d32da58ab8c03ad38e2fac452c15c2b0a2866ebf6ededfe0a5061d \
    --hash=sha256:70cbb9c896901600128cb9b6f06e132954fbede1db30f31f7c6c63f84cb7c31d \
    --hash=sha256:75765820ce4700db3750c94acc8fe27c5fae4c9ec000a0dbac3ca082acf97765 \
    --hash=sha256:8fb4f71cba6129110c3374a33f919001ff130488fc23553698e34cc1c2a1198c \
    --hash=sha256:948f15d3a9545cfe5932f6bd8b440f6ae630aee108f14b7bd6c561f7c2dcc522 \
    --hash=sha256:d62671bb130879cef0ee4c9ebe47a14af6c66ec53e6d84dc15936e5ffdfac82f \
    --hash=sha256:f0906082d9932ae0c0057fa194041c22b4e2cdb46b2592ef3b91f020d62a081a \
    --hash=sha256:f2f7278c05c22fd60cb436cda1269649b3e81db65ecdc8496e5e164aa4143e7b \
    --hash=sha256:fb4fadde1b2b70bf4c0c14a6dccbe7194b1c28947fefd5bbe3fed9d940676c3b
    # via huggingface-hub
huggingface-hub==0.36.2 \
    --hash=sha256:1934304d2fb224f8afa3b87007d58501acfda9215b334eed53072dd5e815ff7a \
    --hash=sha256:48f0c8eac16145dfce371e9d2d7772854a4f591bcb56c9cf548accf531d54270
    # via
    #   blip-captioning-pipeline (pyproject.toml)
    #   tokenizers
    #   transformers
idna==3.20 \
    --hash=sha256:a7db850025b95ded1eae8a46181a1a6c56c92c96f0e2b005d9ff8dc0210cab44 \
    --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c
    # via requests
jinja2==3.1.6 \
    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \
    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67
    # via torch
markupsafe==3.0.4 \
    --hash=sha256:007e1ffd9bf65bb6ee96df7b258fc632a4868dd5566037986c64781f35a36e98 \
    --hash=sha256:02fa4acbc6a3fc5c693c34d4dd8c1130b7fe99cc915181b0ddd6f72aeb296002 \
    --hash=sha256:03470d1a8268e692ecf79ecd565593e59d44219377a7ead61f1f1b94c1f7ff6b \
    --hash=sha256:04e7902ba80ee4bac1d50a549606527a1dcf0476cd81403db41099d3b60ec653 \
    --hash=sha256:051417f74bcaaefa316276e0ff723f541616ca51043d070da00249d9bddd3e3c \
    --hash=sha256:05295589e619b9bed252a86b532b8e27350abc372d18ba89b59375325e91ec1e \
    --hash=sha256:06de8ef6331f6e822c28d577dc8bf43fe398800477c49498f38fc38b67ff33fc \
    --hash=sha256:0764a13d34cae40db7bbf3a09b7e9b491bf4603e20b263a7a9d6b8e324975d0a \
    --hash=sha256:077293e425f28ec737dbcad442a71752e28f8ae27cde3d68acd1fb212091cd92 \
    --hash=sha256:0930db9bdc62d22944e10b066448bb65dc9abe9112880c7cab8da54db4284d5f \
    --hash=sha256:0cee7cb0f9a1b6892ea482237d9403b3d1b4603aee057d0ff01f0fac2d019a97 \
    --hash=sha256:0d9c47709875fdb321452056622e930c52afbc07a7d780762fbb8b4d91ce6fa4 \
    --hash=sha256:11935df9bf455ed0c04eb87bcd720f02b1fe5e02128a9430f23aed6f93336fc7 \
    --hash=sha256:12a606a492de952afcb43b59a14aaaaad120e708d3663dd0fdf2d738d427a691 \
    --hash=sha256:14bd2d845d62ab678eaf81da89d7b621b51756c72346745c1a594c09d49207a2 \
    --hash=sha256:15ba9e28640feef770374b116a6f019c21f52404aeabe516aa7f800587b98cfc \
    --hash=sha256:18a801868a884f216e784d7d14db2a4077143ce7610440aee2ce8f734e7cfcde \
    --hash=sha256:1c0df495a977d10460a94941799c72d5b5ab03d3858d949b55b5a66c8f371c99 \
    --hash=sha256:1caa2fa5a6184fb233153b35f654e6687bd555476f6170f29d8ee9be1a8b0af9 \
    --hash=sha256:1e1451fab512d1bcc3dc26988ec1edb0b82c2db909132872cd9356070a6b63df \
    --hash=sha256:1f1f9477e174582b0a1b583d60b66e1f2cf5d3fe12cee985e4aedf44766600e5 \
    --hash=sha256:2628d3a8cb648ecebb3c5d6b0a1052d400e4d8b7ac0fb786be8d285b50040d17 \
    --hash=sha256:26e9867520db70d37f7fb421a7f0d8adb40171011fb84ce869afa1a83370dfa8 \
    --hash=sha256:2a6ef68ae94aed8721934072b27a3b654ea2100b97e4ab864cf1489c90926fbc \
    --hash=sha256:2b2b1e18af909b448bb3cf9e3433366f7a8726271fc214e8b10e0f62a78c724b \
    --hash=sha256:2cb3dd71fc6be918ad4264346a8ed69485f9b7ed7bf35495d8e22807cd6b8bea \
    --hash=sha256:2d1b7d9308288661f56672b1b157d75fc536714d3638487bbea17b6318a78248 \
    --hash=sha256:2dad610540cb2e6272855c178f08ae9a1c7ac258a7fb71660553a5f104b42741 \
    --hash=sha256:2e5a7cd7fdd14fcb1ae5d7d8bf23d24fbd1daefd1fbca2580132e1ea75f098b5 \
    --hash=sha256:2e9ad7dd851bf45fab9f75cbff4cb493fee9979e8d8c7c9c3ee119022518edd6 \
    --hash=sha256:340cbb1957ba99929cbf19a75626d36ba1ae21d1730b287d1cf7f824a20c4fc7 \
    --hash=sha256:34bdde374c5932765d7dc685c4a1d191a3207852d67e8e0a9eb6ea85156181f1 \
    --hash=sha256:353bd63081912ab8cfa6a0c7d185934cdf8426f04c618bba6bc4b394f2069b67 \
    --hash=sha256:387d8cd30e69b3f0a72877b9ae717033396404e19095b17fe89753a981fda44f \
    --hash=sha256:3882fb412298575bae3b9c46868251f15cc69307359f87bb1b382e53d6e5a2c9 \
    --hash=sha256:38fc55594dab834470b6733dead2ee9e3f657fb0608c769dcafa0ba5ab52f45c \
    --hash=sha256:396ec4e65cc889f69786b3b89478b471cee5a3bcf468b9d9bb03e1a30fb291fc \
    --hash=sha256:39dbacefc411633db5b4378b066a9aca70a3d7e2922c9e578d825f844026eeba \
    --hash=sha256:3a93d9616ddecfb393727a0041a562cf0b15a244e20f2bd25efc7949be4c4f17 \
    --hash=sha256:3d23795802fc8bd72534836d64489bbf0f67c088959091bdb22e10735a5107bf \
    --hash=sha256:434139499bb20b502ed3baa1f169e618f924a97e7a777fea1a49446d80106cf6 \
    --hash=sha256:436e3ffc6310d3c41878c601db29098102fe5d8a467c49da4a4125254e0980f2 \
    --hash=sha256:489505b03f692c3f376394e49194fa7a7f9e8558d6e293a7056a0032b0c38163 \
    --hash=sha256:4a540e2d3192792fc84eced57bef37851ccb2b41f73291bb17408eea77bcd278 \
    --hash=sha256:4a7cdc2a420ca01058182da4253329764d4bfa055564d1eced90e6ba1e8b1d3d \
    --hash=sha256:4bced6e2a6dba6a28f7dd3c6ce14df1b2dd495923f16ea484cad03decd463b2b \
    --hash=sha256:4cf3468d5ec187ffffcaca8e61929a37448f215dafc1386a12c750a72fe53634 \
    --hash=sha256:4e2c4809c14559aa7ef426f27fb35afbb38104c349a903bf8f3600456764bb38 \
    --hash=sha256:4ed644d75aa94a2baf7ec3a96eaa160ea58c742eb9d27c6506053c5c40fc84ed \
    --hash=sha256:4f6e0852a0283b1b1fd776eeb7b766a5f440b3e2bd31ab51af3b400585f3965c \
    --hash=sha256:5066b244f576f91afc8ee3ba029a89f99d39c79b1853fe9d39bea9f0afbec148 \
    --hash=sha256:5086f9975abb1ab531ee6afca1761e4b59a19b446f3f6522ed776963228cfe5a \
    --hash=sha256:50b5bedc9ed8a94fc8857a42ef4f84a81ea88f8d4f05dc8705fb23ee6d8dcca7 \
    --hash=sha256:52704c5d36eb6dda8866493decd61111fff86244c9b1ad225ca01b9e91e5970f \
    --hash=sha256:55ffd6ce583d97dc71dc92e930324c8c0d25aea7e3ade6ae54ef77cedb096811 \
    --hash=sha256:569d65055d367e3dcdf30c3f41119467b73d9ee9faf332bdf40402644f5ac08e \
    --hash=sha256:57f9947a7e57a081c1e3e0a2dd0d2dcf290a4531450e6f611e30084c222a7295 \
    --hash=sha256:5989cb26b2e1efc6a42216a9f6b5ee495ce5ace2e5b352a9af489976b32d1ee2 \
    --hash=sha256:5c22873ad1f0532ba40fa1727f3c0fc1bbbaab6d373d4cbe3f0dc74b2e2521c7 \
    --hash=sha256:5e8b3d0b18fd623afa12ecb2ce8d8becef69f9b5440c6330c7972200e0bb84b0 \
    --hash=sha256:61631e08084be9e21a8967ec3139c7616ed7c5e9368e05c86d1b39562c8a57b6 \
    --hash=sha256:64511c54db4e4987aef4c41923235927428729e8174c5dba488429be70a998ed \
    --hash=sha256:6669c1bf34080161ce49c589cc512ef24d4c704ac9d2b2d3667f519c60418378 \
    --hash=sha256:672d207103e6b16ca098611b0f9efad6bc00afd47c03d6ef62186495ca677dc0 \
    --hash=sha256:6768d67d1bce64270e0fdc2e69309d68b9b18ae56ddf6c711d168e9d051c2cac \
    --hash=sha256:6a45c3d514f2436064db00d7fc8778d888f0236ebfed649b53d13a59e69ad51b \
    --hash=sha256:6bd9e1788e15bfcf6a9082de42e30387e7b85d211ab21e57a939bb8cfaaf8d96 \
    --hash=sha256:6d2a9efe686f9de00d0d1ea32a4a5a86d558a2277501bd78d964214eab625e59 \
    --hash=sha256:6da83a088f8ef93b2d483a8232a4dbf4d69d3d8496b568a03c56becac43e1808 \
    --hash=sha256:7018d4af1cd272e847aa5917983ab5e83e4f6579f9dbfecd4a79c0ca80b144c2 \
    --hash=sha256:71f88e749ea29f67f21f3b36433c1dc54c7729ed2a6d9e2da2e0d9e0d7b224eb \
    --hash=sha256:737c9c3981998eba27f11786f84fddcbabc74068b72a4a1f454ea02094b57b65 \
    --hash=sha256:73e77980c7207854f00fc4e71fb1626868d5740ab4012623d55c7a99ad122a72 \
    --hash=sha256:799c39bdf5e2f1292fedd3009f7b3c9e760f10b2420cb9638d56920840ff6db8 \
    --hash=sha256:7a83aa6e4805df46fed18e989d3d16f86ef60cb50bbc8d9ce3a6be89165fbf6e \
    --hash=sha256:7d3391b2188d18737cb2fa147028b1096236eaa7e156446c650a489fa2cadc91 \
    --hash=sha256:7e1636da3d8dfc220b6dd10264db5f2b165e4888c4518594898fbe381049af8a \
    --hash=sha256:805c8b84534fa10891890f0e4be39f3a99e94615d93e8836bf9fa1fdca2feeb2 \
    --hash=sha256:811d02d5122171c1941357efd8f9bf4ffe907b7f0a1a4e729a880e4be3f46e3e \
    --hash=sha256:8138eb83940ec7299024d92d4dee45f601b9e6c5ffde9d25f4e35e326203c707 \
    --hash=sha256:83b3944fea42a8400edf92fd1770fb8d0d4f7de651353bd2d8525a92dba69a21 \
    --hash=sha256:849dd2bb0e5e4ab2b71c7191726a4a8d5aa8a610daa584728cbee0b710ddc4ef \
    --hash=sha256:8698d70a8081ee8c090dbb394768b5789a1da8b131b5499f89d071dd3cfaf6be \
    --hash=sha256:8781a792a070cf2bd1b86d3aa943894115faaba6e88122a7bf32d62072742453 \
    --hash=sha256:88d59b473bfb03259722600839af9bbd7fa13a2eb514beefeedb95997882f69a \
    --hash=sha256:8909c2f1c6dd65e054ac4b573a91c8384d1492281e55d82d159d653f7a13adf6 \
    --hash=sha256:8965520ac587c94a4ac48b729be3d8b8de00af39699b17585dfb599babe77977 \
    --hash=sha256:8b5d563170ff8ba3181caa967c99a3c804d1dedb702c7cb93a6a7c32247da978 \
    --hash=sha256:8e124f974786f831d6043728e38296969d3579db8896fe004682f5758e613581 \
    --hash=sha256:8f0fac8b13d14bb06c68195f849371924ae53dd7b1c00fed24650f704383b692 \
    --hash=sha256:9240187afb63d2f9ddc3e032c670356fe941f6e20662ea168a5dc3f1f317e1b3 \
    --hash=sha256:925f929d6b59a8b3f8b8c6ac363cd0af7eecc81efb3071770b3c6717c450a369 \
    --hash=sha256:9348cbb300d224fe3b89793262cb093504d4ae927004468463f745188a193e4a \
    --hash=sha256:9388003072b95f2f1e3fd908604194d653ba21330d811961a78b7da1a77e9e36 \
    --hash=sha256:9438a2648b2195980cb2dd8e53ed7b8df91319e2d0b70ae61a9e1d1bc8d3bec9 \
    --hash=sha256:94e4c421742086aeee4c32a506eec8859d7634aad943f7e6aacf70f813478768 \
    --hash=sha256:94f5407f7bc64fa6463906b896f9904beeeb7dd8dc116ee8e9056c8714ff9916 \
    --hash=sha256:971a3bbb75d97ae4e2e8f7d4834236f86f85f0c85e04ab2e191db1123b04f80b \
    --hash=sha256:9e227f3dbe6bde7491cf0a9965d00b88c6b1a4a95d11480ddf88bb96d397c19f \
    --hash=sha256:9e25feb9e330b63edb0278a0acdf85e50d0cb0fbf49c3084abbe4e24ae195346 \
    --hash=sha256:9f098115c247e11d138ab83a28fa0323c77015007ea2df73ba5fd714dfefd67c \
    --hash=sha256:a18f38cafc329bac5e3c2b96c765b4c96d3d103421ed22ab7988c1e3fce27464 \
    --hash=sha256:a4bbd2d87dd233b9fc5812160c3d0ffbe42edc22a26ce0469f58479ede633fe9 \
    --hash=sha256:a5fcffb37e602b0b3c1638a97746b9b96125caa9bcf6fa41d337a9261de231ee \
    --hash=sha256:a8e9f292fcda89b324f2f5c91d13f1424a153e40fc2756f38ee23b15835ff300 \
    --hash=sha256:a9f54054101545a9a9cccefddf54316aa6e4491611fcbef9e91b3b6bebec04f6 \
    --hash=sha256:aa2c838cc024642cc04c6854232f32b43e5e22833dd11119c1766c7873b8370d \
    --hash=sha256:ac0c7c9f1609b0c4c114feb1d7a3409564c7fb77e360bed9e97e5d25dfeaf868 \
    --hash=sha256:add96447a86d205ab616665d53b2950ee81083757f56e6ea833c8b2917646b46 \
    --hash=sha256:ae9dcb8fbe244cb82f8a6458b455b927a03685e383d9bacf1ea5ce180b96dc97 \
    --hash=sha256:b4a635a0487774f841cb1fb62e907e7195cc95bc761e053184b8acc3ceb20733 \
    --hash=sha256:b4d12837e0203bbace818ff4a7461afdcd78bcd782351cea148139180d7bcffe \
    --hash=sha256:b61687d0828e72bf5cda24a2690188f37170bd31c9359ac97e4e66569f120a16 \
    --hash=sha256:b807e598953730f82e4eae3bd30f6a122cf6b31c398c6b504c0e04c13c170429 \
    --hash=sha256:b8cd1f918b26fd7b1832ece557cc18f2d8747309ff8b3f0ef9d4250c5ad67a39 \
    --hash=sha256:b91cc9d336957239ff200f30097e6fea2dc6d6fb3c81e853eaa09eac904fd894 \
    --hash=sha256:bd3ce56ae2cbae3ba82b683bc425cd7e48d2ed8b10f3e818186b6f5646d9271c \
    --hash=sha256:be6cb0c799abb0e2ba3e618e6d28ddddf7e485f6c2ce938dfa237daf3905072c \
    --hash=sha256:befb4158af32106b9a93db8d6d1d1cbbd418c0d5aca0cabb7b1780abf0c89169 \
    --hash=sha256:bf053da3c97a4bc5ecfbb218cdd2983febd91c617be8367d139882aa11e490aa \
    --hash=sha256:c02e8f18bdedba082cef725942ac823b9b60656db07f7e265cb31618dfd00d77 \
    --hash=sha256:c1bc67752d5f21013cfe430df4062441714eab79f65a6a05e01505957e9c35fe \
    --hash=sha256:c61750fadcd119d0825bcb7d7d675dd264dcc89cc05292aab5be68ebdbb374ad \
    --hash=sha256:c90d5b3d4e944e065a301d741b3c1d784f6bd1f503aa68b4967e32b2ba313d85 \
    --hash=sha256:c9a7f43c0b202b334cc9184af09bb8f21d3a209e038efaf106936fb69e6b026e \
    --hash=sha256:cb96e6e088d6cf71c1ea977510948320234824cf226e32f6f6e044f7a9c82b34 \
    --hash=sha256:cf63c214fe879a65e69a386f915e36104fc84254ab141240f8854602d8e0be2a \
    --hash=sha256:d1aca03ede943eb80ab3d63bb082c84b7aab85ea83bd0fd0c200260945fb49d9 \
    --hash=sha256:d2e56fd3b00222722abfb3f5f0759ddbae4b90811b5ad4343c64030ad1bde70c \
    --hash=sha256:d5f93ebbeb8032d47e349328ec8662d973d9b05a70b3c35df1f91fe419b84749 \
    --hash=sha256:d882a373d8093c2941e01291b7ced96e9cbe4781da9a7751ca7e6c70385e5214 \
    --hash=sha256:d920abdfa61279ba1a2ef9484aab07bf03331f8c08a10120fa332353d06e6932 \
    --hash=sha256:da2af0d7aebfc2074080d72efa6ab8317c62481ef1f896f65d9999c1c01f4494 \
    --hash=sha256:dd8ea6ebee7aedbf7c749fa80521d9ccf1ba473e0d1e14805caafbaad281c889 \
    --hash=sha256:de8b364c423ef0a4bad9069657d617f9a5d2b2062457a89b1fa16ee199c399c1 \
    --hash=sha256:df1ae86ff54725a01fa1a0510b914ca53a161b7050be74f6204e24aded5971d0 \
    --hash=sha256:dff05cb7016dff1e9fd68f4122c127b65dfc59de5306cfb7ad92f956f230bee2 \
    --hash=sha256:e1a622f13970d81f95d0c72f9dc090dce9085fccfa4c9f2174377ee32bd15786 \
    --hash=sha256:e49fb0d1ce92cfa0cb198cc5b1b11cdf9d0638658e2a2db2687e39db7c87fc78 \
    --hash=sha256:e5c802729725bd07e2bc3ab7b76dc7e0bbfc53129d8f1eb1c002c24cf774717e \
    --hash=sha256:e841068dc0be4cb6dfb5c890eb88cbdcff2f4a332393c7ec94e8e618bd32c1a8 \
    --hash=sha256:e916035e3e9930cbdfdd10abf48861340221857f45509565898e012263f7b289 \
    --hash=sha256:eba154571c16e032112afac0dc2dfe9e63c2ceb7aedd07bb7eecf2ce26d4dd4c \
    --hash=sha256:f03460ff076f70ab595bb45a0205ccea1971443575b6920c52e755dec2b3fbfe \
    --hash=sha256:f0ec3b750b59375eab5b0fb2b9254810c00a3375be6d789899f1055a1d556237 \
    --hash=sha256:f291bcf42ae98eb5107edb162c3c998b4a89648fd8e99ed4cbd12705292788cd \
    --hash=sha256:f61efe1d2fe0de16158a5fe1d1cf3c14bdb6aecd54d8938fd26512c525c1f624 \
    --hash=sha256:f68edfc67aabac33708941f26f22a7b8e9f81429bc0cf249fcf7d66b23af8d19 \
    --hash=sha256:fa95848c929b6a75f6848d3c9793e59db365ee436776e57db835cdbfa79ba977 \
    --hash=sha256:fd9f8797427910198f95bced71ddfed61130d7e349213bfb8466c9c99e2c46a8 \
    --hash=sha256:fdb4ca07ab75ffadab4a8b135ad59cdbb3156b99310f3d565370da74a15d6bd3
    # via jinja2
mpmath==1.3.0 \
    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \
    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c
    # via sympy
networkx==3.7 \
    --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0 \
    --hash=sha256:fd77a511bd90f39f3d016351345b52cf5319b813bdca01de3f755d3cca62e96a
    # via torch
numpy==2.5.3 \
    --hash=sha256:012e66aca395d795496446e52aeeb5866312a5d4d3f27da270e5a0b43f70dc5c \
    --hash=sha256:09d5a423c71ad5feb5625844ad58050e35df43871004b52ac9c0ad44a56775be \
    --hash=sha256:09ffa5d903faeaa5c4dd05009cf81c8bab9f2cb37c548b8d39b65b4cfa7c97f7 \
    --hash=sha256:0a59a421a32580a009e8a1751345bf829631b990dc1794b80514ab722b435def \
    --hash=sha256:116f96cadd935c6122e9228d676fe7ede19e741f5c8bb1c3cddbe0c51ccebea2 \
    --hash=sha256:1302b90c0e52281681b2975adfe8a860cb7b12216a27b4b0b4207c44bf7bccf0 \
    --hash=sha256:15aa985ac73a8db02db7663381aa109510449d3819d37206caed27b33a65a8a6 \
    --hash=sha256:1aad64d99730d013cfc6debafed22783b4fc5a7f4b8bc744d2d8cf7dcc880551 \
    --hash=sha256:1c80eabb4035ecf4ca9cd49cde8a9fdd69a729e63e6474887d1523ade7aa277f \
    --hash=sha256:1f3ed25271581281f2fccb1adcedfcde4c07362eec69189b50baf6f90e3ae159 \
    --hash=sha256:1fb6f8fb9ff0b3a69f52c66ce397b0246583e9f28616231b0e32ca49259a5fa6 \
    --hash=sha256:214045a5bf00113a146ab9ee9730c44501af6723cdf1f6830932f7b5ef2e7af0 \
    --hash=sha256:26e15e4aecd8617dfbaecb37d223e365d7b39411fba20454be2670a96aa74cb5 \
    --hash=sha256:2c25dfa72943e4336ddb6b0ee4277b47a0c85bede0807530ec68103bf58e2c10 \
    --hash=sha256:2d8240cb4c16fd831074aa2b2cf9fc54664d826341d61c372245b96a74a49a9a \
    --hash=sha256:350ba9783ce969cf9f7ce6e6a9a58e1a6e2a19ca025b7ee448c4db727706212a \
    --hash=sha256:4c8a6d2ebce6305fd82fbefca827775437147052a976ee7c94b36a0c1b52ac6c \
    --hash=sha256:4f8929ee6c96bfbd7b4ed2032e0c03af86fe1826740ab61ddabf9072d06e57ff \
    --hash=sha256:536f963710a4e63934d80ac0dc4f478804a83e9a84b6828018f25d09953ada33 \
    --hash=sha256:54a115e5a73b8fc44f0cebef486365a1894b5c9760685d4558b72b7c3eb846e0 \
    --hash=sha256:595d020938c84e320bcf40ad71089e108eac0d377cd018e14a8c094f39e98d85 \
    --hash=sha256:66a78fe4556c60aceda5916f9eacd638b18e9e681016ec302dcb4682d6d4d034 \
    --hash=sha256:6b05c171afb3aa07adbd20abc00aea86fe375beb0fdb9ef780ec5b7f63bab1c0 \
    --hash=sha256:6cef4bb1706dfec49243c05d921eefb4e190d41e2528b30d8035ea1f36b4c24a \
    --hash=sha256:6f24021b9f22bc6301c37b196974a92c1c18dccedb6fef3dd252e95f2d6adbe4 \
    --hash=sha256:71b39d9f935b6ec0f8753e3e2afb51e3efba6f2e05b68b32a40754d24bcd4a3c \
    --hash=sha256:71cad2b2a7451ab79d8f5e71b453485b6775963d5cf794179144a7463fe6e8ec \
    --hash=sha256:76c2c1e6bfa5c84adc6434dfbf013aa92096a7985221762c8f11fedfd20fff58 \
    --hash=sha256:8617bbfae4486cf99c9f899966699428d19da931d06ca94ad3da986c76e15997 \
    --hash=sha256:86bff898a431c0fb71f7610b75726e75a54d47b37edc9d537f48de63bb3c0b90 \
    --hash=sha256:8e4dd766076855b5ff7ea52fa5f07ce26286726e0f8bff446b7739d02e6ea204 \
    --hash=sha256:92f30e89b8ee0ecf363033576c422b2f58fed6a80bed0aa48dff6d14c654663e \
    --hash=sha256:93e1f5447e2b1e479d7bd74701e84746b86450cff1fc368b132d195e2b8f8211 \
    --hash=sha256:9a37475425b431b4d060f23b4f52cd2f3aef6bc7c654bd760adf0040eec9d435 \
    --hash=sha256:9deb49575e5b0b94ed72c8a64ec4d033381adc27e9060ae842971f697ba96104 \
    --hash=sha256:a5fa86b80fd24bcd1aff83ad23be44ea323de3f787be8f8b15d4a65621e25321 \
    --hash=sha256:a6391fafaba97500887132cd582abc6e19452b1ac775a47caa7b24490e152058 \
    --hash=sha256:a72f874bc9e10e4b8f80426fb49716d5141f64442a0c8418065093ec8017fbb0 \
    --hash=sha256:ac7bb1c52d445bd4f8f7f97fefe6abc3a084dc4d63df50d79b17fa2b78e89297 \
    --hash=sha256:adc1ada2662f8a5f960b8a10d9986897e7499ef07e06d4cfe7197f8cce923c07 \
    --hash=sha256:b00eefbcf0f292945c4b4dec2ae845389ef5bcdcd596e6e4328051db5b5ba694 \
    --hash=sha256:b0521d0f4aebb6e06189451025fa17a913287b13c03d5fe05c017333b654ea5b \
    --hash=sha256:b5d93cf48f687479941d12b69c873ad2cc76bbd487f0091c2200636497f34034 \
    --hash=sha256:b7e18c623bb5c95acb3b3328861272816ba199fb531921c5d6d0b675f1fde9e3 \
    --hash=sha256:bd4cb9ad3c7889b9b3fe0a9a9fb5d2ed26f9879bff2608d9f01aed147a20d231 \
    --hash=sha256:be5a8381859b6da607c84f4f7d6847725f1cf1853ef8a2c9e115b7d58bef47dc \
    --hash=sha256:befa1ae5bd6030b3f512b43ff3fa5290bbed6b84411a44244b14adf835f5b89d \
    --hash=sha256:bf63afbe037eb5d2fe87fbcc7778e61da53ebaf21d938a4515aa73b62532a5d4 \
    --hash=sha256:c00abe94c1a69d75d827dcf1c025b25c8a45d230b3bcd77a9020883a1b047653 \
    --hash=sha256:c2381f82999704f818e2c987a865050e285ec3621262c66d40f5a96c8f899f8e \
    --hash=sha256:c76d5dde9f445058f83d0c02af00557a4db91de9a9a57c0df87d1535001d654b \
    --hash=sha256:cb189f09db39283b26bfd061ec16189e14f71c6755207f72a0f7540867afe5b9 \
    --hash=sha256:ccb32e0525d29e8b0572eb84c9a57af0e7a4e615726927506f55063c62414034 \
    --hash=sha256:ccbc4665079665c3cf3bab4db9f6b095370cd6437d66be549b6c2a1fd19e1958 \
    --hash=sha256:d1c89973648c85069c5046ad460f7b8a00218b29a2e42359ac8cc63e9ab94832 \
    --hash=sha256:df2d5874ff183595a4ba404edd04f6bd9b5505c1d7708573f6a6c17489a67563 \
    --hash=sha256:e01c918ac3d48e18a927cf7b14a26a3e29ff2bdf2eacb976da0aecd6a43ed034 \
    --hash=sha256:e6ab667ba76450084eb64013762c438ea76d9d29cc676dcd6c2e9892ba37f841 \
    --hash=sha256:e931e4f499e0dc7ef29d269a8e5b35dd722e5d14be07df6240166ea7c6532fae \
    --hash=sha256:f54660b0eb6b0b9f36e7fe1cdfdff472028dd0d14acd9b9b65098efbad059469 \
    --hash=sha256:f59a878c33d6b88122d80d239bb3b845d58708750b0cb06a09aebb9b18ec696c \
    --hash=sha256:f7fabeb6cea87d65f3b926de33d03fb016cfdc29314c90974383b5582ae72891 \
    --hash=sha256:f9579f383d1bf9df80081e72760e84960a7fd4f88cf0c9e535a8597c9bb646f5 \
    --hash=sha256:f9a2353b37a1a9e78fd82b27ad7e2a32a2d036604d18f02b05e3136c62ca3b09 \
    --hash=sha256:fc36dc566135b5eceec4cf89758fcb719266a019ef07dae1754ae7c9f617ef3e \
    --hash=sha256:ffdc76bfcae6b255dff75202c5e7feaf95b40246bc0a17944facc1fecf9f79ab
    # via
    #   blip-captioning-pipeline (pyproject.toml)
    #   torchvision
    #   transformers
nvidia-cublas==13.1.1.3 \
    --hash=sha256:37936a16db8fe4ac1f065c2139360608a543a09275cb1a1af612e08cfa065436 \
    --hash=sha256:b6cdce694e47ff6aadf0a69df1cab6628d696f5ff56e8d16af50309d855fa20f \
    --hash=sha256:b7a210458267ac818974c53038fbec2e969d5c99f305ab15c72522fa9f001dd5
    # via
    #   cuda-toolkit
    #   nvidia-cudnn-cu13
    #   nvidia-cusolver
nvidia-cuda-cupti==13.0.85 \
    --hash=sha256:4eb01c08e859bf924d222250d2e8f8b8ff6d3db4721288cf35d14252a4d933c8 \
    --hash=sha256:683f58d301548deeefcb8f6fac1b8d907691b9d8b18eccab417f51e362102f00 \
    --hash=sha256:796bd679890ee55fb14a94629b698b6db54bcfd833d391d5e94017dd9d7d3151
    # via cuda-toolkit
nvidia-cuda-nvrtc==13.0.88 \
    --hash=sha256:6bcd4e7f8e205cbe644f5a98f2f799bef9556fefc89dd786e79a16312ce49872 \
    --hash=sha256:ad9b6d2ead2435f11cbb6868809d2adeeee302e9bb94bcf0539c7a40d80e8575 \
    --hash=sha256:d27f20a0ca67a4bb34268a5e951033496c5b74870b868bacd046b1b8e0c3267b
    # via
    #   cuda-toolkit
    #   nvidia-cublas
nvidia-cuda-runtime==13.0.96 \
    --hash=sha256:7f82250d7782aa23b6cfe765ecc7db554bd3c2870c43f3d1821f1d18aebf0548 \
    --hash=sha256:ef9bcbe90493a2b9d810e43d249adb3d02e98dd30200d86607d8d02687c43f55 \
    --hash=sha256:f79298c8a098cec150a597c8eba58ecdab96e3bdc4b9bc4f9983635031740492
    # via cuda-toolkit
nvidia-cudnn-cu13==9.24.0.43 \
    --hash=sha256:67a7273b5cf062f9446fd76cf464351a1c0f66501e6cd78f6675c0d604d8ac87 \
    --hash=sha256:71f181cd810e90f9b6023b01186fe82d13d65f0ec098581ee201d39fad769e4b \
    --hash=sha256:a6812a554a1ff0413e9c52b84c26c050380649ab9615f9c16bded368ce9f421f
    # via torch
nvidia-cufft==12.0.0.61 \
    --hash=sha256:2708c852ef8cd89d1d2068bdbece0aa188813a0c934db3779b9b1faa8442e5f5 \
    --hash=sha256:2abce5b39d2f5ae12730fb7e5db6696533e36c26e2d3e8fd1750bdd2853364eb \
    --hash=sha256:6c44f692dce8fd5ffd3e3df134b6cdb9c2f72d99cf40b62c32dde45eea9ddad3
    # via cuda-toolkit
nvidia-cufile==1.15.1.6 \
    --hash=sha256:08a3ecefae5a01c7f5117351c64f17c7c62efa5fffdbe24fc7d298da19cd0b44 \
    --hash=sha256:bdc0deedc61f548bddf7733bdc216456c2fdb101d020e1ab4b88d232d5e2f6d1
    # via cuda-toolkit
nvidia-curand==10.4.0.35 \
    --hash=sha256:133df5a7509c3e292aaa2b477afd0194f06ce4ea24d714d616ff36439cee349a \
    --hash=sha256:1aee33a5da6e1db083fe2b90082def8915f30f3248d5896bcec36a579d941bfc \
    --hash=sha256:65b1710aa6961d326b411e314b374290904c5ddf41dc3f766ebc3f1d7d4ca69f
    # via cuda-toolkit
nvidia-cusolver==12.0.4.66 \
    --hash=sha256:02c2457eaa9e39de20f880f4bd8820e6a1cfb9f9a34f820eb12a155aa5bc92d2 \
    --hash=sha256:0a759da5dea5c0ea10fd307de75cdeb59e7ea4fcb8add0924859b944babf1112 \
    --hash=sha256:16515bd33a8e76bb54d024cfa068fa68d30e80fc34b9e1090813ea9362e0cb65
    # via cuda-toolkit
nvidia-cusparse==12.6.3.3 \
    --hash=sha256:2b3c89c88d01ee0e477cb7f82ef60a11a4bcd57b6b87c33f789350b59759360b \
    --hash=sha256:80bcc4662f23f1054ee334a15c72b8940402975e0eab63178fc7e670aa59472c \
    --hash=sha256:cbcf42feb737bd7ec15b4c0a63e62351886bd3f975027b8815d7f720a2b5ea79
    # via
    #   cuda-toolkit
    #   nvidia-cusolver
nvidia-cusparselt-cu13==0.8.1 \
    --hash=sha256:4dca476c50bf4780d46cd0bfbd82e2bc10a08e4fef7950917ce8d7578d22a23f \
    --hash=sha256:786ce87568c303fadb5afcc7102d454cd3040d75f6f8626f5db460d1871f4dd0 \
    --hash=sha256:dccbd362f91a7b9024d1f55ee9f548ac065027ff15d8c8b0db889ab3a8f31215
    # via torch
nvidia-nccl-cu13==2.30.7 \
    --hash=sha256:ca786ffa5a647c75d4d1f5cc72a6c4f537947e2ba8823d7c8aaf768e7a7b9f77 \
    --hash=sha256:cefa7fdb9710efd0f39c5f1be1d61ff6fc9a996c451265bd7fbdcf9455ed4b50
    # via torch
nvidia-nvjitlink==13.4.92 \
    --hash=sha256:25f74fad0d654271c921ac4dca614bd6258bc21791242fc7b2289dad7ae9c099 \
    --hash=sha256:9e4a7ff4f0cafa8c624917055b863dc11f5c2912ead23c462889e166f3b0e57d \
    --hash=sha256:b286f3a4f227a9363efdec263c7b91788cef1478d2b8a5fa8bab7f3e82ff82fd \
    --hash=sha256:e0391f24ed94ec879b84e3da4d4ec320c879aff681f2c7a638462f7199284323
    # via
    #   cuda-toolkit
    #   nvidia-cufft
    #   nvidia-cusolver
    #   nvidia-cusparse
nvidia-nvshmem-cu13==3.4.5 \
    --hash=sha256:290f0a2ee94c9f3687a02502f3b9299a9f9fe826e6d0287ee18482e78d495b80 \
    --hash=sha256:6dc2a197f38e5d0376ad52cd1a2a3617d3cdc150fd5966f4aee9bcebb1d68fe9
    # via torch
nvidia-nvtx==13.0.85 \
    --hash=sha256:4936d1d6780fbe68db454f5e72a42ff64d1fd6397df9f363ae786930fd5c1cd4 \
    --hash=sha256:cb7780edb6b14107373c835bf8b72e7a178bac7367e23da7acb108f973f157a6 \
    --hash=sha256:d66ea44254dd3c6eacc300047af6e1288d2269dd072b417e0adffbf479e18519
    # via cuda-toolkit
packaging==26.3 \
    --hash=sha256:94edc256424af38762eb31306eed28beb9f0efc50a8837492c9d6fd6004aed79 \
    --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c
    # via
    #   huggingface-hub
    #   transformers
pillow==11.3.0 \
    --hash=sha256:023f6d2d11784a465f09fd09a34b150ea4672e85fb3d05931d89f373ab14abb2 \
    --hash=sha256:02a723e6bf909e7cea0dac1b0e0310be9d7650cd66222a5f1c571455c0a45214 \
    --hash=sha256:040a5b691b0713e1f6cbe222e0f4f74cd233421e105850ae3b3c0ceda520f42e \
    --hash=sha256:05f6ecbeff5005399bb48d198f098a9b4b6bdf27b8487c7f38ca16eeb070cd59 \
    --hash=sha256:068d9c39a2d1b358eb9f245ce7ab1b5c3246c7c8c7d9ba58cfa5b43146c06e50 \
    --hash=sha256:0743841cabd3dba6a83f38a92672cccbd69af56e3e91777b0ee7f4dba4385632 \
    --hash=sha256:092c80c76635f5ecb10f3f83d76716165c96f5229addbd1ec2bdbbda7d496e06 \
    --hash=sha256:0b275ff9b04df7b640c59ec5a3cb113eefd3795a8df80bac69646ef699c6981a \
    --hash=sha256:0bce5c4fd0921f99d2e858dc4d4d64193407e1b99478bc5cacecba2311abde51 \
    --hash=sha256:1019b04af07fc0163e2810167918cb5add8d74674b6267616021ab558dc98ced \
    --hash=sha256:106064daa23a745510dabce1d84f29137a37224831d88eb4ce94bb187b1d7e5f \
    --hash=sha256:118ca10c0d60b06d006be10a501fd6bbdfef559251ed31b794668ed569c87e12 \
    --hash=sha256:13f87d581e71d9189ab21fe0efb5a23e9f28552d5be6979e84001d3b8505abe8 \
    --hash=sha256:155658efb5e044669c08896c0c44231c5e9abcaadbc5cd3648df2f7c0b96b9a6 \
    --hash=sha256:1904e1264881f682f02b7f8167935cce37bc97db457f8e7849dc3a6a52b99580 \
    --hash=sha256:19d2ff547c75b8e3ff46f4d9ef969a06c30ab2d4263a9e287733aa8b2429ce8f \
    --hash=sha256:1a992e86b0dd7aeb1f053cd506508c0999d710a8f07b4c791c63843fc6a807ac \
    --hash=sha256:1b9c17fd4ace828b3003dfd1e30bff24863e0eb59b535e8f80194d9cc7ecf860 \
    --hash=sha256:1c627742b539bba4309df89171356fcb3cc5a9178355b2727d1b74a6cf155fbd \
    --hash=sha256:1cd110edf822773368b396281a2293aeb91c90a2db00d78ea43e7e861631b722 \
    --hash=sha256:1f85acb69adf2aaee8b7da124efebbdb959a104db34d3a2cb0f3793dbae422a8 \
    --hash=sha256:23cff760a9049c502721bdb743a7cb3e03365fafcdfc2ef9784610714166e5a4 \
    --hash=sha256:2465a69cf967b8b49ee1b96d76718cd98c4e925414ead59fdf75cf0fd07df673 \
    --hash=sha256:2a3117c06b8fb646639dce83694f2f9eac405472713fcb1ae887469c0d4f6788 \
    --hash=sha256:2aceea54f957dd4448264f9bf40875da0415c83eb85f55069d89c0ed436e3542 \
    --hash=sha256:2d6fcc902a24ac74495df63faad1884282239265c6839a0a6416d33faedfae7e \
    --hash=sha256:30807c931ff7c095620fe04448e2c2fc673fcbb1ffe2a7da3fb39613489b1ddd \
    --hash=sha256:30b7c02f3899d10f13d7a48163c8969e4e653f8b43416d23d13d1bbfdc93b9f8 \
    --hash=sha256:3828ee7586cd0b2091b6209e5ad53e20d0649bbe87164a459d0676e035e8f523 \
    --hash=sha256:3cee80663f29e3843b68199b9d6f4f54bd1d4a6b59bdd91bceefc51238bcb967 \
    --hash=sha256:3e184b2f26ff146363dd07bde8b711833d7b0202e27d13540bfe2e35a323a809 \
    --hash=sha256:41342b64afeba938edb034d122b2dda5db2139b9a4af999729ba8818e0056477 \
    --hash=sha256:41742638139424703b4d01665b807c6468e23e699e8e90cffefe291c5832b027 \
    --hash=sha256:4445fa62e15936a028672fd48c4c11a66d641d2c05726c7ec1f8ba6a572036ae \
    --hash=sha256:45dfc51ac5975b938e9809451c51734124e73b04d0f0ac621649821a63852e7b \
    --hash=sha256:465b9e8844e3c3519a983d58b80be3f668e2a7a5db97f2784e7079fbc9f9822c \
    --hash=sha256:48d254f8a4c776de343051023eb61ffe818299eeac478da55227d96e241de53f \
    --hash=sha256:4c834a3921375c48ee6b9624061076bc0a32a60b5532b322cc0ea64e639dd50e \
    --hash=sha256:4c96f993ab8c98460cd0c001447bff6194403e8b1d7e149ade5f00594918128b \
    --hash=sha256:504b6f59505f08ae014f724b6207ff6222662aab5cc9542577fb084ed0676ac7 \
    --hash=sha256:527b37216b6ac3a12d7838dc3bd75208ec57c1c6d11ef01902266a5a0c14fc27 \
    --hash=sha256:5418b53c0d59b3824d05e029669efa023bbef0f3e92e75ec8428f3799487f361 \
    --hash=sha256:59a03cdf019efbfeeed910bf79c7c93255c3d54bc45898ac2a4140071b02b4ae \
    --hash=sha256:5e05688ccef30ea69b9317a9ead994b93975104a677a36a8ed8106be9260aa6d \
    --hash=sha256:6359a3bc43f57d5b375d1ad54a0074318a0844d11b76abccf478c37c986d3cfc \
    --hash=sha256:643f189248837533073c405ec2f0bb250ba54598cf80e8c1e043381a60632f58 \
    --hash=sha256:65dc69160114cdd0ca0f35cb434633c75e8e7fad4cf855177a05bf38678f73ad \
    --hash=sha256:67172f2944ebba3d4a7b54f2e95c786a3a50c21b88456329314caaa28cda70f6 \
    --hash=sha256:676b2815362456b5b3216b4fd5bd89d362100dc6f4945154ff172e206a22c024 \
    --hash=sha256:6a418691000f2a418c9135a7cf0d797c1bb7d9a485e61fe8e7722845b95ef978 \
    --hash=sha256:6abdbfd3aea42be05702a8dd98832329c167ee84400a1d1f61ab11437f1717eb \
    --hash=sha256:6be31e3fc9a621e071bc17bb7de63b85cbe0bfae91bb0363c893cbe67247780d \
    --hash=sha256:7107195ddc914f656c7fc8e4a5e1c25f32e9236ea3ea860f257b0436011fddd0 \
    --hash=sha256:71f511f6b3b91dd543282477be45a033e4845a40278fa8dcdbfdb07109bf18f9 \
    --hash=sha256:7859a4cc7c9295f5838015d8cc0a9c215b77e43d07a25e460f35cf516df8626f \
    --hash=sha256:7966e38dcd0fa11ca390aed7c6f20454443581d758242023cf36fcb319b1a874 \
    --hash=sha256:79ea0d14d3ebad43ec77ad5272e6ff9bba5b679ef73375ea760261207fa8e0aa \
    --hash=sha256:7aee118e30a4cf54fdd873bd3a29de51e29105ab11f9aad8c32123f58c8f8081 \
    --hash=sha256:7b161756381f0918e05e7cb8a371fff367e807770f8fe92ecb20d905d0e1c149 \
    --hash=sha256:7c8ec7a017ad1bd562f93dbd8505763e688d388cde6e4a010ae1486916e713e6 \
    --hash=sha256:7d1aa4de119a0ecac0a34a9c8bde33f34022e2e8f99104e47a3ca392fd60e37d \
    --hash=sha256:7db51d222548ccfd274e4572fdbf3e810a5e66b00608862f947b163e613b67dd \
    --hash=sha256:819931d25e57b513242859ce1876c58c59dc31587847bf74cfe06b2e0cb22d2f \
    --hash=sha256:83e1b0161c9d148125083a35c1c5a89db5b7054834fd4387499e06552035236c \
    --hash=sha256:857844335c95bea93fb39e0fa2726b4d9d758850b34075a7e3ff4f4fa3aa3b31 \
    --hash=sha256:8797edc41f3e8536ae4b10897ee2f637235c94f27404cac7297f7b607dd0716e \
    --hash=sha256:8924748b688aa210d79883357d102cd64690e56b923a186f35a82cbc10f997db \
    --hash=sha256:89bd777bc6624fe4115e9fac3352c79ed60f3bb18651420635f26e643e3dd1f6 \
    --hash=sha256:8dc70ca24c110503e16918a658b869019126ecfe03109b754c402daff12b3d9f \
    --hash=sha256:91da1d88226663594e3f6b4b8c3c8d85bd504117d043740a8e0ec449087cc494 \
    --hash=sha256:921bd305b10e82b4d1f5e802b6850677f965d8394203d182f078873851dada69 \
    --hash=sha256:932c754c2d51ad2b2271fd01c3d121daaa35e27efae2a616f77bf164bc0b3e94 \
    --hash=sha256:93efb0b4de7e340d99057415c749175e24c8864302369e05914682ba642e5d77 \
    --hash=sha256:97afb3a00b65cc0804d1c7abddbf090a81eaac02768af58cbdcaaa0a931e0b6d \
    --hash=sha256:97f07ed9f56a3b9b5f49d3661dc9607484e85c67e27f3e8be2c7d28ca032fec7 \
    --hash=sha256:98a9afa7b9007c67ed84c57c9e0ad86a6000da96eaa638e4f8abe5b65ff83f0a \
    --hash=sha256:9ab6ae226de48019caa8074894544af5b53a117ccb9d3b3dcb2871464c829438 \
    --hash=sha256:9c412fddd1b77a75aa904615ebaa6001f169b26fd467b4be93aded278266b288 \
    --hash=sha256:a1bc6ba083b145187f648b667e05a2534ecc4b9f2784c2cbe3089e44868f2b9b \
    --hash=sha256:a418486160228f64dd9e9efcd132679b7a02a5f22c982c78b6fc7dab3fefb635 \
    --hash=sha256:a4d336baed65d50d37b88ca5b60c0fa9d81e3a87d4a7930d3880d1624d5b31f3 \
    --hash=sha256:a6444696fce635783440b7f7a9fc24b3ad10a9ea3f0ab66c5905be1c19ccf17d \
    --hash=sha256:a7bc6e6fd0395bc052f16b1a8670859964dbd7003bd0af2ff08342eb6e442cfe \
    --hash=sha256:b4b8f3efc8d530a1544e5962bd6b403d5f7fe8b9e08227c6b255f98ad82b4ba0 \
    --hash=sha256:b5f56c3f344f2ccaf0dd875d3e180f631dc60a51b314295a3e681fe8cf851fbe \
    --hash=sha256:be5463ac478b623b9dd3937afd7fb7ab3d79dd290a28e2b6df292dc75063eb8a \
    --hash=sha256:c37d8ba9411d6003bba9e518db0db0c58a680ab9fe5179f040b0463644bc9805 \
    --hash=sha256:c84d689db21a1c397d001aa08241044aa2069e7587b398c8cc63020390b1c1b8 \
    --hash=sha256:c96d333dcf42d01f47b37e0979b6bd73ec91eae18614864622d9b87bbd5bbf36 \
    --hash=sha256:cadc9e0ea0a2431124cde7e1697106471fc4c1da01530e679b2391c37d3fbb3a \
    --hash=sha256:cc3e831b563b3114baac7ec2ee86819eb03caa1a2cef0b481a5675b59c4fe23b \
    --hash=sha256:cd8ff254faf15591e724dc7c4ddb6bf4793efcbe13802a4ae3e863cd300b493e \
    --hash=sha256:d000f46e2917c705e9fb93a3606ee4a819d1e3aa7a9b442f6444f07e77cf5e25 \
    --hash=sha256:d9da3df5f9ea2a89b81bb6087177fb1f4d1c7146d583a3fe5c672c0d94e55e12 \
    --hash=sha256:e5c5858ad8ec655450a7c7df532e9842cf8df7cc349df7225c60d5d348c8aada \
    --hash=sha256:e67d793d180c9df62f1f40aee3accca4829d3794c95098887edc18af4b8b780c \
    --hash=sha256:ea944117a7974ae78059fcc1800e5d3295172bb97035c0c1d9345fca1419da71 \
    --hash=sha256:eb76541cba2f958032d79d143b98a3a6b3ea87f0959bbe256c0b5e416599fd5d \
    --hash=sha256:ec1ee50470b0d050984394423d96325b744d55c701a439d2bd66089bff963d3c \
    --hash=sha256:ee92f2fd10f4adc4b43d07ec5e779932b4eb3dbfbc34790ada5a6669bc095aa6 \
    --hash=sha256:f0f5d8f4a08090c6d6d578351a2b91acf519a54986c055af27e7a93feae6d3f1 \
    --hash=sha256:f1f182ebd2303acf8c380a54f615ec883322593320a9b00438eb842c1f37ae50 \
    --hash=sha256:f8a5827f84d973d8636e9dc5764af4f0cf2318d26744b3d902931701b0d46653 \
    --hash=sha256:f944255db153ebb2b19c51fe85dd99ef0ce494123f21b9db4877ffdfc5590c7c \
    --hash=sha256:fdae223722da47b024b867c1ea0be64e0df702c5e0a60e27daad39bf960dd1e4 \
    --hash=sha256:fe27fb049cdcca11f11a7bfda64043c37b30e6b91f10cb5bab275806c32f6ab3
    # via
    #   blip-captioning-pipeline (pyproject.toml)
    #   torchvision
pyarrow==25.0.1 \
    --hash=sha256:0b1edbb2f385a6a65e9711b62ba86ac54a7816a3f8d17bb3e8a5929d65fb2485 \
    --hash=sha256:0b726ad7e7b669be982b0c71c07fe4b037d654354130da79a7902a669e93a66b \
    --hash=sha256:0befcf816e45a1af33ac775a9970b749e4868a230c7372f0ae5e932bee27039f \
    --hash=sha256:0fe7c8b6c03969b49c8c66182e4a18e3819ab92d07cfab5d8370c531b9369ef0 \
    --hash=sha256:119297a6dc197e45d9c6d4415f7814a67ffa36c180d26f68c154c58067ae782d \
    --hash=sha256:169d3429d5be7c752125890620f75a60776d38b0035eddae939651640822332e \
    --hash=sha256:25f8720bf6387d5dc2ebd2622112de630760419e4b66134405dd24110d15f37e \
    --hash=sha256:31e49a7888fcdf3a835da33ae777f6bb9a866334e5a789282fc26dcf426f7f15 \
    --hash=sha256:35935cd5de130aa5cf4dea052a63e6bf2e17006c35c3a468194242b9b2bf5956 \
    --hash=sha256:38a9a4b4b9613380e200641891495a56c3d5a98a092db4a870af9975e220471d \
    --hash=sha256:3f89685964f46e4216103c75483aac0c0692a5f72212d7ca835adba5ede56ce3 \
    --hash=sha256:4288f27577352d608ca08553b0865e4a9b3aa14820c5d95b53337218d609835b \
    --hash=sha256:4340f0ba6c1d2e13f21658de1d7c662ca2545018568d0030a1e9afca159d87e3 \
    --hash=sha256:44a9120ce5bd81936b8ab9a88076e3fd47c2c6838e0e43630fed83626aca81d9 \
    --hash=sha256:4facd65742a024a4a366328a1d2292062d72d6e023c1b7dda8d4c37544933a25 \
    --hash=sha256:51093dd9e10325fbdb3c10a2ae7c4806e5c822d94e74ae4938b26524a3323fee \
    --hash=sha256:514ddb60285631af068875550c90eddc181db3e8e63a032b1559be189e82f056 \
    --hash=sha256:5389cdf79447ed1515c9e31620e6e1e2302249564d603f2ad727d4f6d313e4c3 \
    --hash=sha256:59a2de54c0cbd954da861eee4d1d330f8e909c45b53455baef696380f2c55033 \
    --hash=sha256:60e89d8f13861a1f7f8d950fa54aebb8023b30734d0ac51ffa80beabe2df4bba \
    --hash=sha256:6109c94d8b9f3b17a041daca16cacb2f651ad8f1ef70a4232c2c0f37a23da2a8 \
    --hash=sha256:62cd0d785b8aa6675ee355f9fc02252a340f4441257c42674937826fd7594325 \
    --hash=sha256:6943e2fe7954d29d84de45d29d34c8dc36ce96570e67d89aa9976e650a4a9138 \
    --hash=sha256:6a1fdfc6659b6b19022f2e50627fb5cf7156a66c46bf4299379955cbe742382a \
    --hash=sha256:880523be3d29efcf83d3998835d206118ccf35e3871dbd2fb60408cf6b007a80 \
    --hash=sha256:8858d7bfc22e3f51529aeaa4077225029724623e4595dc9eff8c793935c34140 \
    --hash=sha256:9150a83248bfed9813ea3c3af74c3856c1984d444aa28e58bf7733b9750ddf6a \
    --hash=sha256:9171748cdf796972d85a4b60157c279913e242992e350c90c7450182a9838b2a \
    --hash=sha256:a4d6d5e9a3d1879a97c08ded0c797579b7965eafd0f0c26c30b45ccc06db939b \
    --hash=sha256:a4dd8bf99a8fac133efc0ed6a92f5fddbe2adba0d0f6dd720e39ba9855cea85c \
    --hash=sha256:aa0559502e1cd6254d6814614085dd9c5a3dd0419362978a936a3f68a9e5c3df \
    --hash=sha256:b7a296aac7a71fa0886c08e155ddb6c636a50013f801f6178daafa0f9e726188 \
    --hash=sha256:bddd0c4f7630c2a3ddf6347c1bdaa79d97bcf6bd445f9e60c816b7d77c85a5ae \
    --hash=sha256:bf0b672390cdcb640d7288f96b826d71ff4e9abb254a86c89890baf51a29cee6 \
    --hash=sha256:c7c534ec03c358a76ea3e505e74c1b6aef290af90c444dfd092dbfe23e755b85 \
    --hash=sha256:cab40b1edfef0262e0e5251aa2c58d75630f24d06dd7794480243acc001a1d7d \
    --hash=sha256:cc4aa407fde9fc660be3939e49ea31f50f3e9fec17c0ec63159f7711edd3efc9 \
    --hash=sha256:d51592cb7561e87877c506113e7adbf1342ab579e6c21f0ef44b8ba41cb74c80 \
    --hash=sha256:dda9470024204d7bbf2042b47c6e8a0e47a3eeb8e34405882dfaea6577e0c153 \
    --hash=sha256:df961f2e7ae9cf496459259d798652c70625f6c080650d6952f8c04053c58ee9 \
    --hash=sha256:eb6203482ff3746a5632303a7279ae0b5a304c46985b49ed1378cb350ea6728d \
    --hash=sha256:f3831aaa25c67a99f99dc8b05873cb9d64560390372e2aa197ce9dd4a3f06a44 \
    --hash=sha256:f729cfdbd36fd99d543b67a914d2de044c84ebe45be8b34902b299b608c15c8f
    # via blip-captioning-pipeline (pyproject.toml)
pyyaml==6.0.3 \
    --hash=sha256:00c4bdeba853cc34e7dd471f16b4114f4162dc03e6b7afcc2128711f0eca823c \
    --hash=sha256:0150219816b6a1fa26fb4699fb7daa9caf09eb1999f3b70fb6e786805e80375a \
    --hash=sha256:02893d100e99e03eda1c8fd5c441d8c60103fd175728e23e431db1b589cf5ab3 \
    --hash=sha256:02ea2dfa234451bbb8772601d7b8e426c2bfa197136796224e50e35a78777956 \
    --hash=sha256:0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6 \
    --hash=sha256:10892704fc220243f5305762e276552a0395f7beb4dbf9b14ec8fd43b57f126c \
    --hash=sha256:16249ee61e95f858e83976573de0f5b2893b3677ba71c9dd36b9cf8be9ac6d65 \
    --hash=sha256:1d37d57ad971609cf3c53ba6a7e365e40660e3be0e5175fa9f2365a379d6095a \
    --hash=sha256:1ebe39cb5fc479422b83de611d14e2c0d3bb2a18bbcb01f229ab3cfbd8fee7a0 \
    --hash=sha256:214ed4befebe12df36bcc8bc2b64b396ca31be9304b8f59e25c11cf94a4c033b \
    --hash=sha256:2283a07e2c21a2aa78d9c4442724ec1eb15f5e42a723b99cb3d822d48f5f7ad1 \
    --hash=sha256:22ba7cfcad58ef3ecddc7ed1db3409af68d023b7f940da23c6c2a1890976eda6 \
    --hash=sha256:27c0abcb4a5dac13684a37f76e701e054692a9b2d3064b70f5e4eb54810553d7 \
    --hash=sha256:28c8d926f98f432f88adc23edf2e6d4921ac26fb084b028c733d01868d19007e \
    --hash=sha256:2e71d11abed7344e42a8849600193d15b6def118602c4c176f748e4583246007 \
    --hash=sha256:34d5fcd24b8445fadc33f9cf348c1047101756fd760b4dacb5c3e99755703310 \
    --hash=sha256:37503bfbfc9d2c40b344d06b2199cf0e96e97957ab1c1b546fd4f87e53e5d3e4 \
    --hash=sha256:3c5677e12444c15717b902a5798264fa7909e41153cdf9ef7ad571b704a63dd9 \
    --hash=sha256:3ff07ec89bae51176c0549bc4c63aa6202991da2d9a6129d7aef7f1407d3f295 \
    --hash=sha256:41715c910c881bc081f1e8872880d3c650acf13dfa8214bad49ed4cede7c34ea \
    --hash=sha256:418cf3f2111bc80e0933b2cd8cd04f286338bb88bdc7bc8e6dd775ebde60b5e0 \
    --hash=sha256:44edc647873928551a01e7a563d7452ccdebee747728c1080d881d68af7b997e \
    --hash=sha256:4a2e8cebe2ff6ab7d1050ecd59c25d4c8bd7e6f400f5f82b96557ac0abafd0ac \
    --hash=sha256:4ad1906908f2f5ae4e5a8ddfce73c320c2a1429ec52eafd27138b7f1cbe341c9 \
    --hash=sha256:501a031947e3a9025ed4405a168e6ef5ae3126c59f90ce0cd6f2bfc477be31b7 \
    --hash=sha256:5190d403f121660ce8d1d2c1bb2ef1bd05b5f68533fc5c2ea899bd15f4399b35 \
    --hash=sha256:5498cd1645aa724a7c71c8f378eb29ebe23da2fc0d7a08071d89469bf1d2defb \
    --hash=sha256:5cf4e27da7e3fbed4d6c3d8e797387aaad68102272f8f9752883bc32d61cb87b \
    --hash=sha256:5e0b74767e5f8c593e8c9b5912019159ed0533c70051e9cce3e8b6aa699fcd69 \
    --hash=sha256:5ed875a24292240029e4483f9d4a4b8a1ae08843b9c54f43fcc11e404532a8a5 \
    --hash=sha256:5fcd34e47f6e0b794d17de1b4ff496c00986e1c83f7ab2fb8fcfe9616ff7477b \
    --hash=sha256:5fdec68f91a0c6739b380c83b951e2c72ac0197ace422360e6d5a959d8d97b2c \
    --hash=sha256:6344df0d5755a2c9a276d4473ae6b90647e216ab4757f8426893b5dd2ac3f369 \
    --hash=sha256:64386e5e707d03a7e172c0701abfb7e10f0fb753ee1d773128192742712a98fd \
    --hash=sha256:652cb6edd41e718550aad172851962662ff2681490a8a711af6a4d288dd96824 \
    --hash=sha256:66291b10affd76d76f54fad28e22e51719ef9ba22b29e1d7d03d6777a9174198 \
    --hash=sha256:66e1674c3ef6f541c35191caae2d429b967b99e02040f5ba928632d9a7f0f065 \
    --hash=sha256:6adc77889b628398debc7b65c073bcb99c4a0237b248cacaf3fe8a557563ef6c \
    --hash=sha256:79005a0d97d5ddabfeeea4cf676af11e647e41d81c9a7722a193022accdb6b7c \
    --hash=sha256:7c6610def4f163542a622a73fb39f534f8c101d690126992300bf3207eab9764 \
    --hash=sha256:7f047e29dcae44602496db43be01ad42fc6f1cc0d8cd6c83d342306c32270196 \
    --hash=sha256:8098f252adfa6c80ab48096053f512f2321f0b998f98150cea9bd23d83e1467b \
    --hash=sha256:850774a7879607d3a6f50d36d04f00ee69e7fc816450e5f7e58d7f17f1ae5c00 \
    --hash=sha256:8d1fab6bb153a416f9aeb4b8763bc0f22a5586065f86f7664fc23339fc1c1fac \
    --hash=sha256:8da9669d359f02c0b91ccc01cac4a67f16afec0dac22c2ad09f46bee0697eba8 \
    --hash=sha256:8dc52c23056b9ddd46818a57b78404882310fb473d63f17b07d5c40421e47f8e \
    --hash=sha256:9149cad251584d5fb4981be1ecde53a1ca46c891a79788c0df828d2f166bda28 \
    --hash=sha256:93dda82c9c22deb0a405ea4dc5f2d0cda384168e466364dec6255b293923b2f3 \
    --hash=sha256:96b533f0e99f6579b3d4d4995707cf36df9100d67e0c8303a0c55b27b5f99bc5 \
    --hash=sha256:9c57bb8c96f6d1808c030b1687b9b5fb476abaa47f0db9c0101f5e9f394e97f4 \
    --hash=sha256:9c7708761fccb9397fe64bbc0395abcae8c4bf7b0eac081e12b809bf47700d0b \
    --hash=sha256:9f3bfb4965eb874431221a3ff3fdcddc7e74e3b07799e0e84ca4a0f867d449bf \
    --hash=sha256:a33284e20b78bd4a18c8c2282d549d10bc8408a2a7ff57653c0cf0b9be0afce5 \
    --hash=sha256:a80cb027f6b349846a3bf6d73b5e95e782175e52f22108cfa17876aaeff93702 \
    --hash=sha256:b30236e45cf30d2b8e7b3e85881719e98507abed1011bf463a8fa23e9c3e98a8 \
    --hash=sha256:b3bc83488de33889877a0f2543ade9f70c67d66d9ebb4ac959502e12de895788 \
    --hash=sha256:b865addae83924361678b652338317d1bd7e79b1f4596f96b96c77a5a34b34da \
    --hash=sha256:b8bb0864c5a28024fac8a632c443c87c5aa6f215c0b126c449ae1a150412f31d \
    --hash=sha256:ba1cc08a7ccde2d2ec775841541641e4548226580ab850948cbfda66a1befcdc \
    --hash=sha256:bdb2c67c6c1390b63c6ff89f210c8fd09d9a1217a465701eac7316313c915e4c \
    --hash=sha256:c1ff362665ae507275af2853520967820d9124984e0f7466736aea23d8611fba \
    --hash=sha256:c2514fceb77bc5e7a2f7adfaa1feb2fb311607c9cb518dbc378688ec73d8292f \
    --hash=sha256:c3355370a2c156cffb25e876646f149d5d68f5e0a3ce86a5084dd0b64a994917 \
    --hash=sha256:c458b6d084f9b935061bc36216e8a69a7e293a2f1e68bf956dcd9e6cbcd143f5 \
    --hash=sha256:d0eae10f8159e8fdad514efdc92d74fd8d682c933a6dd088030f3834bc8e6b26 \
    --hash=sha256:d76623373421df22fb4cf8817020cbb7ef15c725b9d5e45f17e189bfc384190f \
    --hash=sha256:ebc55a14a21cb14062aa4162f906cd962b28e2e9ea38f9b4391244cd8de4ae0b \
    --hash=sha256:eda16858a3cab07b80edaf74336ece1f986ba330fdb8ee0d6c0d68fe82bc96be \
    --hash=sha256:ee2922902c45ae8ccada2c5b501ab86c36525b883eff4255313a253a3160861c \
    --hash=sha256:efd7b85f94a6f21e4932043973a7ba2613b059c4a000551892ac9f1d11f5baf3 \
    --hash=sha256:f7057c9a337546edc7973c0d3ba84ddcdf0daa14533c2065749c9075001090e6 \
    --hash=sha256:fa160448684b4e94d80416c0fa4aac48967a969efe22931448d853ada8baf926 \
    --hash=sha256:fc09d0aa354569bc501d4e787133afc08552722d3ab34836a80547331bb5d4a0
    # via
    #   huggingface-hub
    #   transformers
regex==2026.9.29 \
    --hash=sha256:01000ddf0e3ffef97f2413ceb514f6313040106b6d18a03ee00a4fe35c1eb1db \
    --hash=sha256:0166844493626c5015c6088ee15c9ca2fd060ca15b7641d1657da6a58432ae33 \
    --hash=sha256:044265d77d94f5e3cb2fd72c76723807c429cb8c533e9d4672d0334a6f14f588 \
    --hash=sha256:0476e5bcbe6e1ba3d1c4cc7bbb1c3ba78e3b979b5c8a88d0a6a8cdd4992b8c84 \
    --hash=sha256:066d0e3dbfdd739bce2bf8c2a41dd16f73e3d8adc2eb06dd803a36a307f56075 \
    --hash=sha256:0b65c72739f981377c9c22e0c5c3cd7f42da7bd8a3c9209330fac772c7d893ed \
    --hash=sha256:0c992c19cd45058a4b92f68f139c93db168b48fb1f322c9a7cd620806afb6b51 \
    --hash=sha256:0cc63b5e47c12a48d90c7e9d7de6a035dd14f62868aaedbb4e0ff8ba2b8bfe7b \
    --hash=sha256:0dd8af32e9f7b56b7f95cc1fd79b23054c3bdc172392ae560acc24d57b7ffe71 \
    --hash=sha256:0def9fb6abac55492d6d51cddb7225d07d6f279e774e0adc08569a54a5fc8d46 \
    --hash=sha256:0fd2c901cc307a745ad4bc87f20060d7a0825a3371d1e93488af22e7a387f78f \
    --hash=sha256:1043aedf5917caa861bcb25a9c11460049656bdf0017a90a309fa8f255467725 \
    --hash=sha256:121a76a0985db80ceae9e171c337f8c927868e37d01b54e3ce87bc87f9c6a208 \
    --hash=sha256:143533cc4b6fbc5b95aca0a5b8d541088d374831593def000ec89322c220221d \
    --hash=sha256:14e953ff3607c92d7675bf79c4d4509ef6782aa8c08509f179f9b3d6d0679e86 \
    --hash=sha256:18ae8eed4526e35bdb754d61562b90bf5c00a67fdcf3cc1380dd59597486631b \
    --hash=sha256:19959129885356df0e97556856f77eb2888380dac18bed075a7c05c5128c618d \
    --hash=sha256:1ba8c6a416569ce0d37e83e28a254a61dc99a419084dfb6476cea02d997f74fa \
    --hash=sha256:1c2a0026062abcc321a53db4a185ceba0b59a66b5d37b0808917a88b55a5257f \
    --hash=sha256:1d9fe8091b2e89d470df68a9331111ed008ae8aae6bf1e8e1fba4086a495c84e \
    --hash=sha256:2089fe39c406784d90101c726755ffa1497bb74638fd434300d2b88006186de8 \
    --hash=sha256:23ae6fdad9e63e54038f5ef78aba2933faca61e24d432786589e737bc5522ebb \
    --hash=sha256:26ec4ccce55aa533fbd603d08911b01101a8fcfec987845ac3ae2c7087b2bde3 \
    --hash=sha256:2f7f7aa47b229f2b39a2ae2596d2ad5625d77b5eb9856fac2dab3eb506cdd0a0 \
    --hash=sha256:31b003f9a070335e2a8233ee9b14a3ca8e6d792012ae011f741bf0aaf11744c5 \
    --hash=sha256:32ab11df9677ca80bcbb5fe4eb1da9109a5019239a054836efc6fa1c64e683cf \
    --hash=sha256:33026515aebc0e70d1c89978e53e8d695d35d9e472f8d5b34465ba3c74028650 \
    --hash=sha256:34b6925af9853bf461950e6508910f179fd6e9b1a7ec8548e069606b7e51a26b \
    --hash=sha256:352cf115a810b357caa35193ab656ecf5ef41056855e82f292c99e8514f8d954 \
    --hash=sha256:39ab5894d971f9ac68baa6eca5c50387db579cfcacf36ae8df3feceb1815e6d0 \
    --hash=sha256:3a21a9509d0ee88e7a70e1ad228cd2f0e0fd1e187458db132e8a8d18c97daf9d \
    --hash=sha256:3c5c2ef13797466aa64170cbb66ad98a32351dd4127694cea7199f80f213750d \
    --hash=sha256:3e778bfccd63075167709136afbc251c1f683758d5bf49c803c60ac3f894ce6b \
    --hash=sha256:3f1e6cb402a89457582cd696f982559217d13484a193202c394015297968c86d \
    --hash=sha256:42e82e578c904445d4c8a35b8f28052cf567593215fa5db06266fbc6f77aaa2e \
    --hash=sha256:4408b2b27a95ca8cc48b7411945753773353b5c93b307754781086c99d3a576f \
    --hash=sha256:446654b29bfaa30500d80947eda42cef1449dc8a87f4e3cf061cc8485d3a1f0b \
    --hash=sha256:45010bcfe66df41522d56c9b6114e87ecc597a08970ff6a2ced24415c141ae5f \
    --hash=sha256:49ee178ca31c94621294bf9b8b676a92a2e6bba8af0529591753719e57edb621 \
    --hash=sha256:4d7d93613b01b0199961330e49cfc52d479b3d5776c56c691db31130c0a07d91 \
    --hash=sha256:4fb41211d2333eb930a51e0546a65999761cf1f572a4da56ef9b8a62966c06f2 \
    --hash=sha256:4fe97894d1b306c919b4e50def1e6f6c522f4d03a7283811f4d108f1ce5d3ac2 \
    --hash=sha256:554bffadcbcb6d5f4e5fb10a61cc52084b9a63d1dab5f10bcd2c4343972e8e2c \
    --hash=sha256:59b49507f47479e299a9e1bc41b5cb83a7afda0540625f1dbae886615978acbf \
    --hash=sha256:5eeb8edc6110d9194a4d0d54610f64c37a31c605b5dbb7e407fc6ec7fa34a4a1 \
    --hash=sha256:612b709381c0355b70d89cdb51b7f670591ed5cbbc0e3b5337488019dc667b65 \
    --hash=sha256:61956f074ecd123f55adca68ee3eab46e6a07ad3f8e64e6db95dfacb444f55c4 \
    --hash=sha256:6398d5145689503412cc1748895242598d8846b8967b851133b20dc2ed1e21e8 \
    --hash=sha256:65b408d8fcb273e3499e7ef2ce796810da1becd208c7fb4373692a242d79d461 \
    --hash=sha256:686ac5350fceae63830bb98805fcb8039325bf4c06d9f6f048ff65229d5bffa5 \
    --hash=sha256:6a1a824fbed817e0a891103886b68f063b1e83cc51bc97192a90a60195a9291f \
    --hash=sha256:6abb75ab16bc3281714a5b99548a2225db70dba1f995f6d7f7419b76eb5a8fbe \
    --hash=sha256:6f7121a8914ed13fcfe2099f895341bfb789f004d4c5a0bdece8fa667da10849 \
    --hash=sha256:7020ed44df30b3aa492c00ee3b52d0548c1f30c2c6c5bb13ae897680900d3413 \
    --hash=sha256:720537c7ea6f80dc61913184edb0ce2497a306b39ef19f28505b322553d52bdb \
    --hash=sha256:724184b4aafed865e4f13ca313fdcb43024300c028ec67319cfa16847d84685e \
    --hash=sha256:7c03031610e3e6ed1768a2b7a8fc84637c1257b50c5eacaf094c6e17a84fc563 \
    --hash=sha256:80a5ea3b4fd9d6a5b9a44f7976a9acaaab35aa3c1f6b29e5bd857dfabaded223 \
    --hash=sha256:80c7cadd3fd2bfde5df8aa0787e315812cad0c313a753095d02f4c2b6c01677b \
    --hash=sha256:80ea96f5c1a30bf09007d48466521d9c294bebe197c708c3359096e3e3691632 \
    --hash=sha256:864e9b87ac33c3fb9fb4ad48166d4fdb579c351d5c77deb0d34bccb36a775cd9 \
    --hash=sha256:87fb80cbe3557e27e7b28b995c2b2eedf689b8886f941ab93e0e288f0976518a \
    --hash=sha256:8873c4a11c50b9989168881aeb3f08859f469d809941866aa1feefd8be5431f6 \
    --hash=sha256:888d60953908dcf761aa320c3e390ab8556efbdb551ace63921de90f6ae0848d \
    --hash=sha256:8b5fcc4771732191b2b7d1dd68d8f0353f47f8d90b6150f6dce58bf1112442cb \
    --hash=sha256:8f39588af4731c8923c26810eb3b33f76f17633985e40f59c3cd45a33805a895 \
    --hash=sha256:9173db3be74a35cb6731701094b98120f7ee4876a287882a59cdea1fa7da342f \
    --hash=sha256:92f05c9c42bde5785dc48770bc2194d9f7442544156f951e19cd31b096cec562 \
    --hash=sha256:951733b1bbdb71e377cec567b409f1a7881b47cfcad84121aa74cb575fa425ea \
    --hash=sha256:957bb708e8057ab1649ba566456429d691ec9b90d1c9ad1af1ba7ffbbeaf05f2 \
    --hash=sha256:9916fda742cd4eede63b286f58c06718324265d727ce0856eb1aac86d0d150d6 \
    --hash=sha256:9e1d3a4cb7993b708f0ada8d0c84590efd853f169e7147d2202c9da503180242 \
    --hash=sha256:9e4482589065c8ecd761cff522dcd85f2d39e62f551e37e025d1c7d54772def3 \
    --hash=sha256:a5300757f8a68f5b6cc33f57338d72a0e3589c5cc9ad5f8504ea06f028be582a \
    --hash=sha256:a540abfab208e1b7ef2df231c40ef3b6cbb30a0aad6204e9b6a81c10a6794628 \
    --hash=sha256:a5758353650079898dc1b2b0e95aa51fa23a30d020e06f62c430dd08ee56cdd8 \
    --hash=sha256:a64b85a4760337cfefdb27d42da6ed8b58e8cde3f2d57b6ef43e76ef6ea9ef47 \
    --hash=sha256:a655d34b2a6943af32401f3d94f72e9d731f6ad16285815550bf2b4ee69d420a \
    --hash=sha256:a714befaacbd10092ffe4cea0d3c5f008fb9efe9bc322c715bcdfdee414b9a3d \
    --hash=sha256:a760da040b47767b4b873adfb7c3b691e9ba2fc60f113f9d0b88f1a62f323e85 \
    --hash=sha256:addd736a0547d553283adaf4e05d7104e7f2c7b0b092e9b4d28756825f14531f \
    --hash=sha256:ae4613d7d9dda60fcba95f846cc6f808017f1843f392cf9daad14a6534493d71 \
    --hash=sha256:b11b589e00095ec69cf79841a76360f9b079e95b0368a25b5ebb951ab0c157ff \
    --hash=sha256:b3e445b66c80b4eb4234e855ce94d9adc183eedbd632816228d89930b91b2c5b \
    --hash=sha256:b7b893976e7fe42053da64f2aa27239c24252fd2ec6df471e1be197c0addc3b1 \
    --hash=sha256:b84f186a7f0536fe4ff9a9fa12d06d007b9b71d4b5352ddcc41f59ad6522a312 \
    --hash=sha256:b89efc38431793d28b7cd91227e2f952ad7c48df19132b17f43a5fec3c14143b \
    --hash=sha256:b97a38fb4c732b6832db6bf108963adbcd82ef1268ba2025dce390f45af75efa \
    --hash=sha256:b9d74e4eee9ddb64c2e92d5d61472c59c21684c059eb7b68767be9628e977859 \
    --hash=sha256:bb90e7177944b6684738c1fc36aabd2dd00d1de3be7dbe09f91e196f1bc0dc81 \
    --hash=sha256:bec37990e3d6121f29ecfb594bd8f1bf009e9f7926daba2e50e3b27d3892a783 \
    --hash=sha256:bf3c49863c23a1ad6da9c30351aed6cff8d5ddbeb63c5c8420ae54e98c7d0138 \
    --hash=sha256:bf48516e35cf848390ea68850aba53e7c333720d2945b4d2c25b69fc5171723f \
    --hash=sha256:bfc71e6d970419c1309b3640305298643e2a734cad3f7cfb6d2ddee4175ab53d \
    --hash=sha256:c0094897d7d01f184b2d7fe8c56c66d64efe01b31f4b7d34205b391387df1111 \
    --hash=sha256:c03c6eb6ece86dfdcbb34799efaa339b093132e1aceed491ba5e08fe06cdf699 \
    --hash=sha256:c1a9a6651197fbed6f0212591418b9def774fc3f8324f78d1bf0e6a63e5f8aa1 \
    --hash=sha256:c3589f40749acce747510bf5d589d54e376cb0930ea58b35effac97e5312b0c1 \
    --hash=sha256:c4e38dd8f39c43a91d2410ad2b85610701b0979342c3df1d69eaf8e838c757d8 \
    --hash=sha256:c6c8fabf1dafc1f1ddcbb67896d3f93efb092e8c4b6322d7389b944e76a484e5 \
    --hash=sha256:c90fcf7804ea0a54b896ce0f2b9565350220b8d4890fd0db461a476a4c687963 \
    --hash=sha256:c9b602fae1e00b7c035d661ce85575365719192a7b46784bd71cf64c68053aa0 \
    --hash=sha256:ccb64d887a9db1cd76dbc0f92051a1a478a2a67e7f56c62d915cb881d7734704 \
    --hash=sha256:d06fcdecc10fc7954d7c8f27a03c96055fe525274dc84a7b0dbdc3d6b9e03dab \
    --hash=sha256:d0c3082bf79bcd6a614d55916590ad4b8f93200e10b97f463ea5d9d07c9b5f23 \
    --hash=sha256:d49c18f1ea294cf4adde2e5ac256e98c82ea9d708462ce4bf799dffa7cfe8a2c \
    --hash=sha256:d60030baaa7bfbb02d650c126cdcddcb6e33dbff14d819434c8fa2fdcaeeeba5 \
    --hash=sha256:d7cab119d0df0b9413f106b4d7fc34f2872d3574ed3806fb48959c830b1537da \
    --hash=sha256:d9b77b25b4f395f92de6099ab08e8ae2bc7e51dfe157f22900902243a5cc90c7 \
    --hash=sha256:dabee8f4935e731fb46b2a3091bdda0d3d94b3bbfb907d2b4f12eefce4009619 \
    --hash=sha256:db5e82ba15c142425b8406690032df89e39cca4a2e8afbbb9a3d84edc2373ac3 \
    --hash=sha256:dc79d36d0618752265f0d575915bdc5c5130ecb9c9f6b3bcefeae32e4bdfafcf \
    --hash=sha256:ddfa987262763c3c22a8367d2a49c244b018a74c3a8e3ab1a864119ad45c5633 \
    --hash=sha256:e1172147d28d8fbcf8cb8d26c41506169f5ad8fe9ec969cb116835a19d4d8eca \
    --hash=sha256:e11edba5bc344a32b029a7af9d4b3173982dd79eeafa0b9dbd787364414b0509 \
    --hash=sha256:e2c89e9b762c57f59d5e99ee8b20202adb892e35f8d3485741340999ca55058e \
    --hash=sha256:e31f72490b7c12f7790e1e25c3afffd20503ee1bfb43461d7838b871ff244b19 \
    --hash=sha256:e8c65ef3862a8ad6e86492b6ed9327805dd66904c012bd3649dc67d822ed6c34 \
    --hash=sha256:ebb8912f565b8cdbbf27debfe00df04202c20e2f651b9e32767930c5eace3621 \
    --hash=sha256:ed511a0708e2297e1d6431e7fb217e3402791e491e02da800658ace4973df1bb \
    --hash=sha256:edf06545875f3efa31560d94121e95c7fd70d98b1dfedc0157097d79b13b52ea \
    --hash=sha256:f0fe9834e5aeccaf19a0d8feb296d66a24be1a7c9922002f842a682cd5abb787 \
    --hash=sha256:f1a0d5117230dd46b399a30a38afa44f79c99f3168988fdc4f425c3f928b39df \
    --hash=sha256:f37964e4a5e993d2fd45147741e9dff7f34a2d8c00ab94c4ea0514a4677f959e \
    --hash=sha256:f57dc6b8fef170f105d2cf5cdce254f47b137d7755086cf7050f47e16582abba \
    --hash=sha256:f93bc1c3486ef3747e07c9d7c1d0a147b8fbaab975f80e348aed6f71309dfaca \
    --hash=sha256:fb00027a09a8f9f08028b40dce4c933cf73e4833240ed356583fdc9cfa721566 \
    --hash=sha256:fb99cc9d45f48895d9d67f6a0b8a57f08d39c174d9f25ad97a313e0470267b1c \
    --hash=sha256:fdd88ed5e20b1bcdd234421e454962c971aa44b653bdb7f1ea9ef683e90fb649 \
    --hash=sha256:fe3fa1dd453ed5c7f5ea23a26218329790ed7197a99b90e94330e313959a7f52
    # via transformers
requests==2.34.2 \
    --hash=sha256:2a0d60c172f83ac6ab31e4554906c0f3b3588d37b5cb939b1c061f4907e278e0 \
    --hash=sha256:f288924cae4e29463698d6d60bc6a4da69c89185ad1e0bcc4104f584e960b9ed
    # via
    #   huggingface-hub
    #   transformers
safetensors==0.8.0 \
    --hash=sha256:040070828e36dc8e122178bbbd5830ff9e97920affb84cbe0f46442497bed358 \
    --hash=sha256:096ec1a98435df7beb08853bb5aa9081a84f23d0adc67ed1a0a10550f608373f \
    --hash=sha256:2ddf52eac562eda224f99acfa7889d02968c1fd59a5b011ae7d8137c37e9c02d \
    --hash=sha256:3ae091f16662658bdc019a4ff6cb4c085bb7d725eb5978b183ffd265863b6d2d \
    --hash=sha256:4124502b78f03534117c848f87a39b8f31e577b15eff423bf8bfb95f2a8c30d0 \
    --hash=sha256:4a95ae2b05d7726d751da4ebf626a2ca782b706e101bd894c95bc2450b1cffcc \
    --hash=sha256:7a46e5ff292c356d6991e60942ba7f79817682d3a2cef0702136448cb9c4d235 \
    --hash=sha256:7bc0a787ba8a35be368ee3574edfa2b1ad389eebd0a72e482ae275490e3f6c98 \
    --hash=sha256:87eec7ffed2b809f05a398a8becb7d013f19f7837cd15d9748580d6cf30dbaf4 \
    --hash=sha256:8e080062fcde23be189565e1c3305d16751a218ecf9412c8601e64204eb6f846 \
    --hash=sha256:8e9f537aa183a38ace122d27303dcd986b26bd2a7591f9181d7f0c396f4677ca \
    --hash=sha256:c554f85858e05226d3c2828e32395e677434685d6d94594a41643361c5e837f0 \
    --hash=sha256:c80201d22cbf405b80647a60ada77bba06c8fba2da2743ba1e89cdcc39a81f25 \
    --hash=sha256:f7838e5135a406ad3e02efdcb8cf2e5397d368b0154537c4fec682dbc544d452 \
    --hash=sha256:fabaf3e0f18a6618d9b36560682562157f77c2b71fcffc7b432be2baed9d753d \
    --hash=sha256:fcdd41ec4628fee5799f807c73c353629130fbd942aa23d83c623dd6c9d52d78 \
    --hash=sha256:fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774
    # via
    #   blip-captioning-pipeline (pyproject.toml)
    #   transformers
setuptools==84.0.0 \
    --hash=sha256:51a52592b3b99e102b609654876bd65f19f999935166d1352678931132b0c670 \
    --hash=sha256:f4695c21257f0d9b537ec2692c941d02ee143b7cc1276941349a546573b2ef73
    # via torch
sympy==1.14.0 \
    --hash=sha256:d3d3fe8df1e5a0b42f0e7bdf50541697dbe7d23746e894990c030e2b05e72517 \
    --hash=sha256:e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5
    # via torch
tokenizers==0.22.2 \
    --hash=sha256:143b999bdc46d10febb15cbffb4207ddd1f410e2c755857b5a0797961bbdc113 \
    --hash=sha256:1a62ba2c5faa2dd175aaeed7b15abf18d20266189fb3406c5d0550dd34dd5f37 \
    --hash=sha256:1c774b1276f71e1ef716e5486f21e76333464f47bece56bbd554485982a9e03e \
    --hash=sha256:1e418a55456beedca4621dbab65a318981467a2b188e982a23e117f115ce5001 \
    --hash=sha256:1e50f8554d504f617d9e9d6e4c2c2884a12b388a97c5c77f0bc6cf4cd032feee \
    --hash=sha256:2249487018adec45d6e3554c71d46eb39fa8ea67156c640f7513eb26f318cec7 \
    --hash=sha256:25b85325d0815e86e0bac263506dd114578953b7b53d7de09a6485e4a160a7dd \
    --hash=sha256:29c30b83d8dcd061078b05ae0cb94d3c710555fbb44861139f9f83dcca3dc3e4 \
    --hash=sha256:319f659ee992222f04e58f84cbf407cfa66a65fe3a8de44e8ad2bc53e7d99012 \
    --hash=sha256:369cc9fc8cc10cb24143873a0d95438bb8ee257bb80c71989e3ee290e8d72c67 \
    --hash=sha256:37ae80a28c1d3265bb1f22464c856bd23c02a05bb211e56d0c5301a435be6c1a \
    --hash=sha256:38337540fbbddff8e999d59970f3c6f35a82de10053206a7562f1ea02d046fa5 \
    --hash=sha256:473b83b915e547aa366d1eee11806deaf419e17be16310ac0a14077f1e28f917 \
    --hash=sha256:544dd704ae7238755d790de45ba8da072e9af3eea688f698b137915ae959281c \
    --hash=sha256:64d94e84f6660764e64e7e0b22baa72f6cd942279fdbb21d46abd70d179f0195 \
    --hash=sha256:753d47ebd4542742ef9261d9da92cd545b2cacbb48349a1225466745bb866ec4 \
    --hash=sha256:791135ee325f2336f498590eb2f11dc5c295232f288e75c99a36c5dbce63088a \
    --hash=sha256:9ce725d22864a1e965217204946f830c37876eee3b2ba6fc6255e8e903d5fcbc \
    --hash=sha256:a6bf3f88c554a2b653af81f3204491c818ae2ac6fbc09e76ef4773351292bc92 \
    --hash=sha256:bfb88f22a209ff7b40a576d5324bf8286b519d7358663db21d6246fb17eea2d5 \
    --hash=sha256:c9ea31edff2968b44a88f97d784c2f16dc0729b8b143ed004699ebca91f05c48 \
    --hash=sha256:df6c4265b289083bf710dff49bc51ef252f9d5be33a45ee2bed151114a56207b \
    --hash=sha256:e10bf9113d209be7cd046d40fbabbaf3278ff6d18eb4da4c500443185dc1896c \
    --hash=sha256:f01a9c019878532f98927d2bacb79bbb404b43d3437455522a00a30718cdedb5
    # via transformers
torch==2.14.0 \
    --hash=sha256:0e7cf18cb0d8bd666b6120932e29c7aef3502b61a08b44da4839580c539a7cdb \
    --hash=sha256:2fb30099be1fec9d163a6dd05ace176305ce8cf914b4d7a5f4f6d7a326aa4b60 \
    --hash=sha256:44b044b9f6f633d982839422a57433d6a1da520037fd88e0c8a47efde589b3b8 \
    --hash=sha256:4dbea8a10d80b10dff2be1ed467d2617b704fe6bf76bbfad4194774c8f6d886b \
    --hash=sha256:553aec938d37d77b783bcf801e638cad068870e7643c47eadac21eb180f551ed \
    --hash=sha256:6981872df75eb5409c439050d36b67cb57d88b4e11080516e2aa7410f6730705 \
    --hash=sha256:731784e3914843c6bcc7aba3987ff7610ac57dbbc816a5d6b9b62e04c240a641 \
    --hash=sha256:731b9ebdea402b8b1996d4c2ae613b16660e559b19e47bdc45d970568bc91c53 \
    --hash=sha256:84bf384779a10c02fc3c6bdbab71a9cb66b0dd93c652d1ed5d6dfc0cb37e5962 \
    --hash=sha256:860423e970f2ce02c4476e8e2d1350131b1c5c5a5e4912180e78b50b53241efa \
    --hash=sha256:8d9e232b6376c62f3090237889fb1cb6887c0ece9f73e6c1052e80fc1481f999 \
    --hash=sha256:96383c62423c3f2767023c4a94774b6435e2c006691daf396e9ddb55d412fbc6 \
    --hash=sha256:9d4b1022a5d9b71282ec67ad0d9e7235870096b8a246dc1c32d6ea1fc83dc998 \
    --hash=sha256:ada340e62591d06a2bcc2d68170f20f45f0b0665d372dc510a8ed7eb3b1d609a \
    --hash=sha256:ae530ddd3f3b94248b77f1fd3313c4f3405bd0d17283d505b81574816767133a \
    --hash=sha256:b2cd92bce63d40bf6fc2e5d840fc2f0063bd180a242daa761cb6088cc2f46e27 \
    --hash=sha256:b985c7defeb8d28691b7513ebaecc64946c5f68ec770e9f4ddba39688864f46a \
    --hash=sha256:c1f844f1c750e87df4b68bc3afbc0e2b0c7ef19d7b8f666e48bdcf6a0c4f0056 \
    --hash=sha256:cad84f41bbdf3dcf333ce394aeeaf25237c4d94fd6b659ba5eb813c829978823 \
    --hash=sha256:cd8cd8f714d511ccdca907282d1da3d9be8322d4e1520b9c3bce39a5c1318a4b \
    --hash=sha256:d2526f71e6638133b97b3cd2881ece3df521460a4727030e8ae2a72a7d3ae31d \
    --hash=sha256:d30207cb89e713dfd6afb5d592b30d9ef252acf80407c5543157ea7b07c4d9a0 \
    --hash=sha256:d70c2b41bde81efde59fcefb147b9247fd062497e778677661b7de1bee5c8c99 \
    --hash=sha256:fecffb58f51fd643d213acd68da21cc3fc19bea05a3bc64b4ee55128f47a4963
    # via
    #   blip-captioning-pipeline (pyproject.toml)
    #   torchvision
torchaudio==2.11.0 \
    --hash=sha256:00e9f71ab9c656f0abdb40c515bd65d4658ab0ad380dee27a2efd7d51dabd3d6 \
    --hash=sha256:13cff988697ccbad539987599f9dc672f40c417bed67570b365e4e5002bbd096 \
    --hash=sha256:1424638adb8bb40087bc7b6eb103e8e4fe398210f09076f33b7b5e61501b5d66 \
    --hash=sha256:1a07ec72fd6f26a588c39b5f029e0130d16bb40bc4221635580bf8fb18fcbc80 \
    --hash=sha256:1be3767064364ae82705bdf2b15c1e8b41fea82c4cd04d47428a8684b634b6ed \
    --hash=sha256:1c1101c1243ef0e4063ec63298977e2d3655c15cf88d9eb0a1bd4fe2db9f47ea \
    --hash=sha256:478110f981e5d40a8d82221732c57a56c85a1d5895fb8fe646e86ee15eded3bd \
    --hash=sha256:492dd64645e9d0bb843e94f1d9a4d1e31426262ffc594fafecc1697df9df5eb9 \
    --hash=sha256:5847fe2022b17c6580aeb39c8797a443411cc09edfd9183cd50ac1a3b8ccf97c \
    --hash=sha256:6503c0bdb29daf2e6281bb70ea2dfe2c3553b782b619eb5d73bdadd8a3f7cecf \
    --hash=sha256:67f6edac29ed004652c11db5c19d9debb5d835695930574f564efc8bdd061bba \
    --hash=sha256:6ebb59c694909eccb5d61b7cc199d297692012c43286e36d92983aa7bad7586d \
    --hash=sha256:73dab4841f94d888bc7c2aed7b5547c643edc974306919fe1adfb65d57cccf4b \
    --hash=sha256:79fb3cb99169fd41bd9719647261402a164da0d105a4d81f42a3260844ec5e79 \
    --hash=sha256:7e2da1df4f6fe885c46db350a0dc90a0dff4b54541dff8846faa904d255e2bfe \
    --hash=sha256:88fb5e29f670a33d9bac6aabb1d2734460cf6e461bde5cdc352826035851b16d \
    --hash=sha256:986f4df5ed17b003dc52489468601720090e65f964f8bebccf90eb45bba75744 \
    --hash=sha256:9fe3083c62e035646483a14e180d33561bdc2eed436c9ab1259c137fb7120b4a \
    --hash=sha256:a1cf1acc883bee9cb906a933572fed6a8a933f86ef34e9ea7d803f72317e8c1b \
    --hash=sha256:b034d7672f1c415434f48ef17807f2cce47f29e8795338c751d4e596c9fbe8b5 \
    --hash=sha256:bb59ba4452bbbe95d75ad3ef18df9824955625f36698ce9a5998a4a9f3c1ba1d \
    --hash=sha256:bc653defca1c16154398517a1adc98d0fb7f1dd08e58ced217558d213c2c6e29 \
    --hash=sha256:bda09ea630ae7207384fb0f28c35e4f8c0d82dd6eba020b6b335ad0caa9fed49 \
    --hash=sha256:be7ad472acb16d16e98c005f0219b0db06a47dfe8f7b4d177062e1638f871e3b \
    --hash=sha256:cc09cd1f6015b8549e7fe255fb1be5346b57e7fee06541d3f3dbb012d8c4715f \
    --hash=sha256:da2725e250866da42a12934c9a6552f65a18b7187fd7a6221387f0e605fb3b96 \
    --hash=sha256:e3f9696a9ef1d49acc452159b052370c636406d072e9d8f10895fda87b591ea9 \
    --hash=sha256:ed404c4399ad7f172c86a47c1b25293d322d1d58e26b10b0456a86cf67d37d84
    # via blip-captioning-pipeline (pyproject.toml)
torchvision==0.29.0 \
    --hash=sha256:01a029fa1b2eacac27e1e4f9e0dba24d112bcbd523261b294e4b4a1c7f8330bc \
    --hash=sha256:0e3b99c62f7f095153887330c2c60c9ebc2da84dfbe08eea564a41ec361629cf \
    --hash=sha256:0e631a2f8b24732672d35224d2574ff89a78d56bef5364ef6094a625fa5f1771 \
    --hash=sha256:2428a13706a354ee3901fd63e3f1ef8c2a59f6e20d24cce13210b4e61be08f7a \
    --hash=sha256:28a3964a9e6db34354d4ca440d2c2038deeef24e7219e8d6f49d9d174f44830c \
    --hash=sha256:710df2c4f03ab6da161520c10d9467f4c0201ca314fd1432499a48c9d7a5818f \
    --hash=sha256:784c8b8de9e81e02dea093373bc35040791378df2daec0017e657985183bf862 \
    --hash=sha256:80789f9b50f5277302e7020c8128344e0269b12f6e3b5855034e56bb7081b874 \
    --hash=sha256:83db0299170502b038640444214c99d7b54bad72db447f0554deb8d2c4f02f94 \
    --hash=sha256:85fa54bec1f7d9227b5e4e201ed4bac92247ee10fb19148d92f9836e3e91d5c3 \
    --hash=sha256:942a3b3fb4e981e1abb7846679ee3882dfc733533a30b68df0dadf6c06f238a9 \
    --hash=sha256:994687b818cac0e6d34cb407a41458e2c3262cb7db927465841c9522ba7ebb92 \
    --hash=sha256:9b99a7385da8d706f2dc14cacffb8cbf653d48fc37013a38c41a36659acc3731 \
    --hash=sha256:a292ca7044236a2702ec3d4664de135e01fc9fb3ae9422db3fecc819546e831c \
    --hash=sha256:add88479361b2bf08790338fce4ba11a11fdc1ae508e80a3a013daf1c0d2893d \
    --hash=sha256:b7a736eaa6b2e22476c95ceb62986d195f3e600cdd7d196d7329aa2dfc951994 \
    --hash=sha256:bb74d08d65785b51c61d4d4ec167bfa50a5ea644b57555a33ba2f4435b0d05dd \
    --hash=sha256:bbe15455b59a6c9d822584fd3faaaac50ee972101d7bb7e2887da456e148a3ea \
    --hash=sha256:bd12e152640d1024fca15deaf6f39c53702e12eb398db4b2b038a5a97ab85f7f \
    --hash=sha256:ce3b58631f0b7c8828e8675301bff5cb0623aaed704e3ac91079e89607753205 \
    --hash=sha256:dc5717feef9a0b430b052895db71f1d695939575b74344da981120e9f4da7620 \
    --hash=sha256:e8fdf234d76dca6fc47bc5f4af86ecf1ef57fab6285a7b21681bdcbb494e4c1b \
    --hash=sha256:ebf54f6744556ebd8d7b5f9b6515e2060e5d3156723970dc257bf093f63b2170 \
    --hash=sha256:f9e9627d5036cac8e6de76aeb2f1acc67841ff5ff4b63c4cf78ff954efeb073e
    # via blip-captioning-pipeline (pyproject.toml)
tqdm==4.70.1 \
    --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73 \
    --hash=sha256:cefd0eca11b2a37a3aee776544d4f4ae913f02688135b5556b8788dfa474afc4
    # via
    #   huggingface-hub
    #   transformers
transformers==4.57.6 \
    --hash=sha256:4c9e9de11333ddfe5114bc872c9f370509198acf0b87a832a0ab9458e2bd0550 \
    --hash=sha256:55e44126ece9dc0a291521b7e5492b572e6ef2766338a610b9ab5afbb70689d3
    # via blip-captioning-pipeline (pyproject.toml)
triton==3.8.0 \
    --hash=sha256:1b84e7d512490ba529111260fa6f7cad8b254a6bb5fbdf41d5ef9a5e57f52d0a \
    --hash=sha256:1f0497218e26b7d79773ad9c2a3fa3b539ee69f587a13fac2e552b1d322a8015 \
    --hash=sha256:1f6b48d0591929a3867973acac3dccd4e058585f91bfb41022de496c9ffab304 \
    --hash=sha256:372285307d4c44ee74cee32de0b4f04bd157e071427e38c6f4ee3e3beb2194f4 \
    --hash=sha256:387dae4cb0089a7b6ba1a428ae0782b65c4c58f57d94617cb22ca8593d8ccbca \
    --hash=sha256:398f4b009c7ab08ed9aeb1d1282dee822945b53f399a5e12a4fecb643cf2007d \
    --hash=sha256:68988ac85d5e7086baeda0ddc175af9667db7529b3c5e11a5c0601b8bef2200a \
    --hash=sha256:6d914c52f89dc942b1819db959e7c07a5999f678ec760f677625c706b7e0d743 \
    --hash=sha256:74217bb56ed8692759227758e4c4b3bd2d608a209c1a7a081bf361fb4c2c1bf9 \
    --hash=sha256:a9c404c69ed4a39e8ec632eaf6b9fe058a060bf98979c177f6ef666f06bb8d50 \
    --hash=sha256:b7004666652f500ed854a86988e4b3d69d247188b5d2092b5df1e44f4a954099 \
    --hash=sha256:e91ffa46d095b252248297292dd22bcbacd53a125a0c2eefbbbf74925a320bc3
    # via torch
typing-extensions==4.16.0 \
    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \
    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5
    # via
    #   huggingface-hub
    #   torch
urllib3==2.8.0 \
    --hash=sha256:0cf3cae568d36aa9576b28dfb35f11328f1cb974ca7647d9475ebb86c75ac6e3 \
    --hash=sha256:63bf2ead4c879426ebf22ef2a781eeb4aa3b4ae798a0435506f8687fd5bb9b63
    # via requests
'''

SKIP_INSTALL = os.environ.get("DIMER_NOTEBOOK_CI_PREINSTALLED") == "1"
ISOLATED_ENV = Path(os.environ.get("DIMER_ISOLATED_ENV", "dimer_isolated_env")).resolve()
ISOLATED_PYTHON = ISOLATED_ENV / "bin" / "python"
ISOLATED_TOOLS = ISOLATED_ENV.with_name(ISOLATED_ENV.name + "_tools")

if SKIP_INSTALL:
    print("DIMER_NOTEBOOK_CI_PREINSTALLED=1: the pins are already installed; the notebook runs in this kernel.")
else:
    if platform.system() != "Linux" or platform.machine() != "x86_64":
        raise RuntimeError("This notebook needs a Linux x86_64 runtime (Google Colab, Kaggle or Linux Jupyter): its locked environment is built for manylinux x86_64.")
    setup_started = time.perf_counter()
    if hashlib.sha256(LOCK_TEXT.encode("utf-8")).hexdigest() != LOCK_SHA256:
        raise RuntimeError("The carried lock does not match its digest: regenerate the notebook from the repository instead of editing this cell.")
    ISOLATED_TOOLS.mkdir(parents=True, exist_ok=True)
    lock_path = ISOLATED_TOOLS / LOCK_NAME
    lock_path.write_text(LOCK_TEXT, encoding="utf-8", newline="\n")
    for attempt in range(3):
        try:
            with urllib.request.urlopen(UV_URL, timeout=90) as response:
                wheel = response.read(UV_BYTES + 1)
            break
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            if attempt == 2:
                raise
            time.sleep(2**attempt)
    if len(wheel) != UV_BYTES or hashlib.sha256(wheel).hexdigest() != UV_SHA256:
        raise RuntimeError("The pinned uv wheel failed its size/SHA-256 check: refusing to run it. Run this cell again; if it repeats, the download is being altered.")
    with zipfile.ZipFile(io.BytesIO(wheel)) as archive:
        member = next(name for name in archive.namelist() if name.endswith(".data/scripts/uv"))
        uv = ISOLATED_TOOLS / "uv"
        uv.write_bytes(archive.read(member))
    uv.chmod(0o700)
    # uv gets no kernel Python path; the managed interpreter is downloaded once and reused on a re-run.
    uv_env = dict(os.environ)
    for name in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP"):
        uv_env.pop(name, None)
    if not ISOLATED_PYTHON.is_file():
        subprocess.run([str(uv), "venv", "--quiet", "--managed-python", "--python", MANAGED_PYTHON, str(ISOLATED_ENV)], env=uv_env, check=True)
    isolated_version = subprocess.run([str(ISOLATED_PYTHON), "-c", "import platform; print(platform.python_version())"], env=uv_env, check=True, capture_output=True, text=True).stdout.strip()
    if isolated_version != MANAGED_PYTHON:
        raise RuntimeError(f"{ISOLATED_ENV} holds Python {isolated_version}, not {MANAGED_PYTHON}: delete that folder (or start a fresh runtime) and run this cell again.")
    subprocess.run([str(uv), "pip", "install", "--quiet", "--python", str(ISOLATED_PYTHON), "--require-hashes", "--only-binary", ":all:", "--index-url", "https://pypi.org/simple", "-r", str(lock_path)], env=uv_env, check=True)
    print({"isolated_environment": str(ISOLATED_ENV), "isolated_python": isolated_version, "kernel_python": platform.python_version(), "locked_packages": LOCKED_PACKAGES, "setup_seconds": round(time.perf_counter() - setup_started)})

{'isolated_environment': '/content/dimer_isolated_env', 'isolated_python': '3.12.12', 'kernel_python': '3.13.15', 'locked_packages': 48, 'setup_seconds': 58}


The next cell starts one Python process in that environment and routes **every later code cell** to it. Printed output comes back to the notebook as usual, variables persist from cell to cell, and an error stops **Run all** as it would in the kernel. These two cells are marked `# dimer: kernel cell` and are the only cells that run in the kernel. If you re-run a single cell later, it still runs in the isolated environment with the variables created so far; to start over, restart the session and choose **Run all**. `DIMER_NOTEBOOK_CI_PREINSTALLED=1` lets an executor that has already installed exactly these pins run every cell in its own kernel instead.

In [ ]:
# @title Infrastructure: route the remaining cells to the isolated environment
# dimer: kernel cell (runs in the notebook kernel, not in the isolated environment)
import signal
from multiprocessing.connection import Connection

# The worker runs in the isolated environment. It executes each routed cell in one persistent namespace and sends
# back printed text, displayed objects and matplotlib figures, so every cell behaves as it would in the kernel.
_WORKER_SOURCE = r"""
import ast, base64, builtins, io, linecache, os, signal, sys, traceback, types
from multiprocessing.connection import Connection

_send = Connection(int(sys.argv[1]), readable=False)
_recv = Connection(int(sys.argv[2]), writable=False)


class _Stream(io.TextIOBase):
    def __init__(self, name):
        self._name = name

    @property
    def encoding(self):
        return "utf-8"

    def writable(self):
        return True

    def isatty(self):
        return False

    def write(self, text):
        if text:
            _send.send(("stream", self._name, str(text)))
        return len(text)


sys.stdout, sys.stderr = _Stream("stdout"), _Stream("stderr")


def _figure_bundle(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", bbox_inches="tight")
    return {"image/png": base64.b64encode(buffer.getvalue()).decode("ascii"), "text/plain": repr(fig)}


def _flush_figures():
    plt = sys.modules.get("matplotlib.pyplot")
    if plt is None:
        return
    for number in plt.get_fignums():
        _send.send(("display", _figure_bundle(plt.figure(number))))
    plt.close("all")


def _mimebundle(obj):
    if hasattr(obj, "savefig"):
        return _figure_bundle(obj)
    data = {"text/plain": repr(obj)}
    for method, mime in (("_repr_html_", "text/html"), ("_repr_markdown_", "text/markdown"), ("_repr_png_", "image/png")):
        render = getattr(obj, method, None)
        if callable(render):
            try:
                value = render()
            except Exception:
                value = None
            if isinstance(value, bytes):
                value = base64.b64encode(value).decode("ascii")
            if value is not None:
                data[mime] = value
    return data


def display(*objects, **kwargs):
    for obj in objects:
        _send.send(("display", _mimebundle(obj)))


try:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot

    matplotlib.pyplot.show = lambda *args, **kwargs: _flush_figures()
except ImportError:
    pass

if os.environ.get("DIMER_KERNEL_IS_COLAB") == "1":
    # google.colab only exists in the kernel; forward the BYOD upload dialog to it.
    def _upload():
        _send.send(("upload",))
        reply = _recv.recv()
        if reply[1] is None:
            raise RuntimeError("The notebook kernel could not open the upload dialog.")
        return reply[1]

    import importlib.machinery

    def _stub(name, package):
        # A spec on every stub: importlib.util.find_spec("google.colab") (accelerate does this) raises on a None __spec__.
        module = types.ModuleType(name)
        module.__spec__ = importlib.machinery.ModuleSpec(name, None, is_package=package)
        if package:
            module.__path__ = []
        return module

    try:
        import google
    except ImportError:
        google = _stub("google", True)
        sys.modules["google"] = google
    _colab = _stub("google.colab", True)
    _files = _stub("google.colab.files", False)
    _files.upload = _upload
    _colab.files = _files
    google.colab = _colab
    sys.modules["google.colab"] = _colab
    sys.modules["google.colab.files"] = _files

_main = types.ModuleType("__main__")
_main.__dict__.update(__builtins__=builtins, display=display)
sys.modules["__main__"] = _main
_count = 0
while True:
    # An interrupt only lands inside a running cell; between cells it is ignored.
    signal.signal(signal.SIGINT, signal.SIG_IGN)
    try:
        message = _recv.recv()
    except EOFError:
        break
    if message[0] != "run":
        continue
    _count += 1
    filename = f"<isolated cell {_count}>"
    source = message[1]
    linecache.cache[filename] = (len(source), None, source.splitlines(True), filename)
    try:
        signal.signal(signal.SIGINT, signal.default_int_handler)
        tree = ast.parse(source, filename)
        tail = ast.Expression(tree.body.pop().value) if tree.body and isinstance(tree.body[-1], ast.Expr) else None
        exec(compile(tree, filename, "exec"), _main.__dict__)
        if tail is not None:
            value = eval(compile(tail, filename, "eval"), _main.__dict__)
            if value is not None:
                display(value)
        _flush_figures()
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        _send.send(("done",))
    except BaseException as exc:
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        frames = exc.__traceback__.tb_next if exc.__traceback__ is not None else None
        _send.send(("error", "".join(traceback.format_exception(type(exc), exc, frames)), f"{type(exc).__name__}: {exc}"))
"""


class IsolatedCellError(RuntimeError):
    """A routed cell raised inside the isolated environment; its traceback is printed above."""


class IsolatedRuntime:
    """One persistent worker process in the isolated environment, fed one cell at a time."""

    def __init__(self, python, display=None):
        to_kernel_r, to_kernel_w = os.pipe()
        to_worker_r, to_worker_w = os.pipe()
        env = dict(os.environ, MPLBACKEND="Agg", PYTHONUNBUFFERED="1", DIMER_NOTEBOOK_CI_PREINSTALLED="1", HF_HUB_DISABLE_IMPLICIT_TOKEN="1")
        for name in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
            env.pop(name, None)
        env["DIMER_KERNEL_IS_COLAB"] = "1" if "google.colab" in sys.modules else "0"
        self.proc = subprocess.Popen(
            [str(python), "-c", _WORKER_SOURCE, str(to_kernel_w), str(to_worker_r)],
            pass_fds=(to_kernel_w, to_worker_r),
            env=env,
            start_new_session=True,  # interrupts reach the worker only through run(), exactly once
        )
        os.close(to_kernel_w)
        os.close(to_worker_r)
        self._recv = Connection(to_kernel_r, writable=False)
        self._send = Connection(to_worker_w, readable=False)
        if display is None:
            from IPython.display import display
        self._display = display

    def _exited(self):
        return RuntimeError(
            f"The isolated environment's Python process exited (code {self.proc.wait()}); a crash of this kind is "
            "usually running out of memory. Restart the session and choose Run all again."
        )

    def run(self, source):
        try:
            self._send.send(("run", source))
        except OSError:
            raise self._exited() from None
        while True:
            try:
                message = self._recv.recv()
            except EOFError:
                raise self._exited() from None
            except KeyboardInterrupt:
                self.proc.send_signal(signal.SIGINT)
                continue
            kind = message[0]
            if kind == "stream":
                (sys.stdout if message[1] == "stdout" else sys.stderr).write(message[2])
            elif kind == "display":
                self._display(message[1], raw=True)
            elif kind == "upload":
                self._send.send(("upload", self._colab_upload()))
            elif kind == "error":
                sys.stderr.write(message[1])
                raise IsolatedCellError(message[2]) from None
            elif kind == "done":
                return

    @staticmethod
    def _colab_upload():
        try:
            from google.colab import files
        except ImportError:
            return None
        return files.upload()

    def close(self):
        self._send.close()
        self.proc.wait(timeout=30)


def _is_user_cell():
    # ipykernel transforms a cell before executing it; its caller knows whether this is a silent frontend request.
    frame = sys._getframe(2)
    while frame is not None:
        local = frame.f_locals
        if "silent" in local and "store_history" in local:
            return bool(local["store_history"]) and not bool(local["silent"])
        frame = frame.f_back
    return True


def _route_to_isolated_runtime(lines):
    source = "".join(lines)
    if not source.strip() or "# dimer: kernel cell" in source or not _is_user_cell():
        return lines
    return [f"_DIMER_ISOLATED_RUNTIME.run({source!r})\n"]


if SKIP_INSTALL:
    print("Routing disabled: the notebook runs in this kernel.")
else:
    from IPython import get_ipython as _kernel_shell

    _ip = _kernel_shell()
    _ip.input_transformers_cleanup[:] = [
        t for t in _ip.input_transformers_cleanup if getattr(t, "__name__", "") != "_route_to_isolated_runtime"
    ]
    if isinstance(globals().get("_DIMER_ISOLATED_RUNTIME"), IsolatedRuntime):
        _DIMER_ISOLATED_RUNTIME.close()
    _DIMER_ISOLATED_RUNTIME = IsolatedRuntime(ISOLATED_PYTHON)
    _ip.input_transformers_cleanup.append(_route_to_isolated_runtime)
    print(f"Every later code cell now runs in {ISOLATED_PYTHON} (pid {_DIMER_ISOLATED_RUNTIME.proc.pid}).")

Every later code cell now runs in /content/dimer_isolated_env/bin/python (pid 2306).


### Record the runtime

This is the first cell that runs in the isolated environment. Nothing is installed here: the locked environment already holds the pins listed in the cell. It records the notebook's source revision and the versions actually imported. Look for a dictionary reporting the notebook's source revision, the isolated Python (3.12.12), `torch`, `transformers` versions, and whether CUDA is available. (The pip install and stale-import guard in the cell are skipped: the isolated worker, like an executor that has pre-installed exactly these pins, sets `DIMER_NOTEBOOK_CI_PREINSTALLED=1`.)

In [ ]:
# @title Infrastructure: record the runtime
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
    'pyarrow==25.0.1',
]
NOTEBOOK_SOURCE = {
    'repository': 'layoutlm-document-qa-pipeline',
    'repository_revision': 'd14e37221df024341c4d136aea23ac56bdedb68f',
    'embedded_module': 'src/layoutlm_document_qa_pipeline/pipeline.py',
    'embedded_modules': ['src/layoutlm_document_qa_pipeline/pipeline.py', 'src/layoutlm_document_qa_pipeline/metrics.py', 'src/layoutlm_document_qa_pipeline/samples.py'],
    'module_sha256': '28edd4dc5c477d08ca1d009ea032be03f8e8a76453804afca103c433d1e3a7c4',
    'generator': 'build_notebook.py/2.1',
    'notebook_spec': '2.2',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

{'notebook_source': {'repository': 'layoutlm-document-qa-pipeline', 'repository_revision': 'd14e37221df024341c4d136aea23ac56bdedb68f', 'embedded_module': 'src/layoutlm_document_qa_pipeline/pipeline.py', 'embedded_modules': ['src/layoutlm_document_qa_pipeline/pipeline.py', 'src/layoutlm_document_qa_pipeline/metrics.py', 'src/layoutlm_document_qa_pipeline/samples.py'], 'module_sha256': '28edd4dc5c477d08ca1d009ea032be03f8e8a76453804afca103c433d1e3a7c4', 'generator': 'build_notebook.py/2.1', 'notebook_spec': '2.2'}, 'python': '3.12.12', 'torch': '2.14.0+cu130', 'transformers': '4.57.6', 'cuda': True}


## 2. Pipeline code (carried verbatim from `src/layoutlm_document_qa_pipeline/` @ `d14e37221df0`)

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/layoutlm_document_qa_pipeline/pipeline.py`

In [ ]:
# @title Infrastructure: carried module `src/layoutlm_document_qa_pipeline/pipeline.py` (you may run it without reading it)
"""Extractive document question answering with the pinned ``impira/layoutlm-document-qa`` checkpoint.

The class loads the tokenizer and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the LayoutLM architecture comes from the pinned ``transformers`` release,
the weights are SafeTensors, and no model-repository code is executed.

LayoutLM (v1) reads **words and their boxes**, not pixels: OCR is an input provider outside the model.
The pipeline therefore takes ``words``/``boxes`` from the caller (bring-your-own OCR) and offers
``ocr_words_with_tesseract`` as an optional adapter that is imported only when called.

The adaptation contract (``check_fit``, ``evaluate``, ``adapt``, ``save_artifact``, ``from_artifact``)
fine-tunes the last encoder blocks plus the span head on a validated ``{id, question, words, boxes,
image_size, answer_start, answer_end}`` dataset with validation-ANLS epoch selection and exports the trained
tensors as a safetensors adapter bound to the pinned base weights. The inference contract above is unchanged.
"""

from __future__ import annotations

import hashlib
import json
import math
import re
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from PIL import Image

MODEL_ID = "impira/layoutlm-document-qa"
MODEL_REVISION = "beed3c4d02d86017ebca5bd0fdf210046b907aa6"
MODEL_LICENSE = "mit"
MODEL_KEY = "layoutlm-document-qa"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
WEIGHT_FILE = "model.safetensors"
WEIGHT_SHA256 = (
    "e4bbad3e4a1b5ae50c787b7afd6049a0bfa99fd823b50436e444e092ae2347b9"  # manifest digest of WEIGHT_FILE
)
PARAMETER_COUNT = 127_792_898
ENCODER_LAYERS = 12  # config.json num_hidden_layers
DEFAULT_TRAINABLE_ENCODER_LAYERS = 4  # the last four encoder blocks plus the span head (28,353,026 params)
MAX_EVAL_RECORDS = 2_000
MAX_RECORDS_FIT = 20_000  # check_fit accepts a whole dataset before it is split
MIN_SCORED_RECORDS = 50  # below this a scored set is labelled a small sample
ARTIFACT_FORMAT = "org.valcorza.layoutlm-document-qa.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"

# Encoding ceilings. The checkpoint's max_position_embeddings is 514 (RoBERTa layout: 512 usable
# tokens); longer documents are split into overlapping windows of MAX_SEQ_LEN with DOC_STRIDE overlap
# and the best-scoring span across windows is returned (the transformers document-question-answering
# pipeline's convention, as is MAX_ANSWER_TOKENS).
MAX_SEQ_LEN = 512
DOC_STRIDE = 128
MAX_ANSWER_TOKENS = 15
MAX_WORDS = 2000
MAX_QUESTION_CHARS = 256
# Box ceilings. Boxes are pixel xyxy in the page's coordinate frame and are normalised to the
# 0..1000 grid LayoutLM expects (max_2d_position_embeddings 1024).
MAX_IMAGE_SIDE = 10000
MIN_IMAGE_SIDE = 1
BOX_GRID = 1000
# ANLS (DocVQA's official metric): a normalised Levenshtein similarity below this threshold scores 0.
ANLS_THRESHOLD = 0.5
_PUNCT_RE = re.compile(r"[^\w\s]")


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def normalize_answer(text: str) -> str:
    """DocVQA-style normalisation: lower-case, punctuation removed, whitespace collapsed."""
    return " ".join(_PUNCT_RE.sub(" ", text.lower()).split())


def _levenshtein(a: str, b: str) -> int:
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        current = [i]
        for j, cb in enumerate(b, 1):
            current.append(min(current[-1] + 1, previous[j] + 1, previous[j - 1] + (ca != cb)))
        previous = current
    return previous[-1]


def anls(prediction: str, golds: Sequence[str], *, threshold: float = ANLS_THRESHOLD) -> float:
    """Average Normalised Levenshtein Similarity for one question (Biten et al., ICDAR 2019).

    ``1 - lev(pred, gold) / max(len(pred), len(gold))`` over normalised strings, maximised over the
    accepted ``golds``; a similarity below ``threshold`` scores 0 so a near-miss is not rewarded.
    """
    if not golds:
        raise ValueError("golds must contain at least one accepted answer")
    pred = normalize_answer(prediction)
    best = 0.0
    for gold in golds:
        ref = normalize_answer(gold)
        longest = max(len(pred), len(ref))
        similarity = 1.0 if longest == 0 else 1.0 - _levenshtein(pred, ref) / longest
        best = max(best, similarity)
    return best if best >= threshold else 0.0


def exact_match(prediction: str, golds: Sequence[str]) -> bool:
    """Whether the normalised prediction equals any normalised accepted answer."""
    pred = normalize_answer(prediction)
    return any(pred == normalize_answer(gold) for gold in golds)


def normalize_box(box: Sequence[float], width: int, height: int) -> list[int]:
    """Pixel xyxy -> LayoutLM's 0..1000 grid (the transformers pipeline's normalize_bbox)."""
    x0, y0, x1, y1 = (float(v) for v in box)
    return [
        int(BOX_GRID * (x0 / width)),
        int(BOX_GRID * (y0 / height)),
        int(BOX_GRID * (x1 / width)),
        int(BOX_GRID * (y1 / height)),
    ]


def ocr_words_with_tesseract(image: Image.Image, *, lang: str = "eng") -> tuple[list[str], list[list[float]]]:
    """Optional OCR adapter: words and pixel xyxy boxes from Tesseract via ``pytesseract``.

    Neither ``pytesseract`` nor the Tesseract binary is part of this package's pinned runtime; the
    import happens here so the rest of the pipeline stays usable with any OCR the caller prefers.
    """
    try:
        import pytesseract
    except ImportError as exc:  # pragma: no cover - depends on the host
        raise RuntimeError("pytesseract is not installed; supply words and boxes from your own OCR") from exc
    data = pytesseract.image_to_data(image.convert("RGB"), lang=lang, output_type=pytesseract.Output.DICT)
    words: list[str] = []
    boxes: list[list[float]] = []
    for text, left, top, w, h in zip(
        data["text"], data["left"], data["top"], data["width"], data["height"], strict=True
    ):
        token = str(text).strip()
        if not token:
            continue
        words.append(token)
        boxes.append([float(left), float(top), float(left + w), float(top + h)])
    return words, boxes


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "one question string plus the page's OCR words (list of str) with one pixel xyxy box per word "
        "and the page size (width, height) the boxes are expressed in; pixels are not read by the model"
    ),
    "words": [1, MAX_WORDS],
    "question_chars": [1, MAX_QUESTION_CHARS],
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "encoding": (
        f"<s> question </s></s> words </s> with RoBERTa byte-level BPE; boxes normalised to 0..{BOX_GRID}; "
        f"windows of {MAX_SEQ_LEN} tokens with {DOC_STRIDE}-token overlap when the words do not fit"
    ),
    "output": (
        f"the best word span of at most {MAX_ANSWER_TOKENS} tokens with its start/end word indices and the "
        "span score (softmax start x softmax end within the window)"
    ),
}


def _check_page(image_size: Any) -> tuple[int, int]:
    if (
        not isinstance(image_size, Sequence)
        or isinstance(image_size, str)
        or len(image_size) != 2
        or any(isinstance(v, bool) or not isinstance(v, int) for v in image_size)
    ):
        raise TypeError("image_size must be a (width, height) pair of ints")
    width, height = int(image_size[0]), int(image_size[1])
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return width, height


def _check_document(
    words: Any, boxes: Any, image_size: Any
) -> tuple[list[str], list[list[int]], tuple[int, int]]:
    """Raise TypeError/ValueError naming the first violated ceiling; return words, grid boxes, size."""
    width, height = _check_page(image_size)
    if isinstance(words, str) or not isinstance(words, Sequence):
        raise TypeError("words must be a sequence of str")
    if not 1 <= len(words) <= MAX_WORDS:
        raise ValueError(f"words has {len(words)} entries; expected 1..MAX_WORDS={MAX_WORDS}")
    if not all(isinstance(word, str) and word.strip() for word in words):
        raise ValueError("every word must be a non-empty str")
    if isinstance(boxes, str) or not isinstance(boxes, Sequence) or len(boxes) != len(words):
        raise ValueError(f"boxes must have one pixel xyxy box per word ({len(words)})")
    grid: list[list[int]] = []
    for index, box in enumerate(boxes):
        if isinstance(box, str) or not isinstance(box, Sequence) or len(box) != 4:
            raise ValueError(f"boxes[{index}] must be [x0, y0, x1, y1]")
        try:
            x0, y0, x1, y1 = (float(v) for v in box)
        except (TypeError, ValueError) as exc:
            raise ValueError(f"boxes[{index}] must hold numbers") from exc
        if not (0 <= x0 <= x1 <= width and 0 <= y0 <= y1 <= height):
            raise ValueError(f"boxes[{index}] {list(box)} is not inside the {width}x{height} page")
        grid.append(normalize_box((x0, y0, x1, y1), width, height))
    return [str(word) for word in words], grid, (width, height)


def _check_question(question: Any) -> str:
    if not isinstance(question, str):
        raise TypeError("question must be a str")
    checked = " ".join(question.split())
    if not checked:
        raise ValueError("question must contain at least one non-whitespace character")
    if len(checked) > MAX_QUESTION_CHARS:
        raise ValueError(f"question has {len(checked)} chars > MAX_QUESTION_CHARS {MAX_QUESTION_CHARS}")
    return checked


def validate_inputs(
    words: Sequence[str],
    boxes: Sequence[Sequence[float]],
    questions: Sequence[str],
    *,
    image_size: Sequence[int],
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Every question is checked exactly as ``answer`` would check it; rejection is reported by raising,
    and a caller that wants the finding recorded catches the exception and stores ``str(exc)`` under
    ``findings``.
    """
    checked_words, _grid, size = _check_document(words, boxes, image_size)
    if isinstance(questions, str) or not isinstance(questions, Sequence) or not questions:
        raise TypeError("questions must be a non-empty sequence of str")
    checked = [_check_question(question) for question in questions]
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (answer takes one page)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {"id": names[0] if names else "page-0", "size": list(size), "n_words": len(checked_words)}
        ],
        "questions": checked,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    results: Sequence[Mapping[str, Any]],
    golds: Sequence[Sequence[str]] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``golds`` (one sequence of accepted answers per result, in order) the report carries the
    mean ``anls`` and the ``exact_match`` rate over the questions plus one per-question entry, verdict
    ``sample-sanity``; without golds it is ``not-measurable`` and says what labelled data would make
    the task measurable.
    """
    if not results:
        raise ValueError("results must contain at least one answer result")
    base = {
        "task": "question + OCR words/boxes -> extractive answer span (LayoutLM v1)",
        "score_semantics": (
            "score is the product of the start and end softmax probabilities of the chosen span within "
            "its window under the model's own head — a ranking signal over spans of this page, not a "
            "calibrated probability that the answer is right, and never a signal that the question is "
            "answerable; the model always returns its best span"
        ),
        "sample_kind": sample_kind,
        "n_questions": len(results),
        "scores": [float(result.get("score", 0.0)) for result in results],
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if golds is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no accepted answers were supplied for the evaluated questions",
            "needs": (
                "question/answer pairs with accepted answers on pages from the deployment domain "
                "(DocVQA-style annotations) scored with ANLS, plus the OCR the deployment will really use; "
                "no such labelled set ships with this repository"
            ),
        }
    if len(golds) != len(results):
        raise ValueError(f"golds has {len(golds)} entries for {len(results)} results")
    per_question = []
    for result, accepted in zip(results, golds, strict=True):
        if isinstance(accepted, str) or not accepted:
            raise ValueError("each golds entry must be a non-empty sequence of accepted answers")
        prediction = str(result["answer"])
        per_question.append(
            {
                "question": result.get("question"),
                "prediction": prediction,
                "score": float(result.get("score", 0.0)),
                "golds": list(accepted),
                "anls": anls(prediction, accepted),
                "exact_match": exact_match(prediction, accepted),
            }
        )
    metrics = [
        {
            "id": "anls",
            "value": sum(entry["anls"] for entry in per_question) / len(per_question),
            "threshold": ANLS_THRESHOLD,
            "normalisation": "lower-cased, punctuation removed, whitespace collapsed; max over golds",
            "estimation": f"{len(per_question)} question(s) on one page, no dispersion estimate",
        },
        {
            "id": "exact_match",
            "value": sum(entry["exact_match"] for entry in per_question) / len(per_question),
            "normalisation": "lower-cased, punctuation removed, whitespace collapsed",
            "estimation": f"{len(per_question)} question(s) on one page, no dispersion estimate",
        },
    ]
    return {
        **base,
        "metrics": metrics,
        "per_question": per_question,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(per_question)} authored question(s) on one tutorial page whose words and boxes you "
            "rendered yourself; plumbing evidence, not a DocVQA benchmark"
        ),
        "needs": (
            "a labelled question/answer set on pages from the deployment domain with the deployment's own "
            "OCR for any accuracy claim; the DocVQA benchmark itself is registration-gated and not bundled"
        ),
    }


def _window_bboxes(
    encoding: Any, window: int, grid_boxes: Sequence[Sequence[int]], sep_id: int
) -> list[list[int]]:
    """One 0..1000 box per token of a window: word tokens take their word's box, separators `[1000]*4`,
    everything else (the question and padding) `[0]*4` — the transformers document-question-answering
    pipeline's convention."""
    bbox = []
    for input_id, sequence_id, word_id in zip(
        encoding["input_ids"][window].tolist(),
        encoding.sequence_ids(window),
        encoding.word_ids(window),
        strict=True,
    ):
        if sequence_id == 1:
            bbox.append(list(grid_boxes[word_id]))
        elif input_id == sep_id:
            bbox.append([BOX_GRID] * 4)
        else:
            bbox.append([0] * 4)
    return bbox


@dataclass
class LayoutLMDocumentQAPipeline:
    """``_runner(question, words, grid_boxes)`` returns ``{"start": int, "end": int, "score": float}``
    (word indices, inclusive) or ``{"start": None, ...}`` when no span could be selected.
    ``_count_windows(question, words)`` returns how many 512-token windows the pair needs (1 = fits); both
    are injectable so the offline tests run without the model."""

    _runner: Callable[..., dict[str, Any]]
    device: str = "cpu"
    dtype: str = "float32"
    source: str = "injected"
    _count_windows: Callable[[str, list[str]], int] | None = field(default=None, repr=False)
    adapter: dict[str, Any] | None = field(default=None, repr=False)
    _model: Any = field(default=None, repr=False)
    _tokenizer: Any = field(default=None, repr=False)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> LayoutLMDocumentQAPipeline:
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        common: dict[str, Any] = {"trust_remote_code": False}
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, common["local_files_only"], source = str(root), True, "local-snapshot"
        elif allow_download:
            location, common["revision"], source = MODEL_ID, MODEL_REVISION, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import AutoTokenizer, LayoutLMForQuestionAnswering

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        tokenizer = AutoTokenizer.from_pretrained(location, **common)
        if not tokenizer.is_fast:
            raise RuntimeError("a fast tokenizer is required for word_ids/sequence_ids; snapshot has none")
        model = LayoutLMForQuestionAnswering.from_pretrained(location, dtype=torch.float32, **common)
        model = model.eval().to(resolved_device)
        for param in model.parameters():
            param.requires_grad_(False)
        sep_id = tokenizer.sep_token_id

        def encode(question: str, words: list[str]) -> Any:
            return tokenizer(
                text=question.split(),
                text_pair=words,
                is_split_into_words=True,
                max_length=MAX_SEQ_LEN,
                stride=DOC_STRIDE,
                truncation="only_second",
                return_overflowing_tokens=True,
                return_token_type_ids=True,
                padding="max_length",
                return_tensors="pt",
            )

        def count_windows(question: str, words: list[str]) -> int:
            return int(encode(question, words)["input_ids"].shape[0])

        def runner(question: str, words: list[str], grid_boxes: list[list[int]]) -> dict[str, Any]:
            encoding = encode(question, words)
            n_windows = int(encoding["input_ids"].shape[0])
            best: dict[str, Any] = {"start": None, "end": None, "score": 0.0, "n_windows": n_windows}
            model_device = next(model.parameters()).device
            for window in range(n_windows):
                sequence_ids = encoding.sequence_ids(window)
                word_ids = encoding.word_ids(window)
                inputs = {
                    "input_ids": encoding["input_ids"][window].unsqueeze(0).to(model_device),
                    "attention_mask": encoding["attention_mask"][window].unsqueeze(0).to(model_device),
                    "token_type_ids": encoding["token_type_ids"][window].unsqueeze(0).to(model_device),
                    "bbox": torch.tensor(_window_bboxes(encoding, window, grid_boxes, sep_id))
                    .unsqueeze(0)
                    .to(model_device),
                }
                with torch.inference_mode():
                    outputs = model(**inputs)
                # Only document tokens may start or end an answer (the pipeline's p_mask).
                allowed = torch.tensor([sid == 1 for sid in sequence_ids], device=model_device)
                start = outputs.start_logits[0].float().masked_fill(~allowed, float("-inf")).softmax(-1)
                end = outputs.end_logits[0].float().masked_fill(~allowed, float("-inf")).softmax(-1)
                candidates = start[:, None] * end[None, :]
                candidates = torch.triu(candidates) - torch.triu(candidates, diagonal=MAX_ANSWER_TOKENS)
                flat = int(candidates.argmax())
                s_index, e_index = divmod(flat, candidates.shape[1])
                score = float(candidates[s_index, e_index])
                if score > best["score"] and word_ids[s_index] is not None and word_ids[e_index] is not None:
                    best = {
                        "start": word_ids[s_index],
                        "end": word_ids[e_index],
                        "score": score,
                        "n_windows": n_windows,
                    }
            return best

        return cls(
            runner, resolved_device, "float32", source, count_windows, _model=model, _tokenizer=tokenizer
        )

    def answer(
        self,
        question: str,
        *,
        words: Sequence[str],
        boxes: Sequence[Sequence[float]],
        image_size: Sequence[int],
    ) -> dict[str, Any]:
        """Answer one question from the page's words and boxes; ``answer`` is the selected word span."""
        checked_words, grid, size = _check_document(words, boxes, image_size)
        checked_question = _check_question(question)
        raw = self._runner(checked_question, checked_words, grid)
        if not isinstance(raw, dict) or "start" not in raw or "end" not in raw or "score" not in raw:
            raise RuntimeError("runner must return a dict with 'start', 'end' and 'score'")
        start, end = raw["start"], raw["end"]
        if start is not None and not (0 <= int(start) <= int(end) < len(checked_words)):
            raise RuntimeError(f"runner returned an invalid word span {start}..{end}")
        span_words = checked_words[start : end + 1] if start is not None else []
        return {
            "answer": " ".join(span_words),
            "score": float(raw["score"]),
            "start": None if start is None else int(start),
            "end": None if end is None else int(end),
            "question": checked_question,
            "n_words": len(checked_words),
            "n_windows": int(raw.get("n_windows", 1)),
            "image_size": list(size),
            "device": self.device,
            "dtype": self.dtype,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # ---- adaptation contract ---------------------------------------------------------------------------

    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._tokenizer is None:
            raise ValueError(
                "this operation needs a pipeline built with from_pretrained() or from_artifact()"
            )
        return self._model, self._tokenizer

    def check_fit(self, records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
        """Split validated records into those whose question + page fit one 512-token window and those that
        would be windowed at inference (a windowed page is answerable but is not trained on; nothing is
        truncated)."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if self._count_windows is None:
            raise ValueError("check_fit needs a pipeline built with from_pretrained() or from_artifact()")
        checked = validate_dataset(records, min_records=1, max_records=MAX_RECORDS_FIT)["records"]
        fitting, dropped = [], []
        for record in checked:
            if self._count_windows(record["question"], list(record["words"])) == 1:
                fitting.append(record)
            else:
                dropped.append(record["id"])
        return {"fitting": fitting, "dropped": dropped, "n_fitting": len(fitting), "n_dropped": len(dropped)}

    def evaluate(self, records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
        """Answer every record and score the predictions against its accepted answers (mean ANLS, exact-match
        rate)."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import docqa_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        started = time.perf_counter()
        predictions = [
            self.answer(r["question"], words=r["words"], boxes=r["boxes"], image_size=r["image_size"])[
                "answer"
            ]
            for r in checked
        ]
        metrics = docqa_metrics(predictions, [[str(a) for a in r["answers"]] for r in checked])
        metrics.update(
            {
                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
                "adapted": self.adapter is not None,
                "seconds": round(time.perf_counter() - started, 3),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            }
        )
        return metrics

    def _trainable_names(self, trainable_encoder_layers: int) -> list[str]:
        if (
            isinstance(trainable_encoder_layers, bool)
            or not isinstance(trainable_encoder_layers, int)
            or not 1 <= trainable_encoder_layers <= ENCODER_LAYERS
        ):
            raise ValueError(f"trainable_encoder_layers must be an int in 1..{ENCODER_LAYERS}")
        model, _ = self._require_model()
        first = ENCODER_LAYERS - trainable_encoder_layers
        prefixes = tuple(f"layoutlm.encoder.layer.{k}." for k in range(first, ENCODER_LAYERS)) + (
            "qa_outputs.",
        )
        return [name for name, _p in model.named_parameters() if name.startswith(prefixes)]

    @staticmethod
    def _span_positions(encoded: Any, index: int, record: Mapping[str, Any]) -> tuple[int, int]:
        """Token start/end of the gold word span inside the page segment of one encoded example."""
        start = end = None
        for pos, (sid, word_id) in enumerate(
            zip(encoded.sequence_ids(index), encoded.word_ids(index), strict=True)
        ):
            if sid != 1:
                continue
            if start is None and word_id == record["answer_start"]:
                start = pos
            if word_id == record["answer_end"]:
                end = pos
        if start is None or end is None or end < start:
            raise ValueError(f"record {record['id']}: gold span does not map onto the encoded page tokens")
        return start, end

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 6,
        lr: float = 3e-5,
        batch_size: int = 16,
        trainable_encoder_layers: int = DEFAULT_TRAINABLE_ENCODER_LAYERS,
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded supervised fine-tuning on a validated document-QA dataset.

        Only the last `trainable_encoder_layers` encoder blocks and the span head train (4 blocks by default:
        28,353,026 of 127,792,898 parameters; the word, position and 2-D box embeddings and the earlier blocks
        stay frozen). Start/end cross-entropy on the gold word span's first and last tokens, AdamW at a fixed
        learning rate with gradient clipping at 1.0, dynamic padding, no scheduler; every training record must
        fit one window (`check_fit`). Epoch 0 records the frozen model's validation ANLS; the epoch with the
        highest validation ANLS is kept.

        An already adapted pipeline (or one with a loaded artifact) is refused: training would continue from
        the adapted weights, record them as the "frozen model" at epoch 0, and export an artifact that leaves
        out any block the first adaptation changed but this one does not train. Build a fresh pipeline with
        `from_pretrained()` first."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 20:
            raise ValueError("epochs must be an int in 1..20")
        if not (0.0 < lr <= 1e-3):
            raise ValueError("lr must be in (0, 1e-3]")
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 64:
            raise ValueError("batch_size must be an int in 1..64")
        if self.adapter is not None:
            raise ValueError(
                "this pipeline is already adapted; adapt() starts from the pretrained base, so build a fresh "
                "pipeline with from_pretrained() first"
            )
        names = self._trainable_names(trainable_encoder_layers)
        train_checked = validate_dataset(train)["records"]
        val_checked = (
            validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS)["records"] if val else []
        )
        if self._count_windows is not None:
            over = [
                r["id"] for r in train_checked if self._count_windows(r["question"], list(r["words"])) != 1
            ]
            if over:
                raise ValueError(
                    f"{len(over)} training record(s) need more than one window (use check_fit): {over[:5]}"
                )
        import torch

        torch.manual_seed(seed)
        model, tokenizer = self._require_model()
        sep_id = tokenizer.sep_token_id
        started = time.perf_counter()
        wanted = set(names)
        for name, param in model.named_parameters():
            param.requires_grad_(name in wanted)
        params = [p for p in model.parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in params)
        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
        device = next(model.parameters()).device

        def score_val() -> dict[str, Any] | None:
            if not val_checked:
                return None
            model.eval()
            return {k: v for k, v in self.evaluate(val_checked).items() if k in ("anls", "exact_match", "n")}

        history: list[dict[str, Any]] = []
        entry: dict[str, Any] = {"epoch": 0, "train_loss": None, "val": score_val(), "note": "frozen model"}
        history.append(entry)
        if progress:
            progress(entry)
        best_anls = entry["val"]["anls"] if entry["val"] else -math.inf
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
        initial_state = {k: v.clone() for k, v in best_state.items()}
        best_epoch = 0
        generator = torch.Generator().manual_seed(seed)
        try:
            for epoch in range(1, epochs + 1):
                model.train()
                order = torch.randperm(len(train_checked), generator=generator).tolist()
                losses = []
                for start in range(0, len(order), batch_size):
                    batch = [train_checked[i] for i in order[start : start + batch_size]]
                    encoded = tokenizer(
                        [r["question"].split() for r in batch],
                        [list(r["words"]) for r in batch],
                        is_split_into_words=True,
                        padding=True,
                        truncation="only_second",
                        max_length=MAX_SEQ_LEN,
                        return_token_type_ids=True,
                        return_tensors="pt",
                    )
                    grids = [
                        [normalize_box(box, r["image_size"][0], r["image_size"][1]) for box in r["boxes"]]
                        for r in batch
                    ]
                    bbox = torch.tensor(
                        [_window_bboxes(encoded, i, grids[i], sep_id) for i in range(len(batch))]
                    )
                    positions = [self._span_positions(encoded, i, r) for i, r in enumerate(batch)]
                    out = model(
                        input_ids=encoded["input_ids"].to(device),
                        attention_mask=encoded["attention_mask"].to(device),
                        token_type_ids=encoded["token_type_ids"].to(device),
                        bbox=bbox.to(device),
                        start_positions=torch.tensor([s for s, _e in positions], device=device),
                        end_positions=torch.tensor([e for _s, e in positions], device=device),
                    )
                    optimiser.zero_grad(set_to_none=True)
                    out.loss.backward()
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    optimiser.step()
                    losses.append(float(out.loss.detach()))
                model.eval()
                entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses), "val": score_val()}
                history.append(entry)
                if progress:
                    progress(entry)
                current = entry["val"]["anls"] if entry["val"] else math.inf
                if current > best_anls or not entry["val"]:
                    best_anls = current
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
                    best_epoch = epoch
        except BaseException:
            # Transactional: a failure in training, validation or the progress callback leaves the base
            # exactly as it was, with every parameter frozen again.
            restore = dict(model.state_dict())
            restore.update(initial_state)
            model.load_state_dict(restore, strict=True)
            model.eval()
            for param in model.parameters():
                param.requires_grad_(False)
            self.adapter = None
            raise
        merged = dict(model.state_dict())
        merged.update(best_state)
        model.load_state_dict(merged, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "trainable_encoder_layers": trainable_encoder_layers,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "selection": "highest validation ANLS" if val_checked else "final epoch (no validation split)",
            "lr": lr,
            "batch_size": batch_size,
            "n_train": len(train_checked),
            "n_val": len(val_checked),
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    # ---- artifacts ------------------------------------------------------------------------------------

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the adapted encoder-block and span-head tensors as safetensors plus a base manifest."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        model, _ = self._require_model()
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items() if k in names}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "id": MODEL_ID,
                "revision": MODEL_REVISION,
                "key": MODEL_KEY,
                "weight_file": WEIGHT_FILE,
                "weight_sha256": WEIGHT_SHA256,
            },
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [
                {
                    "path": ARTIFACT_WEIGHTS_NAME,
                    "bytes": weights_path.stat().st_size,
                    "sha256": _sha256(weights_path),
                }
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        return out

    def _check_artifact_manifest(self, root: Path, manifest: Mapping[str, Any]) -> Path:
        """Refuse an artifact whose manifest is not exactly the one this pipeline writes: the supported format
        and version, the pinned base (id, revision, weight file, digest), exactly one file entry named
        `adapter.safetensors` that resolves inside the artifact directory, and a recorded
        `trainable_encoder_layers` in range. Nothing is deserialised here. The digest check that follows
        detects corruption or drift of the weights relative to the adjacent manifest; it is not authenticity
        against an actor who can replace both files."""
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if manifest.get("format_version") != ARTIFACT_FORMAT_VERSION:
            raise ValueError(
                f"artifact format_version {manifest.get('format_version')!r} is not the supported "
                f"{ARTIFACT_FORMAT_VERSION!r}"
            )
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision"), base.get("weight_sha256")) != (
            MODEL_ID,
            MODEL_REVISION,
            WEIGHT_SHA256,
        ):
            raise ValueError("artifact was adapted from a different base model, revision or weight file")
        if base.get("weight_file", WEIGHT_FILE) != WEIGHT_FILE:
            raise ValueError("artifact was adapted from a different base weight file")
        files = manifest.get("files")
        if not isinstance(files, list) or len(files) != 1:
            raise ValueError("artifact manifest must list exactly one file")
        entry = files[0]
        if not isinstance(entry, Mapping) or entry.get("path") != ARTIFACT_WEIGHTS_NAME:
            raise ValueError(f"artifact manifest must name exactly {ARTIFACT_WEIGHTS_NAME!r}")
        weights_path = (root / entry["path"]).resolve()
        if weights_path.parent != root.resolve():
            raise ValueError("artifact weight path must resolve inside the artifact directory")
        adapter = manifest.get("adapter")
        layers = adapter.get("trainable_encoder_layers") if isinstance(adapter, Mapping) else None
        if isinstance(layers, bool) or not isinstance(layers, int) or not 1 <= layers <= ENCODER_LAYERS:
            raise ValueError("artifact manifest does not record an in-range integer trainable_encoder_layers")
        if not isinstance(manifest.get("tensors"), list):
            raise ValueError("artifact manifest must list its tensors")
        return weights_path

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest, digest and exact tensor set **before** deserialising, then overwrite
        exactly the tensors it carries."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        weights_path = self._check_artifact_manifest(root, manifest)
        entry = manifest["files"][0]
        if not weights_path.is_file():
            raise FileNotFoundError(f"artifact weights missing: {weights_path}")
        if _sha256(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        # The exact tensor set the recorded configuration implies — no subset, no extra, no other layer.
        expected = sorted(self._trainable_names(manifest["adapter"]["trainable_encoder_layers"]))
        if sorted(manifest["tensors"]) != expected:
            raise ValueError("artifact tensor list does not match its recorded configuration")
        model, _ = self._require_model()
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != expected:
            raise ValueError("artifact tensor names differ from its manifest")
        state = model.state_dict()
        for key, value in tensors.items():
            if key not in state or not (
                key.startswith("layoutlm.encoder.layer.") or key.startswith("qa_outputs.")
            ):
                raise ValueError(
                    f"artifact tensor {key} is not an adaptable encoder or span-head tensor of the base"
                )
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(
                    f"artifact tensor {key} has shape {tuple(value.shape)}, "
                    f"base has {tuple(state[key].shape)}"
                )
        merged = dict(state)
        merged.update({k: v.to(state[k].dtype) for k, v in tensors.items()})
        model.load_state_dict(merged, strict=True)
        model.eval()
        self.adapter = {
            **manifest["adapter"],
            "trainable_names": manifest["tensors"],
            "history": manifest.get("history", []),
        }
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> LayoutLMDocumentQAPipeline:
        pipeline = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 2/3:** `src/layoutlm_document_qa_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
# @title Infrastructure: carried module `src/layoutlm_document_qa_pipeline/metrics.py` (you may run it without reading it)
"""Corpus-level document-QA metrics and two non-neural baselines.

The per-question helpers (`normalize_answer`, `anls`, `exact_match`) live in `pipeline.py` and follow DocVQA's
conventions. This module averages them over a dataset and adds two baselines a fine-tuned reader must beat:
**last number** (the last word on the page that contains a digit — receipts end with their totals) and
**keyword lookup** (find the page word that best matches a content word of the question, e.g. `TOTAL` for
"What is the total amount?", and answer with the next word after it that contains a digit — a lookup with no
model that stands in for the "find the label, read the value" heuristic a person applies to a receipt).
"""

from __future__ import annotations

import re
from collections.abc import Mapping, Sequence
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import anls, exact_match, normalize_answer` removed — names are kernel globals defined by the carried modules

_DIGIT_RE = re.compile(r"\d")
# Function words removed from a question before its remaining words are matched against the page.
QUESTION_STOP_WORDS = frozenset(
    [
        "what", "is", "the", "how", "much", "many", "was", "were", "paid", "given", "by", "of", "name",
        "first", "item", "items", "bought", "amount", "charge",
    ]
)  # fmt: skip
METRIC_DEFINITIONS = {
    "anls": (
        "mean over questions of the Average Normalised Levenshtein Similarity between the prediction and its "
        "best-matching accepted answer (lower-cased, punctuation removed, whitespace collapsed; a similarity "
        "below 0.5 scores 0); in 0..1"
    ),
    "exact_match": (
        "fraction of questions whose normalised prediction equals a normalised accepted answer; in 0..1"
    ),
}


def docqa_metrics(predictions: Sequence[str], golds: Sequence[Sequence[str]]) -> dict[str, Any]:
    """Mean ANLS and exact-match rate over parallel predictions and accepted-answer lists."""
    if len(predictions) != len(golds):
        raise ValueError(f"{len(predictions)} predictions but {len(golds)} gold lists")
    if not predictions:
        raise ValueError("no predictions to score")
    scores = [anls(p, g) for p, g in zip(predictions, golds, strict=True)]
    exact = [exact_match(p, g) for p, g in zip(predictions, golds, strict=True)]
    return {
        "n": len(predictions),
        "anls": sum(scores) / len(scores),
        "exact_match": sum(exact) / len(exact),
        "empty_rate": sum(1 for p in predictions if not p.strip()) / len(predictions),
        "definitions": dict(METRIC_DEFINITIONS),
    }


def _golds(records: Sequence[Mapping[str, Any]]) -> list[list[str]]:
    return [[str(a) for a in r["answers"]] for r in records]


def last_number_answer(words: Sequence[str]) -> str:
    """The last word on the page containing a digit; empty when there is none."""
    for word in reversed(list(words)):
        if _DIGIT_RE.search(word):
            return word
    return ""


def last_number_baseline(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Answer every question with the last numeric word of its page."""
    result = docqa_metrics([last_number_answer(r["words"]) for r in records], _golds(records))
    result["baseline"] = "last word on the page containing a digit"
    return result


def _overlap(a: str, b: str) -> int:
    """Length of the longest common prefix of two normalised words (`subtotal` vs `subtot` → 6)."""
    n = 0
    for x, y in zip(a, b, strict=False):
        if x != y:
            break
        n += 1
    return n


def keyword_lookup_answer(question: str, words: Sequence[str], *, min_overlap: int = 3) -> str:
    """Find the page word sharing the longest prefix (at least `min_overlap` characters, last such word on
    ties) with any content word of the question; answer with the first word after it that contains a digit,
    or the last numeric word of the page when no keyword matches."""
    content = [w for w in normalize_answer(question).split() if w not in QUESTION_STOP_WORDS]
    page = [normalize_answer(w) for w in words]
    best_index, best_overlap = None, min_overlap - 1
    for index, word in enumerate(page):
        overlap = max((_overlap(word, q) for q in content), default=0)
        if overlap >= best_overlap and overlap >= min_overlap:
            best_index, best_overlap = index, overlap
    if best_index is not None:
        for word in list(words)[best_index + 1 :]:
            if _DIGIT_RE.search(word):
                return word
    return last_number_answer(words)


def keyword_lookup_baseline(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """A label-then-value lookup with no model."""
    predictions = [keyword_lookup_answer(r["question"], r["words"]) for r in records]
    result = docqa_metrics(predictions, _golds(records))
    result["baseline"] = "keyword lookup (best-matching page word, then the next numeric word)"
    return result

**Module 3/3:** `src/layoutlm_document_qa_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
# @title Infrastructure: carried module `src/layoutlm_document_qa_pipeline/samples.py` (you may run it without reading it)
"""Document-QA dataset contract for fine-tuning: the pinned CORD-v2 receipt sample, validation, seeded
page-disjoint splitting, BYOD loaders and JSONL export.

The default dataset is **real**: CORD-v2 (Park et al., 2019; NAVER Clova; CC BY 4.0) — photographed Indonesian
receipts whose every printed word carries a pixel box and a field category (`total.total_price`,
`sub_total.tax_price`, `menu.nm`, …). LayoutLM reads words and boxes, never pixels, so this module fetches
**only the `ground_truth` column** of the two pinned Hub parquet files (the 100-receipt `test` and
`validation` shards): the parquet footer and the one column chunk are read over HTTP range requests through
`pyarrow` (about 0.4 MB of the 476 MB the two files hold with their images). Each file is pinned three
ways — the immutable dataset revision in the URL, the byte size and SHA-256 the Hub declares for the file
(checked against the file metadata before any byte is read), and a SHA-256 of the decoded column (checked
after reading) — and refused on any mismatch. Questions are templated from the field categories: a category
that occurs on exactly one line of a receipt becomes one question whose gold answer is that line's value
words, so every answer is a contiguous span of the page's OCR words by construction.

A record is ``{id, page_id, question, words, boxes, image_size, answer_start, answer_end, answers}`` — the
page's words with one pixel xyxy box each and the inclusive word indices of the gold span; ``answers`` holds
the span text. Every question on the same page lands in the same split.
"""

from __future__ import annotations

import hashlib
import io
import json
import random
import re
import urllib.request
from collections.abc import Callable, Mapping, Sequence
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import MAX_QUESTION_CHARS, MODEL_ID, _check_document, _check_question` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "CORD-v2"
CORPUS_REPO = "naver-clova-ix/cord-v2"
CORPUS_REVISION = "7f0115a4b758a71d6473b8d085751692da2fef98"
CORPUS_RELEASE = "Hugging Face Hub dataset revision 7f0115a4 (2022-07-19)"
CORPUS_LICENSE = "CC BY 4.0 (Park et al. 2019; NAVER Clova; huggingface.co/datasets/naver-clova-ix/cord-v2)"
CORPUS_COLUMN = "ground_truth"
# The two shards read: path, size and SHA-256 as the Hub declares them for the LFS object (the whole file is
# never downloaded, so these are checked against the file metadata), the row count, and the SHA-256 of the
# decoded `ground_truth` column (canonical JSON list of the row strings), checked after the read.
CORPUS_FILES: dict[str, dict[str, Any]] = {
    "test": {
        "path": "data/test-00000-of-00001-9c204eb3f4e11791.parquet",
        "bytes": 234_202_795,
        "sha256": "51c65f1788faff392abe2a0b55b023eb23e9be551c509138eaa3a832514224e7",
        "rows": 100,
        "column_sha256": "b499e58aa298e5242b5222b7e333e92e6affb79a192def149d0a0aac3e41d9f1",
    },
    "validation": {
        "path": "data/validation-00000-of-00001-cc3c5779fe22e8ca.parquet",
        "bytes": 242_080_800,
        "sha256": "0d0f6dac11fdcc549de2746aa9f53136a3bc22a2a1aff2b0b847f7622ad60c15",
        "rows": 100,
        "column_sha256": "adf8303ec0295af1ef11a63dd0b72453e95399cfd2cd76e3d494c284730bdef9",
    },
}
DEFAULT_CACHE_DIR = Path("weights") / "cord-v2"
# One question per field category that occurs on exactly one line of the receipt; the gold answer is that
# line's words. `menu.nm` (item names) occurs on most receipts several times, so it asks for the *first*
# item — the topmost `menu.nm` line — and is skipped when that item's name spans more than one line.
QUESTION_TEMPLATES: dict[str, str] = {
    "total.total_price": "What is the total amount?",
    "sub_total.subtotal_price": "What is the subtotal?",
    "sub_total.tax_price": "What is the tax amount?",
    "sub_total.service_price": "What is the service charge?",
    "sub_total.discount_price": "What is the discount amount?",
    "total.cashprice": "How much cash was paid?",
    "total.changeprice": "How much change was given?",
    "total.creditcardprice": "How much was paid by card?",
    "total.menuqty_cnt": "How many items were bought?",
    "menu.nm": "What is the name of the first item?",
}
FIRST_ITEM_CATEGORY = "menu.nm"
SAMPLE_SEED = 42
SAMPLE_SPLIT = {
    "train": 119,
    "validation": 30,
    "test": 50,
}  # receipts (pages), not questions; 199 unique pages
MIN_RECORDS = 8
MAX_RECORDS = 20_000
# A BYOD split must leave records in every split the notebook scores: at least MIN_RECORDS training records
# and at least two validation (epoch selection) and two test (held-out evaluation) records.
MIN_VAL_RECORDS = 2
MIN_TEST_RECORDS = 2
BYOD_VAL_FRACTION = 0.15
BYOD_TEST_FRACTION = 0.2
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def column_digest(rows: Sequence[str]) -> str:
    """SHA-256 of the decoded column as a canonical JSON list of its row strings."""
    return _sha256_bytes(json.dumps(list(rows), ensure_ascii=False, separators=(",", ":")).encode("utf-8"))


class _HttpRangeFile(io.RawIOBase):
    """A seekable read-only view of one HTTPS object served with `Range` requests (what `pyarrow` needs to
    read a parquet footer and a single column chunk without downloading the file)."""

    def __init__(self, url: str, size: int) -> None:
        self.url, self.size, self.pos = url, size, 0
        self.fetched = 0

    def readable(self) -> bool:
        return True

    def seekable(self) -> bool:
        return True

    def tell(self) -> int:
        return self.pos

    def seek(self, offset: int, whence: int = 0) -> int:
        base = {0: 0, 1: self.pos, 2: self.size}[whence]
        self.pos = max(0, base + offset)
        return self.pos

    def read(self, n: int = -1) -> bytes:
        if n is None or n < 0:
            n = self.size - self.pos
        if n <= 0 or self.pos >= self.size:
            return b""
        end = min(self.size, self.pos + n) - 1
        request = urllib.request.Request(self.url, headers={"Range": f"bytes={self.pos}-{end}"})
        with urllib.request.urlopen(request, timeout=180) as response:  # noqa: S310 (pinned https URL)
            if response.status != 206:
                raise ValueError(f"{self.url}: server ignored the Range request (HTTP {response.status})")
            data = response.read()
        self.fetched += len(data)
        self.pos += len(data)
        return data

    def readinto(self, buffer: Any) -> int:
        data = self.read(len(buffer))
        buffer[: len(data)] = data
        return len(data)


def _hub_column(split: str, spec: Mapping[str, Any]) -> list[str]:
    """Read the pinned shard's `ground_truth` column from the Hub: the file's declared size and LFS SHA-256
    are checked against the pins first, then only the parquet footer and that column are fetched."""
    import pyarrow.parquet as pq
    from huggingface_hub import get_hf_file_metadata, hf_hub_url

    url = hf_hub_url(CORPUS_REPO, spec["path"], repo_type="dataset", revision=CORPUS_REVISION)
    metadata = get_hf_file_metadata(url)
    declared = (metadata.etag or "").strip('"')
    if metadata.size != spec["bytes"] or declared != spec["sha256"]:
        raise ValueError(
            f"{split} shard: the Hub declares {metadata.size} bytes / sha256 {declared[:16]}…, "
            f"pinned {spec['bytes']} / {spec['sha256'][:16]}…"
        )
    handle = _HttpRangeFile(url, spec["bytes"])
    table = pq.ParquetFile(handle).read(columns=[CORPUS_COLUMN])
    return [str(value) for value in table.column(CORPUS_COLUMN).to_pylist()]


def fetch_corpus(
    *,
    cache_dir: str | Path | None = None,
    fetcher: Callable[[str, Mapping[str, Any]], Sequence[str]] | None = None,
) -> dict[str, list[str]]:
    """Return the pinned shards' `ground_truth` rows per split from the cache or the Hub, digest-verified."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    out: dict[str, list[str]] = {}
    for split, spec in CORPUS_FILES.items():
        local = cache / f"{split}.{CORPUS_COLUMN}.json"
        rows: list[str] | None = None
        if local.is_file():
            cached = json.loads(local.read_text(encoding="utf-8"))
            if isinstance(cached, list) and column_digest(cached) == spec["column_sha256"]:
                rows = [str(value) for value in cached]
        if rows is None:
            rows = [
                str(value)
                for value in (fetcher(split, spec) if fetcher is not None else _hub_column(split, spec))
            ]
            if len(rows) != spec["rows"] or column_digest(rows) != spec["column_sha256"]:
                raise ValueError(
                    f"{split} shard: fetched {len(rows)} rows with column sha256 "
                    f"{column_digest(rows)[:16]}…, "
                    f"pinned {spec['rows']} / {spec['column_sha256'][:16]}…"
                )
            local.write_text(json.dumps(rows, ensure_ascii=False), encoding="utf-8")
        out[split] = rows
    return out


def _quad_to_box(quad: Mapping[str, Any], width: int, height: int) -> list[float] | None:
    xs = [float(quad[k]) for k in ("x1", "x2", "x3", "x4")]
    ys = [float(quad[k]) for k in ("y1", "y2", "y3", "y4")]
    x0, y0 = max(0.0, min(xs)), max(0.0, min(ys))
    x1, y1 = min(float(width), max(xs)), min(float(height), max(ys))
    if x1 <= x0 or y1 <= y0:
        return None
    return [x0, y0, x1, y1]


def page_from_ground_truth(ground_truth: str | Mapping[str, Any], page_id: str) -> dict[str, Any]:
    """One CORD `ground_truth` JSON → the page's words, boxes, size and the labelled lines over them.

    Words are concatenated line by line in the annotation's order (`valid_line`), every word box is the
    axis-aligned hull of its quadrilateral clipped to the page, and words with an empty text or a degenerate
    box are dropped. Each line records its field `category`, `group_id`, its inclusive word span and the span
    of its **value** words (`is_key` 0 — a printed key such as `TOTAL` is part of the line but not of the
    answer); `value_start` is `None` when the value words are not one contiguous run."""
    data = json.loads(ground_truth) if isinstance(ground_truth, str) else ground_truth
    size = data["meta"]["image_size"]
    width, height = int(size["width"]), int(size["height"])
    words: list[str] = []
    boxes: list[list[float]] = []
    lines: list[dict[str, Any]] = []
    for line in data.get("valid_line", []):
        start = len(words)
        values: list[int] = []
        for word in line.get("words", []):
            text = " ".join(str(word.get("text", "")).split())
            box = _quad_to_box(word["quad"], width, height) if text else None
            if box is None:
                continue
            if not int(word.get("is_key", 0)):
                values.append(len(words))
            words.append(text)
            boxes.append(box)
        if len(words) > start:
            contiguous = bool(values) and values == list(range(values[0], values[-1] + 1))
            lines.append(
                {
                    "category": str(line.get("category", "")),
                    "group_id": int(line.get("group_id", -1)),
                    "start": start,
                    "end": len(words) - 1,
                    "value_start": values[0] if contiguous else None,
                    "value_end": values[-1] if contiguous else None,
                }
            )
    return {"id": page_id, "words": words, "boxes": boxes, "image_size": [width, height], "lines": lines}


def read_corpus(rows: Mapping[str, Sequence[str]]) -> dict[str, list[dict[str, Any]]]:
    """The fetched shards as pages, ids `<split>-<row>`."""
    out: dict[str, list[dict[str, Any]]] = {}
    for split, values in rows.items():
        out[split] = [
            page_from_ground_truth(value, f"{split}-{index:03d}") for index, value in enumerate(values)
        ]
    return out


def questions_for_page(page: Mapping[str, Any]) -> list[dict[str, Any]]:
    """The templated questions a page supports: one per category on exactly one line, plus the first item;
    the gold answer is the line's contiguous run of value words (lines without one are skipped)."""
    by_category: dict[str, list[dict[str, Any]]] = {}
    for line in page["lines"]:
        by_category.setdefault(line["category"], []).append(line)
    out = []
    for category, question in QUESTION_TEMPLATES.items():
        candidates = by_category.get(category, [])
        if not candidates:
            continue
        if category == FIRST_ITEM_CATEGORY:
            topmost = min(candidates, key=lambda line: (page["boxes"][line["start"]][1], line["start"]))
            same_group = [line for line in candidates if line["group_id"] == topmost["group_id"]]
            if len(same_group) != 1:
                continue
            line = topmost
        elif len(candidates) == 1:
            line = candidates[0]
        else:
            continue
        if line.get("value_start") is None:
            continue
        start, end = int(line["value_start"]), int(line["value_end"])
        out.append(
            {
                "page_id": page["id"],
                "question": question,
                "field": category,
                "words": list(page["words"]),
                "boxes": [list(box) for box in page["boxes"]],
                "image_size": list(page["image_size"]),
                "answer_start": start,
                "answer_end": end,
                "answers": [" ".join(page["words"][start : end + 1])],
            }
        )
    return out


def build_sample_dataset(
    pages: Mapping[str, Sequence[Mapping[str, Any]]],
    *,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Pool every fetched page that supports at least one question, drop any page whose lower-cased word
    sequence repeats an earlier page (CORD-v2 holds a few identical receipts across its shards), shuffle the
    pages with `seed`, cut **by page** into `sizes` (train / validation / test receipts) and expand each page
    into its questions — so no page, and no question, is shared between splits."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    pool = [dict(page) for split in sorted(pages) for page in pages[split]]
    seen: set[tuple[str, ...]] = set()
    supported = []
    for page in pool:
        key = _page_key(page)
        if key in seen or not questions_for_page(page):
            continue
        seen.add(key)
        supported.append(page)
    needed = sum(sizes.values())
    if len(supported) < needed:
        raise ValueError(f"{len(supported)} pages support a question; the split sizes need {needed}")
    random.Random(seed).shuffle(supported)
    out: dict[str, list[dict[str, Any]]] = {}
    offset = 0
    for name in ("train", "validation", "test"):
        chosen = supported[offset : offset + sizes[name]]
        offset += sizes[name]
        records = [record for page in chosen for record in questions_for_page(page)]
        out[name] = [{"id": f"{name}-{index:04d}", **record} for index, record in enumerate(records)]
    return out


def fetch_sample_dataset(
    *,
    cache_dir: str | Path | None = None,
    fetcher: Callable[[str, Mapping[str, Any]], Sequence[str]] | None = None,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned shards."""
    return build_sample_dataset(
        read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher)), seed=seed, sizes=sizes
    )


def _check_record(record: Any, index: int) -> dict[str, Any]:
    label = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(
            f"{label} must be a mapping with id/question/words/boxes/image_size/answer_start/answer_end"
        )
    for key in ("id", "question", "words", "boxes", "image_size", "answer_start", "answer_end"):
        if key not in record:
            raise ValueError(f"{label} is missing {key!r}")
    rid = record["id"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{label}: id must match {_ID_RE.pattern}")
    try:
        words, _grid, size = _check_document(record["words"], record["boxes"], record["image_size"])
        question = _check_question(record["question"])
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{label}: {exc}") from exc
    start, end = record["answer_start"], record["answer_end"]
    if any(isinstance(v, bool) or not isinstance(v, int) for v in (start, end)):
        raise ValueError(f"{label}: answer_start and answer_end must be ints")
    if not 0 <= start <= end < len(words):
        raise ValueError(f"{label}: answer span {start}..{end} is not inside the {len(words)} words")
    text = " ".join(words[start : end + 1])
    answers = record.get("answers", [text])
    if isinstance(answers, str) or not isinstance(answers, Sequence) or not answers or answers[0] != text:
        raise ValueError(f"{label}: answers[0] must be the span text {text[:40]!r}")
    item = {
        "id": rid,
        "page_id": str(record.get("page_id", rid)),
        "question": question,
        "words": words,
        "boxes": [[float(v) for v in box] for box in record["boxes"]],
        "image_size": list(size),
        "answer_start": start,
        "answer_end": end,
        "answers": [str(a) for a in answers],
    }
    if "field" in record:
        item["field"] = str(record["field"])
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]], *, min_records: int = MIN_RECORDS, max_records: int = MAX_RECORDS
) -> dict[str, Any]:
    """Structural validation of a document-QA dataset; raises ValueError before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, (str, bytes)):
        raise ValueError(
            "records must be a list of {id, question, words, boxes, image_size, answer_start, answer_end}"
        )
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked = []
    ids: set[str] = set()
    pages: set[str] = set()
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        pages.add(item["page_id"])
        checked.append(item)
    return {
        "records": checked,
        "n_records": len(checked),
        "unique_pages": len(pages),
        "words_per_page": {
            "min": min(len(r["words"]) for r in checked),
            "max": max(len(r["words"]) for r in checked),
        },
        "answer_words": {
            "min": min(r["answer_end"] - r["answer_start"] + 1 for r in checked),
            "max": max(r["answer_end"] - r["answer_start"] + 1 for r in checked),
        },
        "question_chars": {
            "min": min(len(r["question"]) for r in checked),
            "max": max(len(r["question"]) for r in checked),
        },
        "max_question_chars": MAX_QUESTION_CHARS,
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [
        [r["id"], r["question"], r["words"], r["boxes"], r["image_size"], r["answer_start"], r["answer_end"]]
        for r in records
    ]
    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))


def gold_texts(record: Mapping[str, Any]) -> list[str]:
    """The accepted answers of a record (the gold span text)."""
    return [str(a) for a in record["answers"]]


def _page_key(record: Mapping[str, Any]) -> tuple[str, ...]:
    return tuple(str(w).lower() for w in record["words"])


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no page id and no page content (its lower-cased word sequence) appears in two splits."""
    seen_ids: dict[str, str] = {}
    seen_words: dict[tuple[str, ...], str] = {}
    for name, records in splits.items():
        for record in records:
            page_id = str(record.get("page_id", record["id"]))
            if page_id in seen_ids and seen_ids[page_id] != name:
                raise ValueError(f"page {page_id!r} appears in both {seen_ids[page_id]} and {name}")
            seen_ids[page_id] = name
            key = _page_key(record)
            if key in seen_words and seen_words[key] != name:
                raise ValueError(
                    f"a page's words ({' '.join(key[:6])!r}…) appear in both {seen_words[key]} and {name}"
                )
            seen_words[key] = name
    return {name: len(records) for name, records in splits.items()}


def split_minimums() -> dict[str, int]:
    """The fewest records each split of a BYOD dataset must hold for the notebook to run end to end."""
    return {"train": MIN_RECORDS, "validation": MIN_VAL_RECORDS, "test": MIN_TEST_RECORDS}


def _target_sizes(n: int, val_fraction: float, test_fraction: float) -> tuple[int, int]:
    return max(1, round(n * test_fraction)), round(n * val_fraction)


def byod_record_limits(
    val_fraction: float = BYOD_VAL_FRACTION, test_fraction: float = BYOD_TEST_FRACTION
) -> tuple[int, int]:
    """`(minimum, maximum)` record counts of a BYOD dataset with one question per page: the smallest
    dataset whose seeded split leaves every split at its `split_minimums()` (12 with the default
    fractions). Pages with several questions move whole, so such a dataset can need more records."""
    minimums = split_minimums()
    for n in range(1, MAX_RECORDS + 1):
        n_test, n_val = _target_sizes(n, val_fraction, test_fraction)
        n_val = min(n_val, n - n_test)
        if (
            n_test >= minimums["test"]
            and n_val >= minimums["validation"]
            and n - n_test - n_val >= minimums["train"]
        ):
            return n, MAX_RECORDS
    raise ValueError("no dataset size satisfies the split minimums with these fractions")


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = BYOD_VAL_FRACTION,
    test_fraction: float = BYOD_TEST_FRACTION,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded split of a BYOD dataset into train/validation/test **by page**: every question on the same page
    lands in the same split, so a test page is never seen in training. Each split must reach its
    `split_minimums()`; a refusal names the split, the counts and the dataset minimum."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records, min_records=1)["records"]
    groups: dict[str, list[dict[str, Any]]] = {}
    for record in checked:
        groups.setdefault(record["page_id"], []).append(record)
    order = list(groups.values())
    random.Random(seed).shuffle(order)
    n_test, n_val = _target_sizes(len(checked), val_fraction, test_fraction)
    splits: dict[str, list[dict[str, Any]]] = {"test": [], "validation": [], "train": []}
    for group in order:
        if len(splits["test"]) < n_test:
            splits["test"].extend(group)
        elif len(splits["validation"]) < n_val:
            splits["validation"].extend(group)
        else:
            splits["train"].extend(group)
    counts = "/".join(str(len(splits[name])) for name in ("train", "validation", "test"))
    for name, least in split_minimums().items():
        if len(splits[name]) < least:
            try:
                need = (
                    "a dataset needs at least "
                    f"{byod_record_limits(val_fraction, test_fraction)[0]} records with "
                    "one question per page (more when pages carry several questions, because a page never "
                    "straddles two splits). Add records"
                )
            except ValueError:
                need = "no dataset size gives every split its minimum with these fractions"
            raise ValueError(
                f"the {name} split would hold {len(splits[name])} records (at least {least} are required): "
                f"{len(checked)} records on {len(groups)} pages, split by page into train/validation/test as "
                f"{counts}; {need}"
            )
    return splits


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read records from a JSON array or a JSONL file of
    ``{id, page_id, question, words, boxes, image_size, answer_start, answer_end}`` objects. A parse
    error names the file and its line."""
    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"dataset not found: {file_path}")
    suffix = file_path.suffix.lower()
    if suffix not in (".json", ".jsonl"):
        raise ValueError("BYOD datasets must be .json or .jsonl")
    text = file_path.read_text(encoding="utf-8")
    if suffix == ".jsonl":
        out = []
        for number, line in enumerate(text.splitlines(), start=1):
            if not line.strip():
                continue
            try:
                out.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"{file_path.name}, line {number}: not valid JSON ({exc.msg} at column {exc.colno}); "
                    "each line of a .jsonl file must hold one complete record object"
                ) from None
        return out
    try:
        data = json.loads(text)
    except json.JSONDecodeError as exc:
        raise ValueError(
            f"{file_path.name}, line {exc.lineno}: not valid JSON ({exc.msg} at column {exc.colno})"
        ) from None
    if not isinstance(data, list):
        raise ValueError("JSON dataset must be an array of records")
    return data


def write_dataset_jsonl(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """One record per line in the shape `load_byod_dataset` reads back."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    keys = (
        "id",
        "page_id",
        "question",
        "words",
        "boxes",
        "image_size",
        "answer_start",
        "answer_end",
        "answers",
    )
    with open(out, "w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps({k: record[k] for k in keys if k in record}, ensure_ascii=False) + "\n")
    return out

## 3. Pin, stage and verify the model

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `8`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `beed3c4d02d8…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `LayoutLMDocumentQAPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
# @title Infrastructure: pin, stage and verify the model (you may run it without reading it)
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "layoutlm-document-qa",
  "modelId": "impira/layoutlm-document-qa",
  "revision": "beed3c4d02d86017ebca5bd0fdf210046b907aa6",
  "files": [
    {
      "path": "README.md",
      "bytes": 2326,
      "sha256": "40fd65fc734cc7eb73cc815465b8073bc4e5b7484406e537c7b8d763f88493f7"
    },
    {
      "path": "config.json",
      "bytes": 789,
      "sha256": "6d0fc068193109d0d053fa4de00963778beffbde05067c3e9f3454235044380f"
    },
    {
      "path": "merges.txt",
      "bytes": 456356,
      "sha256": "fe36cab26d4f4421ed725e10a2e9ddb7f799449c603a96e7f29b5a3c82a95862"
    },
    {
      "path": "model.safetensors",
      "bytes": 511200628,
      "sha256": "e4bbad3e4a1b5ae50c787b7afd6049a0bfa99fd823b50436e444e092ae2347b9"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 239,
      "sha256": "378eb3bf733eb16e65792d7e3fda5b8a4631387ca04d2015199c4d4f22ae554d"
    },
    {
      "path": "tokenizer.json",
      "bytes": 1355881,
      "sha256": "33465117406b9007673e8ba283f7f1383d9b5094df947481af60eec94ed7d7bd"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 315,
      "sha256": "ea11996be5d083d63c72700810b18a6cfdf78c55131b754a75ae94e66b0ad6ab"
    },
    {
      "path": "vocab.json",
      "bytes": 798293,
      "sha256": "ed19656ea1707df69134c4af35c8ceda2cc9860bf2c3495026153a133670ab5e"
    }
  ],
  "totalBytes": 513814827
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = LayoutLMDocumentQAPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

{'model_id': 'impira/layoutlm-document-qa', 'revision': 'beed3c4d02d86017ebca5bd0fdf210046b907aa6', 'license': 'mit', 'files': 8, 'total_bytes': 513814827}


{'weights_dir': '/content/weights/layoutlm-document-qa', 'fetched': ['README.md', 'config.json', 'merges.txt', 'model.safetensors', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']}


{'verified_files': 8, 'revision': 'beed3c4d02d86017ebca5bd0fdf210046b907aa6'}


{'device': 'cuda:0', 'source': 'local-snapshot'}


## 4. CORD-v2 receipt corpus, validation and split

`fetch_corpus` reads the pinned shards' `ground_truth` column (or the cache under `weights/cord-v2/`): for each shard it first checks the byte size and SHA-256 the Hub declares for the file against the pins, then reads only the parquet footer and the one column chunk through `pyarrow` over HTTPS range requests, and refuses the decoded column unless its SHA-256 matches. `read_corpus` turns each row into a page — the words of every annotated line in order, each box the clipped axis-aligned hull of the annotated quadrilateral, and the lines' field categories with their value-word spans (`is_key` 0; a printed key such as `TOTAL` is on the line but not in the answer). `build_sample_dataset` keeps every receipt that supports at least one templated question, drops the one receipt whose words repeat another shard's, shuffles the 199 unique receipts with `SPLIT_SEED` and cuts them **by receipt** into 119 / 30 / 50 training, validation and test pages, then expands each page into its questions (a category that occurs on exactly one line becomes one question; *first item* is the topmost single-line `menu.nm`). `validate_dataset` checks every record against the contract (each split against its own minimum from `split_minimums()`), `check_split_disjoint` asserts no page id and no page content is shared, and the training split is written to `outputs/layoutlm_document_qa_train.jsonl` in the shape BYOD expects.

**What the split does and does not protect against.** No receipt is in two splits, so the test score is never computed on a page the model trained on. But CORD-v2 records no shop or vendor, and many receipts come from the same few point-of-sale layouts, so receipts of one shop can sit in training and in test. The held-out scores in Section 8 are therefore an estimate for *more receipts like these*, not for another country's or another business's documents.

**Bring your own data:** set `USE_BYOD = True`, either leave `BYOD_PATH` empty to get an upload dialog (Colab only) or set it to a `.json` / `.jsonl` file already in this runtime, then select this cell and choose **Runtime → Run after**. An empty or cancelled upload, a runtime without the Colab upload dialog and no `BYOD_PATH`, or a path that does not exist stops with a message saying what to supply. `load_byod_dataset` names the line of a malformed JSONL record, `validate_dataset` names the record and the rule it breaks, and `split_dataset` refuses a dataset that leaves fewer than 8 training, 2 validation or 2 test records, naming the split (at least 12 records with one question per page). Under BYOD the printed provenance and `result.json` describe your file, not the CORD-v2 corpus.

Look for: 100 + 100 raw rows, three digests, splits 595 / 152 / 229 questions over 119 / 30 / 50 receipts, the BYOD minimum the code computes, the field mix of the test split, and four refusal probes — a duplicate id, a gold span outside the words, a box outside the page and a dataset too small to use — each rejected before `torch` does anything.

**Predict before running:** the test split has 229 questions over ten field types. Which field do you expect to have the fewest test questions, and how many questions would you want before trusting a per-field score?

In [ ]:
import collections
import hashlib
import json

USE_BYOD = False  # @param {type:"boolean"}
BYOD_PATH = ''  # @param {type:"string"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
byod = None
if USE_BYOD:
    if BYOD_PATH.strip():
        byod_path = Path(BYOD_PATH.strip()).expanduser()
        if not byod_path.is_file():
            raise FileNotFoundError(f'BYOD_PATH = {BYOD_PATH!r} is not a file in this runtime: fix the path, or leave BYOD_PATH empty to upload a .json or .jsonl file in Colab.')
        file_name = byod_path.name
    else:
        try:
            from google.colab import files
        except ImportError:
            raise RuntimeError('The upload dialog needs Google Colab. Elsewhere, put the .json or .jsonl file in this runtime and set BYOD_PATH to its path.') from None
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError(f'Upload exactly one .json or .jsonl file (received {len(uploaded)}). Run this cell again and choose a file, or set BYOD_PATH to a file already in this runtime.')
        file_name, payload = next(iter(uploaded.items()))
        byod_path = Path('work') / Path(file_name).name
        byod_path.parent.mkdir(parents=True, exist_ok=True)
        byod_path.write_bytes(payload)
    records = load_byod_dataset(byod_path)
    print({'byod_records': len(records), 'needed': f'at least {byod_record_limits()[0]} records with one question per page', 'split_minimums': split_minimums()})
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    raw_rows = {'byod': len(records)}
    byod = {'file': file_name, 'records': len(records), 'sha256': hashlib.sha256(byod_path.read_bytes()).hexdigest()}
else:
    corpus_rows = fetch_corpus(cache_dir='weights/cord-v2')
    raw_rows = {name: len(rows) for name, rows in corpus_rows.items()}
    splits = build_sample_dataset(read_corpus(corpus_rows), seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME} {CORPUS_RELEASE} ({CORPUS_LICENSE})'
minimums = split_minimums()
dataset_manifests = {name: validate_dataset(part, min_records=minimums[name]) for name, part in splits.items()}
disjoint = check_split_disjoint(splits)
pages = {name: len({r['page_id'] for r in part}) for name, part in splits.items()}
fields = {name: dict(collections.Counter(r.get('field', 'byod') for r in part)) for name, part in splits.items()}
write_dataset_jsonl(splits['train'], 'outputs/layoutlm_document_qa_train.jsonl')
print({'data_source': data_source, 'raw_rows': raw_rows, 'splits': disjoint, 'pages': pages, 'split_unit': 'page (receipt); no shop or vendor field', 'column_sha256': None if USE_BYOD else {name: spec['column_sha256'][:16] + '...' for name, spec in CORPUS_FILES.items()}})
print({'byod_minimum_records': byod_record_limits()[0], 'split_minimums': minimums})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'unique_pages': manifest['unique_pages'], 'words_per_page': manifest['words_per_page'], 'answer_words': manifest['answer_words'], 'digest': manifest['digest'][:16] + '...'}})
print({'fields_test': fields['test']})
example = splits['train'][0]
print({'example': {'id': example['id'], 'page_id': example['page_id'], 'question': example['question'], 'answers': example['answers'], 'span': [example['answer_start'], example['answer_end']], 'words': ' '.join(example['words'][:18]) + ' ...'}})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in splits['train'][:8]],
    'gold span outside the words': [{**splits['train'][0], 'answer_end': len(splits['train'][0]['words'])}, *splits['train'][1:8]],
    'box outside the page': [{**splits['train'][0], 'boxes': [[0, 0, splits['train'][0]['image_size'][0] + 1, 5], *splits['train'][0]['boxes'][1:]]}, *splits['train'][1:8]],
    'too small': splits['train'][:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

{'data_source': 'CORD-v2 Hugging Face Hub dataset revision 7f0115a4 (2022-07-19) (CC BY 4.0 (Park et al. 2019; NAVER Clova; huggingface.co/datasets/naver-clova-ix/cord-v2))', 'raw_rows': {'test': 100, 'validation': 100}, 'splits': {'train': 595, 'validation': 152, 'test': 229}, 'pages': {'train': 119, 'validation': 30, 'test': 50}, 'split_unit': 'page (receipt); no shop or vendor field', 'column_sha256': {'test': 'b499e58aa298e524...', 'validation': 'adf8303ec0295af1...'}}
{'byod_minimum_records': 12, 'split_minimums': {'train': 8, 'validation': 2, 'test': 2}}
{'train': {'n': 595, 'unique_pages': 119, 'words_per_page': {'min': 9, 'max': 60}, 'answer_words': {'min': 1, 'max': 7}, 'digest': 'd063f10092c47e39...'}}
{'validation': {'n': 152, 'unique_pages': 30, 'words_per_page': {'min': 9, 'max': 70}, 'answer_words': {'min': 1, 'max': 5}, 'digest': '07e37d514b1e81ea...'}}
{'test': {'n': 229, 'unique_pages': 50, 'words_per_page': {'min': 7, 'max': 66}, 'answer_words': {'min': 1, 'max': 9}, 

**What to notice:** the three split sizes and page counts, `split_unit`, the BYOD minimum (12), the field mix of the test split, and four rejected probes.

<details><summary>Check your reasoning</summary>The service-charge field: in the recorded Kaggle Tesla T4 run the 229 test questions were 50 first item, 46 total, 35 cash, 29 change, 27 subtotal, 17 tax, 12 item count, 7 credit card, 4 discount and **2 service charge**. A score on 2 or 4 questions moves by 0.25 or 0.5 when a single answer changes, so it cannot be read as a property of the model; treat any field with fewer than about 10 questions as anecdote (Section 8 marks them). The BYOD minimum the code prints, 12, is the smallest one-question-per-page dataset whose seeded 65 / 15 / 20 % split leaves 8 training, 2 validation and 2 test records.</details>

## 5. Fit check, then answer through the inference contract

The window ceiling needs the real tokenizer, so it is applied now that the model is loaded: `pipe.check_fit` partitions each split into the records whose question and page fit one 512-token window and the ones that would be **windowed at inference and are therefore not trained on** — nothing is truncated. The dropped ids are printed and the fitting records are what every later cell uses (the recorded run dropped none of the 976 sample records; receipts are short).

Then the inference contract is exercised as the inference-only tutorial exercised it: an invoice-style form is rendered in code with Pillow's bundled font and the renderer records the pixel box of **every word it draws** — perfect OCR by construction, a different document family from the receipts, and a page the model will be asked to read again after adaptation. `validate_inputs` applies exactly the checks `answer` applies (page size, 1..`MAX_WORDS` non-empty words with one box each inside the page, questions up to `MAX_QUESTION_CHARS`) and returns an input manifest; a box outside the page is validated too and its rejection recorded as a finding. `answer` returns the word span, its inclusive `start`/`end` indices (so the answer's boxes are known), `score`, `n_words`, `n_windows` and the model identity. **Score semantics:** the `score` is the **product of the start and end softmax probabilities of the chosen span within its window** — a ranking signal over spans of this page, **not a calibrated probability** that the answer is right, and never a signal that the question is answerable (Section 10 shows this). Whether the answers are *right* is what Section 6 measures on 229 gold spans, not what five authored pairs can tell you.

**Every pass starts from the pretrained model.** Section 7 changes the model in memory. If this cell, Section 6 or Section 7 runs again after Section 7 (a BYOD run or one of the optional experiments), it first reloads the pretrained model from the verified snapshot with `reset_to_pretrained()`, so a *frozen* number is never read from the adapted model and a new adapter is never trained on top of the old one.

**Predict before running:** will the pretrained model answer all five invoice questions exactly? Roughly what `score` do you expect on a correct answer — near 0.5, or near 1.0?

In [ ]:
import gc
import time

import numpy as np
from PIL import Image, ImageDraw, ImageFont


def reset_to_pretrained():
    """Sections 5-7 start from the pretrained model: if an earlier pass adapted `pipe`, reload it from the verified snapshot."""
    global pipe
    if pipe.adapter is None:
        return
    pipe = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    pipe = LayoutLMDocumentQAPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
    print({'reloaded': 'the pretrained model, from the verified snapshot', 'reason': 'an earlier pass had adapted the model in memory'})


reset_to_pretrained()
fit = {name: pipe.check_fit(part) for name, part in splits.items()}
train_records, val_records, test_records = fit['train']['fitting'], fit['validation']['fitting'], fit['test']['fitting']
print({'fit_check': {name: {'fitting': f['n_fitting'], 'dropped': f['dropped']} for name, f in fit.items()}})
ceilings = {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_WORDS': MAX_WORDS, 'MAX_QUESTION_CHARS': MAX_QUESTION_CHARS, 'MAX_SEQ_LEN': MAX_SEQ_LEN, 'DOC_STRIDE': DOC_STRIDE, 'MAX_ANSWER_TOKENS': MAX_ANSWER_TOKENS, 'BOX_GRID': BOX_GRID, 'MIN_RECORDS': MIN_RECORDS, 'MAX_RECORDS': MAX_RECORDS}
print(ceilings)


def synthetic_form(width=850, height=1100):
    """Invoice-style form rendered with Pillow's bundled font; every drawn word is recorded with its pixel box."""
    page = Image.new('RGB', (width, height), 'white')
    d = ImageDraw.Draw(page)
    body, bold, head = ImageFont.load_default(size=18), ImageFont.load_default(size=20), ImageFont.load_default(size=30)
    words, boxes = [], []

    def put(x, y, text, font, fill='black'):
        for token in text.split():
            x0, y0, x1, y1 = d.textbbox((x, y), token, font=font)
            d.text((x, y), token, fill=fill, font=font)
            words.append(token)
            boxes.append([float(x0), float(y0), float(x1), float(y1)])
            x = x1 + d.textlength(' ', font=font)

    put(70, 60, 'INVOICE', head)
    put(70, 110, 'Northwind Traders Ltd.', bold)
    put(70, 136, '14 Harbour Road, Portsmouth PO1 3AX', body, (40, 40, 40))
    y = 200
    for label, value in [('Invoice number:', 'NW-2026-0417'), ('Invoice date:', '12 March 2026'), ('Due date:', '11 April 2026'), ('Customer:', 'Blue Yonder Airlines'), ('Purchase order:', 'PO-88213')]:
        put(70, y, label, bold)
        put(300, y, value, body)
        y += 32
    d.rectangle([70, 400, 780, 640], outline='black', width=2)
    cols = [70, 420, 540, 660, 780]
    rows = [('Description', 'Qty', 'Unit price', 'Amount'), ('Cargo pallets (standard)', '40', '$18.50', '$740.00'), ('Shrink wrap rolls', '12', '$9.25', '$111.00'), ('Handling fee', '1', '$65.00', '$65.00')]
    for r, row in enumerate(rows):
        yy = 400 + r * 48
        if r:
            d.line([(70, yy), (780, yy)], fill=(120, 120, 120), width=1)
        for c, cell in enumerate(row):
            put(cols[c] + 10, yy + 14, cell, bold if r == 0 else body)
    for c in cols[1:-1]:
        d.line([(c, 400), (c, 640)], fill=(120, 120, 120), width=1)
    put(540, 670, 'Subtotal:', bold)
    put(680, 670, '$916.00', body)
    put(540, 700, 'VAT (20%):', bold)
    put(680, 700, '$183.20', body)
    put(540, 736, 'Total due:', head)
    put(680, 736, '$1,099.20', head)
    put(70, 900, 'Payment terms: 30 days from invoice date. Bank: Solent Mutual, sort code 40-11-22.', body, (40, 40, 40))
    qa = [
        ('What is the invoice number?', ['NW-2026-0417']),
        ('Who is the customer?', ['Blue Yonder Airlines']),
        ('What is the total due?', ['$1,099.20', '1,099.20']),
        ('What is the due date?', ['11 April 2026']),
        ('How many cargo pallets were invoiced?', ['40']),
    ]
    return page, words, boxes, qa


image, words, boxes, qa = synthetic_form()
questions, golds = [q for q, _ in qa], [g for _, g in qa]
image_name = 'synthetic_invoice_850x1100.png'
image_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
input_manifest = validate_inputs(words, boxes, questions, image_size=image.size, names=[image_name])
try:
    validate_inputs(words[:1], [[0, 0, image.width + 1, 10]], questions, image_size=image.size)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'box-outside-page-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/layoutlm_document_qa_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print({'page': image_name, 'size': image.size, 'rgb_sha256': image_sha256[:16] + '...', 'n_words': len(words), 'manifest_verdict': input_manifest['verdict'], 'findings': len(input_manifest['findings'])})
results = []
for question in questions:
    started = time.perf_counter()
    result = pipe.answer(question, words=words, boxes=boxes, image_size=image.size)
    results.append({'seconds': round(time.perf_counter() - started, 3), **result})
    print(f"Q: {result['question']}\n   A: {result['answer']!r}  score {result['score']:.4f}  words {result['start']}..{result['end']}  windows {result['n_windows']}")
checks = {
    'one_result_per_question': len(results) == len(questions),
    'span_is_the_indexed_words': all(r['answer'] == ' '.join(words[r['start'] : r['end'] + 1]) for r in results if r['start'] is not None),
    'indices_inside_the_page': all(0 <= r['start'] <= r['end'] < r['n_words'] for r in results if r['start'] is not None),
    'scores_in_unit_interval': all(0.0 <= r['score'] <= 1.0 for r in results),
    'one_window': all(r['n_windows'] == 1 for r in results),
}
if not all(checks.values()):
    raise RuntimeError(f'answer output failed a sanity check: {checks}')
frozen_form = evaluation_report(results, golds, sample_kind='synthetic')
print({'checks': checks, 'frozen_invoice': {m['id']: round(m['value'], 3) for m in frozen_form['metrics']}, 'verdict': frozen_form['verdict']})

{'fit_check': {'train': {'fitting': 595, 'dropped': []}, 'validation': {'fitting': 152, 'dropped': []}, 'test': {'fitting': 229, 'dropped': []}}}
{'MIN_IMAGE_SIDE': 1, 'MAX_IMAGE_SIDE': 10000, 'MAX_WORDS': 2000, 'MAX_QUESTION_CHARS': 256, 'MAX_SEQ_LEN': 512, 'DOC_STRIDE': 128, 'MAX_ANSWER_TOKENS': 15, 'BOX_GRID': 1000, 'MIN_RECORDS': 8, 'MAX_RECORDS': 20000}
{'page': 'synthetic_invoice_850x1100.png', 'size': (850, 1100), 'rgb_sha256': 'ee1796645be8a8f5...', 'n_words': 73, 'manifest_verdict': 'accepted', 'findings': 1}


Q: What is the invoice number?
   A: 'NW-2026-0417'  score 1.0000  words 12..12  windows 1
Q: Who is the customer?
   A: 'Blue Yonder Airlines'  score 1.0000  words 24..26  windows 1
Q: What is the total due?
   A: '$1,099.20'  score 0.9998  words 59..59  windows 1
Q: What is the due date?
   A: '11 April 2026'  score 1.0000  words 20..22  windows 1


Q: How many cargo pallets were invoiced?
   A: '40'  score 0.9994  words 38..38  windows 1
{'checks': {'one_result_per_question': True, 'span_is_the_indexed_words': True, 'indices_inside_the_page': True, 'scores_in_unit_interval': True, 'one_window': True}, 'frozen_invoice': {'anls': 1.0, 'exact_match': 1.0}, 'verdict': 'sample-sanity'}


**What to notice:** the fit-check line (nothing dropped), the five answers with their word indices and scores, the five `checks`, and the invoice ANLS with its `sample-sanity` verdict.

<details><summary>Check your reasoning</summary>Yes, all five, with scores near 1.0. In the recorded T4 run the pretrained model answered `NW-2026-0417`, `Blue Yonder Airlines`, `$1,099.20`, `11 April 2026` and `40` with scores 1.0000, 1.0000, 0.9998, 1.0000 and 0.9994 (invoice ANLS 1.0). A clean, perfectly OCR'd form with labelled fields is the easy case the model was trained for. The scores say the model put almost all its start and end mass on one span of *this* page; they would be just as high for a wrong span it was sure of.</details>

## 6. Baselines and the frozen model's score on the test receipts

Three numbers frame the adaptation. The **last-number baseline** answers every question with the last word on the page that contains a digit — receipts end with their totals, so it is right more often than chance and wrong for everything else. The **keyword-lookup baseline** finds the page word sharing the longest prefix with a content word of the question (`TOTAL` for *What is the total amount?*) and answers with the next numeric word after it — the find-the-label-then-read-the-value heuristic a person applies, with no model. The **frozen model** answers the 229 test questions and is scored with the same two metrics: mean **ANLS** (DocVQA's normalised Levenshtein similarity, 0 below 0.5) and the **exact-match** rate, both after lower-casing, punctuation removal and whitespace collapsing. The per-field breakdown prints `n` beside each field and marks the fields with fewer than 10 questions. The cell records whether the frozen model beats each baseline; it does not stop when it does not, because on your own data that can be the honest result.

**Predict before running:** will the frozen model land closer to the keyword-lookup baseline or well above it? Which field do you expect it to read worst — a total, an item name, or the number of items bought?

In [ ]:
reset_to_pretrained()
baseline_last = last_number_baseline(test_records)
baseline_lookup = keyword_lookup_baseline(test_records)
print({'last_number_baseline': {'anls': round(baseline_last['anls'], 3), 'exact_match': round(baseline_last['exact_match'], 3), 'n': baseline_last['n']}})
print({'keyword_lookup_baseline': {'anls': round(baseline_lookup['anls'], 3), 'exact_match': round(baseline_lookup['exact_match'], 3), 'n': baseline_lookup['n']}})
t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records)
if frozen_test['adapted']:
    raise RuntimeError('Section 6 must score the pretrained model, but the model in memory is adapted. Restart the session and choose Run all.')
print({'frozen_model_test': {'anls': round(frozen_test['anls'], 3), 'exact_match': round(frozen_test['exact_match'], 3), 'n': frozen_test['n'], 'adapted': frozen_test['adapted'], 'verdict': frozen_test['verdict']}, 'seconds': round(time.perf_counter() - t0, 1)})
print({'definitions': frozen_test['definitions']})
SMALL_FIELD = 10  # a per-field score on fewer questions than this is anecdote, not evidence


def by_field(pipeline, records):
    """Per-field ANLS with its question count, plus every answer (so Section 8 can show where the models differ)."""
    scores, answers = collections.defaultdict(list), {}
    for record in records:
        item = pipeline.answer(record['question'], words=record['words'], boxes=record['boxes'], image_size=record['image_size'])
        answers[record['id']] = item['answer']
        scores[record.get('field', 'byod')].append(anls(item['answer'], gold_texts(record)))
    table = {field: {'n': len(values), 'anls': round(sum(values) / len(values), 2), 'too_few_to_read': len(values) < SMALL_FIELD} for field, values in sorted(scores.items())}
    return table, answers


frozen_fields, frozen_answers = by_field(pipe, test_records)
print({'frozen_by_field': frozen_fields})
frozen_beats_last_number = frozen_test['anls'] > baseline_last['anls']
frozen_beats_keyword_lookup = frozen_test['anls'] > baseline_lookup['anls']
print({'frozen_beats_last_number': frozen_beats_last_number, 'frozen_beats_keyword_lookup': frozen_beats_keyword_lookup})

{'last_number_baseline': {'anls': 0.406, 'exact_match': 0.253, 'n': 229}}
{'keyword_lookup_baseline': {'anls': 0.635, 'exact_match': 0.594, 'n': 229}}


{'frozen_model_test': {'anls': 0.843, 'exact_match': 0.782, 'n': 229, 'adapted': False, 'verdict': 'measured'}, 'seconds': 7.4}
{'definitions': {'anls': 'mean over questions of the Average Normalised Levenshtein Similarity between the prediction and its best-matching accepted answer (lower-cased, punctuation removed, whitespace collapsed; a similarity below 0.5 scores 0); in 0..1', 'exact_match': 'fraction of questions whose normalised prediction equals a normalised accepted answer; in 0..1'}}


{'frozen_by_field': {'menu.nm': {'n': 50, 'anls': 0.79, 'too_few_to_read': False}, 'sub_total.discount_price': {'n': 4, 'anls': 1.0, 'too_few_to_read': True}, 'sub_total.service_price': {'n': 2, 'anls': 1.0, 'too_few_to_read': True}, 'sub_total.subtotal_price': {'n': 27, 'anls': 0.98, 'too_few_to_read': False}, 'sub_total.tax_price': {'n': 17, 'anls': 0.92, 'too_few_to_read': False}, 'total.cashprice': {'n': 35, 'anls': 0.85, 'too_few_to_read': False}, 'total.changeprice': {'n': 29, 'anls': 0.87, 'too_few_to_read': False}, 'total.creditcardprice': {'n': 7, 'anls': 0.96, 'too_few_to_read': True}, 'total.menuqty_cnt': {'n': 12, 'anls': 0.17, 'too_few_to_read': False}, 'total.total_price': {'n': 46, 'anls': 0.91, 'too_few_to_read': False}}}
{'frozen_beats_last_number': True, 'frozen_beats_keyword_lookup': True}


**What to notice:** `adapted: False`, the frozen ANLS beside the two baselines, and the per-field rows with their `n` and `too_few_to_read` flags.

<details><summary>Check your reasoning</summary>Well above, and the item count. In the recorded T4 run the last-number baseline scored ANLS 0.406 (exact match 0.253), keyword lookup 0.635 (0.594) and the frozen model 0.843 (0.782). Per field it read subtotals at 0.98 and totals at 0.91 but item names at 0.79 and *How many items were bought?* at only **0.17** on 12 questions: the count is a small number printed somewhere on the receipt with no label the question names, so the DocVQA-trained reader picks a different number. Those weak fields are where Section 7 has room to move.</details>

## 7. Bounded fine-tuning of the last encoder blocks and the span head

`pipe.adapt` trains only the last `TRAINABLE_ENCODER_LAYERS` encoder blocks plus the span head `qa_outputs` — four blocks by default, 28,353,026 of 127,792,898 parameters; the word, position and 2-D box embeddings and the earlier blocks stay frozen — with start/end cross-entropy on the first and last token of the gold word span, AdamW at a fixed learning rate, gradient clipping at 1.0, dynamic padding, seeded shuffling and no scheduler. Nothing is truncated: every training record passed the fit check. Epoch 0 records the frozen model's validation ANLS; every epoch is scored on the 152 validation questions, and the epoch with the highest validation ANLS is kept. The test receipts play no part in this choice. `adapt` refuses a pipeline that is already adapted, and the cell first reloads the pretrained model if an earlier pass adapted it.

**Expected result.** Validation ANLS rises from about **0.82** to about **0.95**: in the recorded Kaggle Tesla T4 run of the previous notebook version (same model stages; `torch 2.14.0+cu130`, `transformers 4.57.6`, `cuda:0`, Python 3.12.13) it went **0.819 → 0.952** and epoch 5 was kept, in 82 s for six epochs. **Run-to-run spread.** Training is deterministic for one device and library set but not bit-reproducible across devices: a CPU pre-flight of the same configuration (`torch 2.14.0`, float32) reached 0.960 at epoch 5 and kept it too, with test ANLS 0.942 against the T4 run's 0.940, and took about two and a half minutes per epoch. A result within about 0.01 of the quoted one is that spread, not a fault.

**Predict before running:** will the validation ANLS rise in every epoch? Which epoch do you expect the selector to keep — always the last one?

In [ ]:
EPOCHS = 6  # @param {type:"integer"}
LEARNING_RATE = 3e-5  # @param {type:"number"}
BATCH_SIZE = 16  # @param {type:"integer"}
TRAINABLE_ENCODER_LAYERS = 4  # @param {type:"integer"}


def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row['val_anls'] = round(entry['val']['anls'], 4)
        row['val_exact_match'] = round(entry['val']['exact_match'], 4)
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)


reset_to_pretrained()
t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable_encoder_layers=TRAINABLE_ENCODER_LAYERS, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_encoder_layers': adapt_result['trainable_encoder_layers'], 'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'learning_rate': adapt_result['lr'], 'seed': adapt_result['seed'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'selection_split': 'validation', 'seconds': adapt_seconds})

{'epoch': 0, 'train_loss': None, 'val_anls': 0.8191, 'val_exact_match': 0.7368, 'note': 'frozen model'}


{'epoch': 1, 'train_loss': 1.3726, 'val_anls': 0.8594, 'val_exact_match': 0.8158}


{'epoch': 2, 'train_loss': 0.7737, 'val_anls': 0.9023, 'val_exact_match': 0.8618}


{'epoch': 3, 'train_loss': 0.591, 'val_anls': 0.9237, 'val_exact_match': 0.9013}


{'epoch': 4, 'train_loss': 0.5003, 'val_anls': 0.9337, 'val_exact_match': 0.9276}


{'epoch': 5, 'train_loss': 0.4376, 'val_anls': 0.9523, 'val_exact_match': 0.9408}


{'epoch': 6, 'train_loss': 0.3775, 'val_anls': 0.9518, 'val_exact_match': 0.9474}
{'trainable_encoder_layers': 4, 'trainable_parameters': 28353026, 'total_parameters': 127792898, 'learning_rate': 3e-05, 'seed': 0, 'best_epoch': 5, 'selection': 'highest validation ANLS', 'selection_split': 'validation', 'seconds': 78.9}


**What to notice:** epoch 0 is labelled the frozen model, how the training loss and the validation ANLS move from epoch to epoch, and which epoch is kept.

<details><summary>Check your reasoning</summary>Almost, and no. In the recorded T4 run the validation ANLS went 0.819 (epoch 0, the frozen model) → 0.859 → 0.902 → 0.924 → 0.934 → **0.952** → 0.952 while the training loss fell from 1.37 to 0.38; epoch 6 (0.9518) was a hair below epoch 5 (0.9523), so the selector kept **epoch 5**. On the CPU pre-flight the curve dipped once (0.908 → 0.894 at epoch 3) and epoch 5 was kept again. The selector keeps the best validation epoch, not the last one, and never looks at the test receipts.</details>

## 8. Held-out evaluation

The test receipts were never used for training or epoch selection, and no test page — by id or by word content — appears in the training or validation splits. The adapted model is scored exactly as the frozen model was in Section 6, the four systems are put side by side on both metrics, and the per-field breakdown is repeated with `n` and the `too_few_to_read` flag. **Read it field by field, not only overall:** a gain can be concentrated in the fields the frozen model missed, and a field can also **lose** score after adaptation — in the recorded run one did. A field with only 2 or 4 questions moves by 0.25 or 0.5 when one answer changes, so its row is anecdote. The cell then prints the test questions where the frozen and the adapted model **disagree** (up to six), records whether the adapted model beats the frozen one and both baselines, prints a one-line reading of the outcome and keeps one row per pass in a run history; a result where adaptation did not help is reported, and the export below still runs. Two hundred and twenty-nine questions over 50 receipts from one seeded split of one corpus give **no dispersion estimate**; the deltas are sample-sanity evidence that the adaptation contract works, not a benchmark, and a gain on CORD receipts with their annotated OCR says nothing about your documents or your OCR until you measure it there.

**Predict before running:** will every field improve after fine-tuning? Which field do you expect to gain most, and could one get worse?

In [ ]:
adapted_test = pipe.evaluate(test_records)
adapted_val = pipe.evaluate(val_records)
adapted_fields, adapted_answers = by_field(pipe, test_records)
comparison = {
    'anls': {'last_number': round(baseline_last['anls'], 3), 'keyword_lookup': round(baseline_lookup['anls'], 3), 'frozen': round(frozen_test['anls'], 3), 'adapted': round(adapted_test['anls'], 3)},
    'exact_match': {'last_number': round(baseline_last['exact_match'], 3), 'keyword_lookup': round(baseline_lookup['exact_match'], 3), 'frozen': round(frozen_test['exact_match'], 3), 'adapted': round(adapted_test['exact_match'], 3)},
    'delta_vs_frozen': {'anls': round(adapted_test['anls'] - frozen_test['anls'], 3), 'exact_match': round(adapted_test['exact_match'] - frozen_test['exact_match'], 3)},
    'by_field': {field: {'n': frozen_fields[field]['n'], 'frozen': frozen_fields[field]['anls'], 'adapted': adapted_fields[field]['anls'], 'too_few_to_read': frozen_fields[field]['too_few_to_read']} for field in frozen_fields},
}
for metric, row in comparison.items():
    print({metric: row})
worse_fields = sorted(field for field, row in comparison['by_field'].items() if row['adapted'] < row['frozen'])
print({'fields_that_lost_score': {field: comparison['by_field'][field] for field in worse_fields}})
disagreements = [r for r in test_records if frozen_answers[r['id']] != adapted_answers[r['id']]]
print({'questions_where_the_models_disagree': len(disagreements), 'of': len(test_records)})
for record in disagreements[:6]:
    print({'field': record.get('field', 'byod'), 'question': record['question'], 'frozen': frozen_answers[record['id']], 'adapted': adapted_answers[record['id']], 'gold': gold_texts(record)})

# Differences below 0.001 (the printed precision) count as no change.
adapted_beats_frozen = comparison['delta_vs_frozen']['anls'] > 0
adapted_beats_both_baselines = round(adapted_test['anls'] - max(baseline_last['anls'], baseline_lookup['anls']), 3) > 0
if adapt_result['best_epoch'] == 0:
    reading = 'The selector kept epoch 0, so the adapted model is the frozen model: no epoch beat it on validation.'
elif not adapted_beats_frozen:
    reading = f"The adapted ANLS is not above the frozen ANLS on the test receipts (delta {comparison['delta_vs_frozen']['anls']:+.3f}): this adaptation did not help here. The export and reload below still run."
elif worse_fields:
    reading = f"The adapted ANLS is above the frozen ANLS (delta {comparison['delta_vs_frozen']['anls']:+.3f}), but not every field gained: {', '.join(worse_fields)} lost score (check its n before reading anything into it)."
else:
    reading = f"The adapted ANLS is above the frozen ANLS (delta {comparison['delta_vs_frozen']['anls']:+.3f}) and no field lost score."
outcomes = {'frozen_beats_last_number': frozen_beats_last_number, 'frozen_beats_keyword_lookup': frozen_beats_keyword_lookup, 'adapted_beats_frozen': adapted_beats_frozen, 'adapted_beats_both_baselines': adapted_beats_both_baselines, 'fields_that_lost_score': worse_fields, 'best_epoch': adapt_result['best_epoch']}
print(outcomes)
print('Reading: ' + reading)
run_history = globals().get('run_history', [])
run_history.append({'run': len(run_history) + 1, 'data': 'BYOD' if USE_BYOD else 'sample', 'blocks': TRAINABLE_ENCODER_LAYERS, 'epochs': EPOCHS, 'learning_rate': LEARNING_RATE, 'best_epoch': adapt_result['best_epoch'], 'frozen_anls': round(frozen_test['anls'], 3), 'adapted_anls': round(adapted_test['anls'], 3), 'adapted_exact_match': round(adapted_test['exact_match'], 3)})
print(f"{'run':>3} {'data':>6} {'blocks':>6} {'epochs':>6} {'lr':>8} {'best':>4} {'frozen':>7} {'ANLS':>7} {'EM':>7}")
for h in run_history:
    print(f"{h['run']:>3} {h['data']:>6} {h['blocks']:>6} {h['epochs']:>6} {h['learning_rate']:>8g} {h['best_epoch']:>4} {h['frozen_anls']:>7.3f} {h['adapted_anls']:>7.3f} {h['adapted_exact_match']:>7.3f}")
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'byod': byod,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'pages': pages,
    'split_unit': 'page (receipt); no shop or vendor field',
    'fit_check': {name: {'fitting': f['n_fitting'], 'dropped': f['dropped']} for name, f in fit.items()},
    'baselines': {'last_number': baseline_last, 'keyword_lookup': baseline_lookup},
    'frozen_test': frozen_test,
    'validation_metrics': adapted_val,
    'test_metrics': adapted_test,
    'comparison': comparison,
    'disagreements': [{'id': r['id'], 'frozen': frozen_answers[r['id']], 'adapted': adapted_answers[r['id']], 'gold': gold_texts(r)} for r in disagreements],
    'outcomes': outcomes,
    'reading': reading,
    'run_history': run_history,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/layoutlm_document_qa_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
print({'report': 'outputs/layoutlm_document_qa_evaluation_report.json'})

{'anls': {'last_number': 0.406, 'keyword_lookup': 0.635, 'frozen': 0.843, 'adapted': 0.94}}
{'exact_match': {'last_number': 0.253, 'keyword_lookup': 0.594, 'frozen': 0.782, 'adapted': 0.921}}
{'delta_vs_frozen': {'anls': 0.097, 'exact_match': 0.14}}
{'by_field': {'menu.nm': {'n': 50, 'frozen': 0.79, 'adapted': 0.92, 'too_few_to_read': False}, 'sub_total.discount_price': {'n': 4, 'frozen': 1.0, 'adapted': 0.75, 'too_few_to_read': True}, 'sub_total.service_price': {'n': 2, 'frozen': 1.0, 'adapted': 1.0, 'too_few_to_read': True}, 'sub_total.subtotal_price': {'n': 27, 'frozen': 0.98, 'adapted': 1.0, 'too_few_to_read': False}, 'sub_total.tax_price': {'n': 17, 'frozen': 0.92, 'adapted': 0.94, 'too_few_to_read': False}, 'total.cashprice': {'n': 35, 'frozen': 0.85, 'adapted': 0.9, 'too_few_to_read': False}, 'total.changeprice': {'n': 29, 'frozen': 0.87, 'adapted': 1.0, 'too_few_to_read': False}, 'total.creditcardprice': {'n': 7, 'frozen': 0.96, 'adapted': 1.0, 'too_few_to_read': True}, 'total.

**What to notice:** the delta row, the per-field rows with their `n`, `fields_that_lost_score`, the disagreement examples, the printed reading and the run history.

<details><summary>Check your reasoning</summary>No, not every field. In the recorded T4 run test ANLS rose 0.843 → 0.940 and exact match 0.782 → 0.921, past both baselines (0.406 and 0.635). The gain was concentrated where the frozen model had been weak: item count 0.17 → 0.83 (n = 12), first item 0.79 → 0.92 (n = 50), change 0.87 → 1.00 (n = 29). But **the discount field went down, 1.00 → 0.75**: one of its four test questions was answered correctly before adaptation and wrongly after. That is a real regression on this split, and four questions are far too few to say whether adaptation hurts discounts in general — the same holds for the two service-charge questions (1.00 → 1.00). Fine-tuning moves every answer, and an overall gain can hide a field that got worse.</details>

## 9. Re-read the invoice, export the adapter and reload it

The five authored invoice questions from Section 5 are answered again by the adapted model — a page from a different document family than the receipts it was tuned on, so this is a small look at what the adaptation did *outside* its corpus (a lost answer here is a finding to record, not a failure) — and scored with the per-page `evaluation_report`, whose verdict is `sample-sanity` because five authored pairs on one perfectly-OCR'd page carry no dispersion estimate. The answers are written as CSV and an annotated PNG draws the adapted answer spans' boxes on the page.

`pipe.save_artifact` writes the trained tensors — the last four encoder blocks and the span head, about 113 MB — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `LayoutLMDocumentQAPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest, its digest and its exact tensor set **before** deserialising, refuses any tensor that is not an adaptable encoder-block or span-head tensor, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell compares the answers of the in-memory and the reloaded model on eight test receipts and stops with an explanation if any differ (VER4). Parity holds because every adaptation starts from the pretrained base (Section 7), so the adapter carries every tensor that differs from it.

**Predict before running:** after tuning on Indonesian receipts, will the adapted model still answer the five English invoice questions exactly?

In [ ]:
import csv
import shutil

adapted_results = [pipe.answer(question, words=words, boxes=boxes, image_size=image.size) for question in questions]
adapted_form = evaluation_report(adapted_results, golds, sample_kind='synthetic')
for before, after in zip(results, adapted_results, strict=True):
    print({'question': before['question'], 'frozen': before['answer'], 'adapted': after['answer'], 'gold': golds[questions.index(before['question'])]})
print({'invoice_after_adaptation': {m['id']: round(m['value'], 3) for m in adapted_form['metrics']}, 'verdict': adapted_form['verdict'], 'reason': adapted_form['reason'][:80] + '...'})
with open('outputs/layoutlm_document_qa_answers.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['image', 'question', 'frozen_answer', 'adapted_answer', 'adapted_score', 'start', 'end', 'gold'])
    for before, after, gold in zip(results, adapted_results, golds, strict=True):
        writer.writerow([image_name, after['question'], before['answer'], after['answer'], f"{after['score']:.6f}", after['start'], after['end'], ' | '.join(gold)])
COLOURS = [(200, 30, 30), (0, 140, 0), (40, 90, 220), (200, 120, 0), (130, 0, 160)]
annotated = image.convert('RGB').copy()
draw = ImageDraw.Draw(annotated)
for index, item in enumerate(adapted_results):
    if item['start'] is None:
        continue
    for box in boxes[item['start'] : item['end'] + 1]:
        draw.rectangle([box[0] - 2, box[1] - 2, box[2] + 2, box[3] + 2], outline=COLOURS[index % len(COLOURS)], width=2)
annotated.save('outputs/layoutlm_document_qa_annotated.png')

artifact_dir = Path('outputs/layoutlm_document_qa_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'layoutlm_document_qa', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = LayoutLMDocumentQAPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
before = [pipe.answer(r['question'], words=r['words'], boxes=r['boxes'], image_size=r['image_size'])['answer'] for r in test_records[:8]]
after = [reloaded.answer(r['question'], words=r['words'], boxes=r['boxes'], image_size=r['image_size'])['answer'] for r in test_records[:8]]
parity = {'identical_answers': sum(a == b for a, b in zip(before, after, strict=True)), 'of': len(before)}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
if parity['identical_answers'] != parity['of']:
    raise RuntimeError(f'Reload parity failed: {parity}. The saved adapter does not reproduce the evaluated model; re-run from Section 7, or restart the session and choose Run all.')
reloaded = None

weight_entry = next(entry for entry in MANIFEST['files'] if entry['path'] == WEIGHT_FILE)
result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': snapshot['files'], 'total_bytes': snapshot.get('total_bytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHT_FILE, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': weight_entry['sha256']},
    'data_source': data_source,
    'byod': byod,
    'corpus': None if USE_BYOD else {'name': CORPUS_NAME, 'repo': CORPUS_REPO, 'revision': CORPUS_REVISION, 'release': CORPUS_RELEASE, 'license': CORPUS_LICENSE, 'column': CORPUS_COLUMN, 'files': CORPUS_FILES},
    'inference_contract': {'input_manifest': input_manifest, 'sanity_checks': checks, 'page': {'name': image_name, 'size': list(image.size), 'rgb_sha256': image_sha256, 'n_words': len(words)}, 'items': [{k: r[k] for k in ('question', 'answer', 'start', 'end', 'score', 'n_windows', 'seconds')} for r in results], 'golds': golds, 'frozen_report': frozen_form, 'adapted_report': adapted_form},
    'comparison': comparison,
    'outcomes': outcomes,
    'reading': reading,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'device': pipe.device, 'dtype': 'float32', 'source': pipe.source},
}
with open('outputs/layoutlm_document_qa_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

{'question': 'What is the invoice number?', 'frozen': 'NW-2026-0417', 'adapted': 'NW-2026-0417', 'gold': ['NW-2026-0417']}
{'question': 'Who is the customer?', 'frozen': 'Blue Yonder Airlines', 'adapted': 'Blue Yonder Airlines', 'gold': ['Blue Yonder Airlines']}
{'question': 'What is the total due?', 'frozen': '$1,099.20', 'adapted': '$1,099.20', 'gold': ['$1,099.20', '1,099.20']}
{'question': 'What is the due date?', 'frozen': '11 April 2026', 'adapted': '11 April 2026', 'gold': ['11 April 2026']}
{'question': 'How many cargo pallets were invoiced?', 'frozen': '40', 'adapted': '40', 'gold': ['40']}
{'invoice_after_adaptation': {'anls': 1.0, 'exact_match': 1.0}, 'verdict': 'sample-sanity', 'reason': '5 authored question(s) on one tutorial page whose words and boxes you rendered y...'}


{'artifact': 'outputs/layoutlm_document_qa_adapter', 'format': 'org.valcorza.layoutlm-document-qa.adapter.v1', 'tensors': 66, 'bytes': 113419976, 'sha256': 'cf4ffc7966de0fbc...'}


{'reload_parity': {'identical_answers': 8, 'of': 8}, 'reloaded_best_epoch': 5}
['layoutlm_document_qa_adapter', 'layoutlm_document_qa_annotated.png', 'layoutlm_document_qa_answers.csv', 'layoutlm_document_qa_evaluation_report.json', 'layoutlm_document_qa_input_manifest.json', 'layoutlm_document_qa_result.json', 'layoutlm_document_qa_train.jsonl']


**What to notice:** the five frozen / adapted / gold rows, the invoice ANLS after adaptation, the artifact's tensor count and size, and the reload-parity line.

<details><summary>Check your reasoning</summary>Yes. In the recorded T4 run the adapted model answered all five invoice questions exactly as before (ANLS 1.0, `sample-sanity`): the four tuned blocks did not erase what the model reads on a clean English form — one page of evidence, not a measurement. The artifact held 66 tensors, 113,419,976 bytes, and reload parity was 8/8: the exported adapter is the model that was evaluated.</details>

## 10. Try it — questions the page cannot answer

The model has no abstention: `answer` always returns its best span, and `score` only ranks spans *within* the page. This cell asks three questions the rendered invoice cannot answer — it has no delivery address, no VAT registration number and no signature — and prints each answer with its score from the pretrained model (reloaded from the verified snapshot for this comparison) and from the adapted model, beside the score of one question the page does answer. Edit `UNSUPPORTED_QUESTIONS` and run only this cell to try your own.

**Predict before running:** will either model return an empty answer for *What is the delivery address?* If it returns a span, will its score be clearly lower than the scores in Section 5?

In [ ]:
UNSUPPORTED_QUESTIONS = ['What is the delivery address?', 'What is the VAT registration number?', 'Who signed the invoice?']

frozen_base = LayoutLMDocumentQAPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, device=pipe.device)
probe_rows = []
for question in [questions[0], *UNSUPPORTED_QUESTIONS]:
    frozen_item = frozen_base.answer(question, words=words, boxes=boxes, image_size=image.size)
    adapted_item = pipe.answer(question, words=words, boxes=boxes, image_size=image.size)
    probe_rows.append({'question': question, 'answerable': question == questions[0], 'frozen': frozen_item['answer'], 'frozen_score': round(frozen_item['score'], 4), 'adapted': adapted_item['answer'], 'adapted_score': round(adapted_item['score'], 4)})
    print(probe_rows[-1])
frozen_base = None
gc.collect()
print({'empty_answers': sum(1 for row in probe_rows if not row['frozen'] or not row['adapted']), 'note': 'the model has no abstention; a span and a score are returned for every question'})

{'question': 'What is the invoice number?', 'answerable': True, 'frozen': 'NW-2026-0417', 'frozen_score': 1.0, 'adapted': 'NW-2026-0417', 'adapted_score': 0.9991}
{'question': 'What is the delivery address?', 'answerable': False, 'frozen': '14 Harbour Road,', 'frozen_score': 0.2774, 'adapted': '$740.00', 'adapted_score': 0.0651}
{'question': 'What is the VAT registration number?', 'answerable': False, 'frozen': '$183.20', 'frozen_score': 0.9997, 'adapted': '$183.20', 'adapted_score': 0.998}


{'question': 'Who signed the invoice?', 'answerable': False, 'frozen': 'Northwind Traders Ltd.', 'frozen_score': 0.9892, 'adapted': 'Northwind Traders Ltd.', 'adapted_score': 0.5801}
{'empty_answers': 0, 'note': 'the model has no abstention; a span and a score are returned for every question'}


**What to notice:** whether any answer is empty, which words the unsupported questions were answered with, and how their scores compare with the answerable question's.

<details><summary>Check your reasoning</summary>No answer is empty, and the scores do not reliably separate the cases: the model always returns its best span, and the span score only says how concentrated its start and end mass was on this page. In a CPU check of this notebook version the pretrained model answered *What is the delivery address?* with `14 Harbour Road,` (score 0.2774) — low, but *What is the VAT registration number?* got `$183.20` (0.9997) and *Who signed the invoice?* got `Northwind Traders Ltd.` (0.9892), as high as the correct invoice number (1.0000). An unsupported question lands on a plausible-looking span — an address line, a number near a related label, a name — often with a high score. That is why the notebook never treats `score` as a confidence and ships no rejection threshold: deciding that a page cannot answer a question needs labelled unanswerable examples and a calibration you fit yourself.</details>

## 11. Your turn — change one thing: two trainable blocks

Optional; **Predict → Change one thing → Run → Observe → Explain**. It retrains, so it takes about as long as Sections 7–10 did (Section 7 alone took 82 s on the T4 record and about 15 minutes on CPU).

1. **Predict:** with `TRAINABLE_ENCODER_LAYERS = 2` (half the default), how many tensors will the adapter hold (the default holds 66), will the test ANLS end above or below the four-block run's, and will the item-count field still gain? Write your guesses next to the default run's row in the Section 8 run history.
2. **Change:** in Section 7 set `TRAINABLE_ENCODER_LAYERS = 2`. Change nothing else.
3. **Run:** select the Section 7 cell and choose **Runtime → Run after**. Section 7 reloads the pretrained model before adapting, Section 8 compares against the same Section 6 frozen score and adds a row to the run history, and Section 9 exports and reloads the new adapter.
4. **Observe:** in Section 7, epoch 0 shows the *frozen* validation ANLS again (0.819 in the recorded T4 run), not the adapted one — the sign that the model was reset — and the trainable-parameter count. In Section 8, compare the two rows of the run history and the per-field rows; in Section 9, the tensor count, the bytes and the reload parity.
5. **Explain:** in one or two sentences, what did training half as many blocks change, and is the difference between the two runs larger than the run-to-run spread (about 0.01) and than what 229 test questions can resolve?

<details><summary>Check your reasoning</summary>Two blocks train 14,177,282 parameters (half the blocks, about half the trainable weights) and the adapter holds **34 tensors**, about 57 MB. Epoch 0 is 0.819 again and the reload parity passes, because the run started from the pretrained base. The build record has no run of exactly this setting; its nearest ones, on an RTX 5070 Ti and the same split, reached test ANLS 0.854 with two blocks after three epochs at this learning rate and 0.909 after six epochs at `5e-5`, against 0.946 for the default four blocks there. Expect a lower test ANLS than the four-block run by a few points — more than the run-to-run spread, so a real effect of depth on this split, but still one split.</details>

## Interpretation and limits

The frozen model reads most receipt totals already — its test ANLS (0.843 in the recorded T4 run) sits well above both non-neural baselines (0.406 and 0.635) — and a bounded fine-tuning of the last four encoder blocks and the span head on 119 receipts lifts held-out ANLS by several points (to 0.940), concentrated in the fields it had been missing (item count 0.17 → 0.83, item names 0.79 → 0.92, change 0.87 → 1.00), with a 113 MB adapter that reloads to identical answers. Not every field gained: the discount field fell from 1.00 to 0.75 — one of its four test questions lost — which is a real regression on this split and far too few questions to generalise from. That is the claim: the adaptation contract works end to end on a real gold-span document corpus whose OCR the model never sees as pixels, and the numbers it produces are read per field against two non-neural baselines and the frozen model rather than in isolation.

The test split is 229 questions over 50 receipts from one seeded split of one corpus, the questions are templated from field categories rather than written by people, the metrics are two reference-based scores (own implementations of DocVQA's normalisation, neither a human judgement), and the OCR is CORD's annotation — clean words in reading order with tight boxes, which no OCR engine on a photographed receipt reproduces. The split is by receipt, not by shop, so the held-out score is an estimate for more receipts of the same kind. So a gain here says the contract works, not that the adapted model is better on your documents or your OCR, that it handles other languages, handwriting or long pages, or that its spans are faithful — Section 10 shows a reader returning a span for questions the page cannot answer, and the adaptation changes nothing about that. Fine-tuning on a narrow corpus can also erode the model elsewhere; the invoice re-read in Section 9 is one page of evidence about that, not a measurement.

Three things to carry to real data. **Baselines first:** the last-number and keyword-lookup baselines and the frozen model's score on *your* gold spans, with *your* OCR, are the numbers to read before any adapted one, per field, with the number of questions beside each. **Leakage:** keep every question on a page in one split (the contract does this) and split by document or by vendor when your pages come from few sources, never at random over near-duplicate pages — CORD-v2 itself contains one duplicated receipt, which the sample drops. **Ceilings:** a question + page over one 512-token window is answered window by window at inference and dropped from training by the fit check; long-document training is out of scope.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify one column of a real annotated corpus, validate the demonstrated dataset contract with no page in two splits, execute the inference contract and a bounded fine-tuning, evaluate against two trivial baselines and the frozen model on a page-disjoint split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, document-understanding accuracy on any other domain or OCR, a usable rejection threshold, or production fitness.

**Optional experiments (they do not affect the default path).** Each one changes one thing; note the numbers you want to compare first (the Section 8 run history keeps one row per pass). In Colab, **Runtime → Run after** runs the selected cell and every cell below it; Sections 5, 6 and 7 reload the pretrained model before they use it, so no experiment inherits an earlier adaptation.

- **Two trainable blocks:** Section 11.
- **More epochs:** set `EPOCHS = 8` in Section 7, select Section 7 and choose **Run after**. Epoch 0 is the frozen 0.819 again; does the validation ANLS keep rising after epoch 6, and which epoch does the selector keep?
- **A higher learning rate:** set `LEARNING_RATE = 5e-5` in Section 7, select Section 7 and choose **Run after**. The build record measured 0.936 test ANLS for four blocks at this rate against 0.946 at the default on another GPU: is your difference larger than the run-to-run spread?
- **Your own question:** edit `UNSUPPORTED_QUESTIONS` in Section 10 and run only that cell. Which wrong spans come back with the highest scores?
- **Another split:** change `SPLIT_SEED` in Section 4, select Section 4 and choose **Run after**. How far do the frozen and adapted ANLS move with a different draw of 50 test receipts — more or less than the run-to-run spread?
- **Your own pages:** BYOD (Section 4, then **Run after**). Before the adapted number, how do the two baselines and the frozen model score on your gold spans, field by field?

## Troubleshooting

- **Section 1 stops with "needs a Linux x86_64 runtime".** The locked environment is built from manylinux wheels. Use Google Colab, Kaggle or a Linux Jupyter server; Windows and macOS kernels are not supported.
- **Section 1 fails while downloading** (`uv` wheel, Python build or packages). The runtime needs PyPI and the python-build-standalone download; run the cell again once the network is back. A "size/SHA-256" or "does not match its digest" error means a file was altered: do not edit the cell, regenerate the notebook from the repository.
- **The run is slow.** On CPU the six epochs take about 15 minutes (Prerequisites); a GPU runtime (**Runtime → Change runtime type**) takes under two minutes for them.
- **"The isolated environment's Python process exited" or an out-of-memory error.** Usually the runtime ran out of memory (Sections 9 and 10 briefly hold two copies of the model). Restart the session and choose **Run all**; in Section 7, `BATCH_SIZE = 8` needs less memory.
- **`FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 3.** A staged file is incomplete or altered — delete it from `weights/layoutlm-document-qa/` and run Section 3 again. A column-digest mismatch in Section 4 is the same for the cache under `weights/cord-v2/`; a download error there means the Hub was unreachable — run Section 4 again.
- **BYOD is refused in Section 4.** The message names what to fix: a split with too few records (at least 12 records with one question per page), a line of a `.jsonl` file that is not valid JSON (the message names the line), a record missing a key or breaking a rule (it names the record), a file that is not `.json` or `.jsonl`, or a duplicate id. Fix the file and run Section 4 with **Run after** again. An empty or cancelled upload, or a runtime without the Colab upload dialog, asks you to choose a file or set `BYOD_PATH`.
- **Section 8 reads "did not help" or "The selector kept epoch 0".** Not an error: under these settings the adaptation did not beat the frozen model. Read the epoch history in Section 7; the export and reload in Section 9 still run.
- **Reload parity fails in Section 9.** The saved adapter does not reproduce the evaluated model. Re-run from Section 7 (it reloads the pretrained model first), or restart the session and choose **Run all**.
- **Numbers differ from the worked answers.** The frozen numbers are the same on CPU and GPU to the printed precision; training results vary by about 0.01 between devices (Section 7, **Run-to-run spread**). Larger differences on the default settings mean a form field was changed.
- **You re-ran the routing cell in Section 1.** It starts a new isolated process and discards every variable created so far; choose **Run all** again.

## Glossary

- **Extractive question answering:** answering by pointing at a span of the given text (here, a run of OCR words) rather than generating new text.
- **OCR words and boxes:** the words an OCR engine (or here, an annotation) found on the page, each with its pixel box; the model reads the boxes scaled to a 0–1000 grid.
- **Window:** one 512-token slice of question + page; a longer page is read in overlapping windows.
- **Span score:** the product of the start and end softmax probabilities of the chosen span within its window — a ranking signal, not a probability that the answer is right.
- **ANLS / exact match:** average normalised Levenshtein similarity to the gold answer (0 below 0.5), and the share of answers equal to the gold after normalisation.
- **Last-number / keyword-lookup baseline:** two readers with no model — the last number on the page, and the next number after the page word that best matches the question.
- **Encoder block / span head:** one of the 12 transformer layers, and the final layer that turns each token into a start and an end logit; fine-tuning here trains the last blocks and the head only.
- **Epoch / validation selection:** one pass over the training questions; the epoch kept is the one with the highest validation ANLS (epoch 0 = the frozen model).
- **Training / validation / test split:** questions used to train, to choose the epoch, and to report the final result once; split by page, so no page is in two splits.
- **Fit check:** the step that drops (never truncates) a training question whose page needs more than one window.
- **safetensors / adapter / reload parity:** a tensor file format that cannot run code when loaded; the saved trained tensors; and the check that a pipeline rebuilt from them and the pinned base gives the same answers.

## Conclusion (your notes)

Optional personal notes, not a required submission. Complete each sentence from your own run:

- The task: from ___ to ___. On the test receipts the last-number baseline scored ANLS ___, keyword lookup ___, the pretrained model ___ and the adapted model ___ (kept epoch ___).
- The field that gained most was ___ (n = ___); a field that lost score was ___ (n = ___), which I can / cannot read because ___.
- For a question the page cannot answer, the model returned ___ with score ___, which shows ___.
- Training two blocks instead of four (Section 11) changed ___; the difference is / is not larger than the run-to-run spread, because ___.
- Before trusting this adaptation on my own documents I would measure ___ on ___, with my own OCR, split by ___, with the baselines first.

## References

- Repository README: https://github.com/kurtvalcorza/layoutlm-document-qa-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/layoutlm-document-qa-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/layoutlm-document-qa-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model (Impira, MIT): https://huggingface.co/impira/layoutlm-document-qa
- Upstream architecture code (LayoutLM, Microsoft): https://github.com/microsoft/unilm/tree/master/layoutlm
- LayoutLM: Pre-training of Text and Layout for Document Image Understanding (Xu et al., 2019): https://arxiv.org/abs/1912.13318
- DocVQA: A Dataset for VQA on Document Images (Mathew, Karatzas, Jawahar, 2020): https://arxiv.org/abs/2007.00398
- Scene Text Visual Question Answering — the ANLS metric (Biten et al., 2019): https://arxiv.org/abs/1905.13648
- CORD: A Consolidated Receipt Dataset for Post-OCR Parsing (Park et al., NeurIPS 2019 Document Intelligence Workshop; CORD-v2, CC BY 4.0): https://huggingface.co/datasets/naver-clova-ix/cord-v2
- DIMER Notebook Specification 2.2 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)